In [1]:
import praca_core
import re
# Auto-generated from notebook
# imports + functions + classes only
from transformers.models import phi


from praca_core import *
from praca_magisterska.v2.ContextsAndLP import LittleProblem

from praca_magisterska.v2.TermAndFormulas import *

from praca_magisterska.v2.ContextsAndLP import *

from praca_magisterska.v2.HelpfullFunctions import *

import re

from typing import List

import sys

from pathlib import Path

from concurrent.futures import ProcessPoolExecutor

from itertools import chain

from tqdm.auto import tqdm

from nanoGPT.model import GPTConfig, GPT

from dataclasses import dataclass

import torch

from dataclasses import dataclass

from typing import Callable, Optional

import os

import pickle

import torch

import tiktoken

import numpy as np

import torch

from typing import Optional

import random

import numpy as np

from tqdm import tqdm




/home/lukasz/PycharmProjects/Magisterka/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1


NameError: name 'bad_iffs' is not defined

In [ ]:
random_iff()

In [ ]:
x = [1,2,3]
x.remove(2)
x

In [ ]:
formulas_str = '''((p → q) ∧ p) → q
((p → q) ∧ ¬q) → ¬p
((p ∨ q) → r) → (p → r)
((p ∨ q) → r) → (q → r)
(p ∧ q) → p
q → (p ∨ q)
p → (q → p)
(¬p → q) → (p ∨ q)
((p → r) ∧ (q → r) ∧ (p ∨ q)) → r
((p ∧ q) → r) → (p → (q → r))
((p → (q → r)) → ((p ∧ q) → r))
((p → ¬p) → ¬p)
¬(p ∧ ¬p)
(p ∨ ¬p)
((p → q) ∧ (q → r)) → (p → r)
(((p → q) → p) → p)
((¬p → p) → p)
p ↔ p
(p ∨ q ∨ r) → (¬p → ((q ∨ r) ∧ ¬p))
(p → q) ↔ ((p ∧ q) → p)
((p ∨ q) → r) → ((p → r) ∨ (q → r))
((p → q) ∧ (r → s)) → (p ∧ r → q ∨ r)
(p ∨ q ∨ r) → (((p ∨ q) ∧ ¬r) ∨ ((r ∧ p) ∧ q))
(((p ∨ q) ∧ ¬r) ∨ ((r ∧ p) ∧ q)) → (p ∨ q ∨ r)
((p ∨ q) ↔ (r ∨ s)) → (((p ↔ r) ∨ (q ↔ s)))
((p ↔ r) ∨ (q ↔ s)) → ((p ∨ q) ↔ (r ∨ s))
((p ∧ q) ↔ (r ∧ s)) → (((p ↔ r) ∧ (q ↔ s)))
((p → q) → r) → (((p → q) → (p → r)))
((p ∨ q) ∧ ¬p) → q
((p → q) → (p ∧ r → q))
((p → q) → (p → (q ∨ r)))
(p → q) → (p ∨ q)
((p ∨ q) ∧ ((p → q) → q)) → p
¬(p ∧ (¬p ∧ q))
(p → ¬q) ∧ (q → r)
((p → q) ∧ (q → p)) → (p ∨ q)
((p ∨ q) → (p ∨ ¬q)) → (p ∨ q)
((p → q) ∨ (p → ¬q)) → (p → q)
((p → q) ∧ (q → r)) → (p → r)
((p → q) ∧ (r → s)) → ((p ∨ r) → (q ∨ s))
((p ∧ q) → r) → ((p → r) ∧ (q → r))
((p → q) ∧ (r → s)) → ((p ∧ r) → (q ∧ s))
((p ∧ q) ∧ (r ∨ s)) → (p ∧ q ∧ r)
((¬p → q) ∧ (q → p)) → (p ∧ ¬q)
(((p → q) → r) → ((r → p) → (q → p)))
(((p → q) ∨ (p → r)) ∨ ((p → s))) → (p → (q ∨ r ∨ s))
((p → q) ∧ (r → s)) → (p ∧ r → s ∧ q)
((p → q) ∧ (r → q) ∧ (s → q)) → (p ∧ r ∧ s → q)
((p ∧ q) ∧ (p ∧ q → r)) → (¬p ∧ ¬q → ¬r)
((¬p → q) ∨ (p ∨ ¬q)) → ((p → q) ∨ r)
(((p ∨ q) ∧ (r ∨ s)) → (((p → q) ∨ (p → r)) ∨ ((q → s) ∨ (q → p))))
(((p → q) ∧ (r → s) ∧ (t → u)) → (p ∧ r ∧ t → q ∧ s ∧ u))
((p ↔ q) ∧ p) → q
((¬p → ¬q) → (¬p → q)) → p'''
formulas_str = formulas_str.splitlines()
formulas = []
for i in formulas_str:
    f = parse_infix(i)
    if is_tautology(f):
        formulas.append(f)
    if is_tautology(Negation(f)):
        formulas.append(Negation(f))
print(formulas.__len__())

proofs = [
    ("((p → q) ∧ p) → q",
"""1. (p → q) ∧ p  assumption
2. p → q  ∧-elimination1, 1
3. p  ∧-elimination2, 1
4. q  →-elimination, 2, 3
5. ((p → q) ∧ p) → q  →-introduction, 1, 4"""),

    ("((p → q) ∧ ¬q) → ¬p",
"""1. (p → q) ∧ ¬q  assumption
2. p → q  ∧-elimination1, 1
3. ¬q  ∧-elimination2, 1
4. p  assumption
5. q  →-elimination, 2, 4
6. ⊥  ¬-elimination, 3, 5
7. ¬p  ¬-introduction, 4, 6
8. ((p → q) ∧ ¬q) → ¬p  →-introduction, 1, 7"""),

    ("((p ∨ q) → r) → (p → r)",
"""1. (p ∨ q) → r  assumption
2. p  assumption
3. p ∨ q  ∨-introduction1, 2
4. r  →-elimination, 1, 3
5. p → r  →-introduction, 2, 4
6. ((p ∨ q) → r) → (p → r)  →-introduction, 1, 5"""),

    ("((p ∨ q) → r) → (q → r)",
"""1. (p ∨ q) → r  assumption
2. q  assumption
3. p ∨ q  ∨-introduction2, 2
4. r  →-elimination, 1, 3
5. q → r  →-introduction, 2, 4
6. ((p ∨ q) → r) → (q → r)  →-introduction, 1, 5"""),

    ("(p ∧ q) → p",
"""1. p ∧ q  assumption
2. p  ∧-elimination1, 1
3. (p ∧ q) → p  →-introduction, 1, 2"""),

    ("q → (p ∨ q)",
"""1. q  assumption
2. p ∨ q  ∨-introduction2, 1
3. q → (p ∨ q)  →-introduction, 1, 2"""),

    ("p → (q → p)",
"""1. p  assumption
2. q  assumption
3. p  weakening,   2, 1
4. p  context,   3
5. q → p  →-introduction,   2, 4
6. p → (q → p)  →-introduction,   1, 5"""),

    ("(¬p → q) → (p ∨ q)",
"""1. ¬p → q  assumption
2. p ∨ ¬p  TND
3. p  assumption
4. p ∨ q  ∨-introduction1, 3
5. ¬p  assumption
6. q  →-elimination, 1, 5
7. p ∨ q  ∨-introduction2, 6
8. p ∨ q  ∨-elimination, 2, 4, 7
9. (¬p → q) → (p ∨ q)  →-introduction, 1, 8"""),

    ("(((p → r) ∧ (q → r)) ∧ (p ∨ q)) → r",
"""1. ((p → r) ∧ (q → r)) ∧ (p ∨ q)  assumption
2. (p → r) ∧ (q → r)  ∧-elimination1, 1
3. p ∨ q  ∧-elimination2, 1
4. p → r  ∧-elimination1, 2
5. q → r  ∧-elimination2, 2
6. p  assumption
7. r  →-elimination, 4, 6
8. q  assumption
9. r  →-elimination, 5, 8
10. r  ∨-elimination, 3, 7, 9
11. (((p → r) ∧ (q → r)) ∧ (p ∨ q)) → r  →-introduction, 1, 10"""),

    ("((p ∧ q) → r) → (p → (q → r))",
"""1. (p ∧ q) → r  assumption
2. p  assumption
3. q  assumption
4. p ∧ q  ∧-introduction,   2, 3
5. r  →-elimination,   1, 4
6. q → r  →-introduction,   3, 5
7. p → (q → r)  →-introduction,   2, 6
8. ((p ∧ q) → r) → (p → (q → r))  →-introduction,   1, 7"""),

    ("(p → (q → r)) → ((p ∧ q) → r)",
"""1. p → (q → r)  assumption
2. p ∧ q  assumption
3. p  ∧-elimination1, 2
4. q  ∧-elimination2, 2
5. q → r  →-elimination, 1, 3
6. r  →-elimination, 5, 4
7. (p ∧ q) → r  →-introduction, 2, 6
8. (p → (q → r)) → ((p ∧ q) → r)  →-introduction, 1, 7"""),

    ("(p → ¬p) → ¬p",
"""1. p → ¬p  assumption
2. p  assumption
3. ¬p  →-elimination, 1, 2
4. ⊥  ¬-elimination, 3, 2
5. ¬p  ¬-introduction, 2, 4
6. (p → ¬p) → ¬p  →-introduction, 1, 5"""),

    ("¬(p ∧ ¬p)",
"""1. p ∧ ¬p  assumption
2. p  ∧-elimination1, 1
3. ¬p  ∧-elimination2, 1
4. ⊥  ¬-elimination, 3, 2
5. ¬(p ∧ ¬p)  ¬-introduction, 1, 4"""),

    ("p ∨ ¬p",
"""1. p ∨ ¬p  TND"""),

    ("((p → q) ∧ (q → r)) → (p → r)",
"""1. (p → q) ∧ (q → r)  assumption
2. p → q  ∧-elimination1, 1
3. q → r  ∧-elimination2, 1
4. p  assumption
5. q  →-elimination, 2, 4
6. r  →-elimination, 3, 5
7. p → r  →-introduction, 4, 6
8. ((p → q) ∧ (q → r)) → (p → r)  →-introduction, 1, 7"""),

    ("((p → q) → p) → p",
"""1. (p → q) → p  assumption
2. p ∨ ¬p  TND
3. p  assumption
4. p  context,   3
5. ¬p  assumption
6. p  assumption
7. ¬p  context,   5
8. ⊥  ¬-elimination,   7, 6
9. q  ⊥-elimination,   8
10. p → q  →-introduction,   6, 9
11. p  →-elimination,   1, 10
12. p  ∨-elimination,   2, 4, 11
13. ((p → q) → p) → p  →-introduction,   1, 12"""),

    ("(¬p → p) → p",
"""1. ¬p → p  assumption
2. p ∨ ¬p  TND
3. p  assumption
4. p  context,   3
5. ¬p  assumption
6. p  →-elimination,   1, 5
7. p  ∨-elimination,   2, 4, 6
8. (¬p → p) → p  →-introduction,   1, 7"""),

    ("p ↔ p",
"""1. p  assumption
2. p  context,   1
3. p  assumption
4. p  context,   3
5. p ↔ p  ↔-introduction,   2, 4"""),

    ("((p ∨ q) ∨ r) → (¬p → ((q ∨ r) ∧ ¬p))",
"""1. (p ∨ q) ∨ r  assumption
2. ¬p  assumption
3. p ∨ q  assumption
4. p  assumption
5. ¬p  context,   2
6. ⊥  ¬-elimination,   5, 4
7. q ∨ r  ⊥-elimination,   6
8. q  assumption
9. q ∨ r  ∨-introduction1,   8
10. q ∨ r  ∨-elimination,   3, 7, 9
11. r  assumption
12. q ∨ r  ∨-introduction2,   11
13. q ∨ r  ∨-elimination,   1, 10, 12
14. ¬p  context,   13
15. (q ∨ r) ∧ ¬p  ∧-introduction,   13, 14
16. ¬p → ((q ∨ r) ∧ ¬p)  →-introduction,   2, 15
17. ((p ∨ q) ∨ r) → (¬p → ((q ∨ r) ∧ ¬p))  →-introduction,   1, 16"""),

    ("((p ∨ q) → r) → ((p → r) ∨ (q → r))",
"""1. (p ∨ q) → r  assumption
2. p  assumption
3. p ∨ q  ∨-introduction1, 2
4. r  →-elimination, 1, 3
5. p → r  →-introduction, 2, 4
6. (p → r) ∨ (q → r)  ∨-introduction1, 5
7. ((p ∨ q) → r) → ((p → r) ∨ (q → r))  →-introduction, 1, 6"""),

    ("((p → q) ∧ (r → s)) → ((p ∧ r) → (q ∨ r))",
"""1. (p → q) ∧ (r → s)  assumption
2. p → q  ∧-elimination1, 1
3. p ∧ r  assumption
4. p  ∧-elimination1, 3
5. q  →-elimination, 2, 4
6. q ∨ r  ∨-introduction1, 5
7. (p ∧ r) → (q ∨ r)  →-introduction, 3, 6
8. ((p → q) ∧ (r → s)) → ((p ∧ r) → (q ∨ r))  →-introduction, 1, 7"""),

    ("(((p ∨ q) ∧ ¬r) ∨ ((r ∧ p) ∧ q)) → ((p ∨ q) ∨ r)",
"""1. ((p ∨ q) ∧ ¬r) ∨ ((r ∧ p) ∧ q)  assumption
2. (p ∨ q) ∧ ¬r  assumption
3. p ∨ q  ∧-elimination1, 2
4. (p ∨ q) ∨ r  ∨-introduction1, 3
5. ((r ∧ p) ∧ q)  assumption
6. r ∧ p  ∧-elimination1, 5
7. r  ∧-elimination1, 6
8. (p ∨ q) ∨ r  ∨-introduction2, 7
9. (p ∨ q) ∨ r  ∨-elimination, 1, 4, 8
10. (((p ∨ q) ∧ ¬r) ∨ ((r ∧ p) ∧ q)) → ((p ∨ q) ∨ r)  →-introduction, 1, 9"""),

    ("((p → q) → r) → ((p → q) → (p → r))",
"""1. (p → q) → r  assumption
2. p → q  assumption
3. r  →-elimination,   1, 2
4. p  assumption
5. r  weakening,   4, 3
6. p → r  →-introduction,   4, 5
7. (p → q) → (p → r)  →-introduction,   2, 6
8. ((p → q) → r) → ((p → q) → (p → r))  →-introduction,   1, 7"""),

    ("((p ∨ q) ∧ ¬p) → q",
"""1. (p ∨ q) ∧ ¬p  assumption
2. p ∨ q  ∧-elimination1,   1
3. ¬p  ∧-elimination2,   1
4. p  assumption
5. ⊥  ¬-elimination,   3, 4
6. q  ⊥-elimination,   5
7. q  assumption
8. q  ∨-elimination,   2, 6, 7
9. ((p ∨ q) ∧ ¬p) → q  →-introduction,   1, 8"""),

    ("(p → q) → ((p ∧ r) → q)",
"""1. p → q  assumption
2. p ∧ r  assumption
3. p  ∧-elimination1, 2
4. q  →-elimination, 1, 3
5. (p ∧ r) → q  →-introduction, 2, 4
6. (p → q) → ((p ∧ r) → q)  →-introduction, 1, 5"""),

    ("(p → q) → (p → (q ∨ r))",
"""1. p → q  assumption
2. p  assumption
3. q  →-elimination, 1, 2
4. q ∨ r  ∨-introduction1, 3
5. p → (q ∨ r)  →-introduction, 2, 4
6. (p → q) → (p → (q ∨ r))  →-introduction, 1, 5"""),

    ("¬(p ∧ (¬p ∧ q))",
"""1. p ∧ (¬p ∧ q)  assumption
2. p  ∧-elimination1, 1
3. ¬p ∧ q  ∧-elimination2, 1
4. ¬p  ∧-elimination1, 3
5. ⊥  ¬-elimination, 4, 2
6. ¬(p ∧ (¬p ∧ q))  ¬-introduction, 1, 5"""),

    ("((p → q) ∧ (q → r)) → (p → r)",
"""1. (p → q) ∧ (q → r)  assumption
2. p → q  ∧-elimination1, 1
3. q → r  ∧-elimination2, 1
4. p  assumption
5. q  →-elimination, 2, 4
6. r  →-elimination, 3, 5
7. p → r  →-introduction, 4, 6
8. ((p → q) ∧ (q → r)) → (p → r)  →-introduction, 1, 7"""),

    ("((p → q) ∧ (r → s)) → ((p ∨ r) → (q ∨ s))",
"""1. (p → q) ∧ (r → s)  assumption
2. p → q  ∧-elimination1, 1
3. r → s  ∧-elimination2, 1
4. p ∨ r  assumption
5. p  assumption
6. q  →-elimination, 2, 5
7. q ∨ s  ∨-introduction1, 6
8. r  assumption
9. s  →-elimination, 3, 8
10. q ∨ s  ∨-introduction2, 9
11. q ∨ s  ∨-elimination, 4, 7, 10
12. (p ∨ r) → (q ∨ s)  →-introduction, 4, 11
13. ((p → q) ∧ (r → s)) → ((p ∨ r) → (q ∨ s))  →-introduction, 1, 12"""),

    ("((p → q) ∧ (r → s)) → ((p ∧ r) → (q ∧ s))",
"""1. (p → q) ∧ (r → s)  assumption
2. p → q  ∧-elimination1, 1
3. r → s  ∧-elimination2, 1
4. p ∧ r  assumption
5. p  ∧-elimination1, 4
6. r  ∧-elimination2, 4
7. q  →-elimination, 2, 5
8. s  →-elimination, 3, 6
9. q ∧ s  ∧-introduction, 7, 8
10. (p ∧ r) → (q ∧ s)  →-introduction, 4, 9
11. ((p → q) ∧ (r → s)) → ((p ∧ r) → (q ∧ s))  →-introduction, 1, 10"""),

    ("((p → q) → r) → ((r → p) → (q → p))",
"""1. (p → q) → r  assumption
2. r → p  assumption
3. q  assumption
4. p ∨ ¬p  TND
5. p  assumption
6. p  context,   5
7. ¬p  assumption
8. p  assumption
9. q  weakening,   8, 3
10. q  weakening,   7, 9
11. p → q  →-introduction,   8, 10
12. r  →-elimination,   1, 11
13. p  →-elimination,   2, 12
14. p  ∨-elimination,   4, 6, 13
15. q → p  →-introduction,   3, 14
16. (r → p) → (q → p)  →-introduction,   2, 15
17. ((p → q) → r) → ((r → p) → (q → p))  →-introduction,   1, 16"""),

    ("(((p → q) ∨ (p → r)) ∨ (p → s)) → (p → ((q ∨ r) ∨ s))",
"""1. ((p → q) ∨ (p → r)) ∨ (p → s)  assumption
2. p  assumption
3. (p → q) ∨ (p → r)  assumption
4. p → q  assumption
5. q  →-elimination,    4, 2
6. q ∨ r  ∨-introduction1,    5
7. (q ∨ r) ∨ s  ∨-introduction1,    6
8. p → r  assumption
9. r  →-elimination,    8, 2
10. q ∨ r  ∨-introduction2,    9
11. (q ∨ r) ∨ s  ∨-introduction1,    10
12. (q ∨ r) ∨ s  ∨-elimination,    3, 7, 11
13. p → s  assumption
14. s  →-elimination,    13, 2
15. (q ∨ r) ∨ s  ∨-introduction2,    14
16. (q ∨ r) ∨ s  ∨-elimination,    1, 12, 15
17. p → ((q ∨ r) ∨ s)  →-introduction,    2, 16
18. (((p → q) ∨ (p → r)) ∨ (p → s)) → (p → ((q ∨ r) ∨ s))  →-introduction,    1, 17"""),

    ("((p → q) ∧ (r → s)) → ((p ∧ r) → (s ∧ q))",
"""1. (p → q) ∧ (r → s)  assumption
2. p → q  ∧-elimination1,    1
3. r → s  ∧-elimination2,    1
4. p ∧ r  assumption
5. p  ∧-elimination1,    4
6. r  ∧-elimination2,    4
7. q  →-elimination,    2, 5
8. s  →-elimination,    3, 6
9. s ∧ q  ∧-introduction,    8, 7
10. (p ∧ r) → (s ∧ q)  →-introduction,    4, 9
11. ((p → q) ∧ (r → s)) → ((p ∧ r) → (s ∧ q))  →-introduction,    1, 10"""),

    ("(((p → q) ∧ (r → q)) ∧ (s → q)) → (((p ∧ r) ∧ s) → q)",
"""1. ((p → q) ∧ (r → q)) ∧ (s → q)  assumption
2. s → q  ∧-elimination2,    1
3. (p ∧ r) ∧ s  assumption
4. s  ∧-elimination2,    3
5. q  →-elimination,    2, 4
6. ((p ∧ r) ∧ s) → q  →-introduction,    3, 5
7. (((p → q) ∧ (r → q)) ∧ (s → q)) → (((p ∧ r) ∧ s) → q)  →-introduction,    1, 6"""),

    ("((p ∧ q) ∧ ((p ∧ q) → r)) → ((¬p ∧ ¬q) → ¬r)",
"""1. (p ∧ q) ∧ ((p ∧ q) → r)  assumption
2. p ∧ q  ∧-elimination1,    1
3. ¬p ∧ ¬q  assumption
4. r  assumption
5. p  ∧-elimination1,    2
6. ¬p  ∧-elimination1,    3
7. ⊥  ¬-elimination,    6, 5
8. ⊥  weakening,   4, 7
9. ¬r  ¬-introduction,   4, 8
10. (¬p ∧ ¬q) → ¬r  →-introduction,   3, 9
11. ((p ∧ q) ∧ ((p ∧ q) → r)) → ((¬p ∧ ¬q) → ¬r)  →-introduction,   1, 10"""),

    ("((p ∨ q) ∧ (r ∨ s)) → (((p → q) ∨ (p → r)) ∨ ((q → s) ∨ (q → p)))",
"""1. (p ∨ q) ∧ (r ∨ s)  assumption
2. r ∨ s  ∧-elimination2,   1
3. r  assumption
4. p  assumption
5. r  weakening, 4, 3
6. p → r  →-introduction, 4, 5
7. (p → q) ∨ (p → r)  ∨-introduction2, 6
8. ((p → q) ∨ (p → r)) ∨ ((q → s) ∨ (q → p))  ∨-introduction1, 7
9. s  assumption
10. q  assumption
11. s  weakening, 10, 9
12. q → s  →-introduction, 10, 11
13. (q → s) ∨ (q → p)  ∨-introduction1, 12
14. ((p → q) ∨ (p → r)) ∨ ((q → s) ∨ (q → p))  ∨-introduction2, 13
15. ((p → q) ∨ (p → r)) ∨ ((q → s) ∨ (q → p))  ∨-elimination, 2, 8, 14
16. ((p ∨ q) ∧ (r ∨ s)) → (((p → q) ∨ (p → r)) ∨ ((q → s) ∨ (q → p)))  →-introduction, 1, 15"""),

    ("(((p → q) ∧ (r → s)) ∧ (t → u)) → (((p ∧ r) ∧ t) → ((q ∧ s) ∧ u))",
"""1. ((p → q) ∧ (r → s)) ∧ (t → u)  assumption
2. (p → q) ∧ (r → s)  ∧-elimination1, 1
3. t → u  ∧-elimination2, 1
4. p → q  ∧-elimination1, 2
5. r → s  ∧-elimination2, 2
6. (p ∧ r) ∧ t  assumption
7. p ∧ r  ∧-elimination1, 6
8. t  ∧-elimination2, 6
9. p  ∧-elimination1, 7
10. r  ∧-elimination2, 7
11. q  →-elimination, 4, 9
12. s  →-elimination, 5, 10
13. u  →-elimination, 3, 8
14. q ∧ s  ∧-introduction, 11, 12
15. (q ∧ s) ∧ u  ∧-introduction, 14, 13
16. ((p ∧ r) ∧ t) → ((q ∧ s) ∧ u)  →-introduction, 6, 15
17. (((p → q) ∧ (r → s)) ∧ (t → u)) → (((p ∧ r) ∧ t) → ((q ∧ s) ∧ u))  →-introduction, 1, 16"""),

    ("((p ↔ q) ∧ p) → q",
"""1. (p ↔ q) ∧ p    assumption
2. p ↔ q  ∧-elimination1, 1
3. p  ∧-elimination2, 1
4. q  ↔-elimination1, 2, 3
5. ((p ↔ q) ∧ p) → q  →-introduction, 1, 4"""),
]
proofs = [  #dodalem
    (formula, "\n".join(  #dodalem
        f"{m.group(1)}{m.group(2)}    {re.sub(r'\s*,\s*', ', ', m.group(3))}" if (m := re.match(r'^(\d+\.\s*)(.*?)\s{2,}(\S.*)$', line)) else line  #dodalem
        for line in proof.splitlines()  #dodalem
    ))  #dodalem
    for formula, proof in proofs  #dodalem
]  #dodalem



In [ ]:
def replace_full_numbers_not_after_letter(text, old, new):  #dodalem
    if not isinstance(text, str):  #dodalem
        raise TypeError("Bad arguments")  #dodalem
    if len(old) != len(new):  #dodalem
        raise ValueError("old and new must have the same length")  #dodalem
    old_s = [str(x) for x in old]  #dodalem
    new_s = [str(x) for x in new]  #dodalem
    if not all(x.isdigit() for x in old_s + new_s):  #dodalem
        raise ValueError("all values in old/new must be whole numbers")  #dodalem
    if len(set(old_s)) != len(old_s):  #dodalem
        raise ValueError("values in old must be unique")  #dodalem
    if len(old_s) == 0:  #dodalem
        return text  #dodalem
    mapping = {old_s[i]: new_s[i] for i in range(len(old_s))}  #dodalem
    alternatives = "|".join(sorted((re.escape(x) for x in old_s), key=len, reverse=True))  #dodalem
    pattern = rf'(?<!\d)(?<![^\W\d_])({alternatives})(?!\d)'  #dodalem
    return re.sub(pattern, lambda m: mapping[m.group(1)], text)  #dodalem

def replace_full_number_not_after_letter(text, old, new):  #dodalem
    return replace_full_numbers_not_after_letter(text, [old], [new])  #dodalem

In [ ]:
BigProof = ""
for i in range(len(proofs)):
    new_lines_number = len(proofs[i][1].splitlines())
    old_lines_number = len(BigProof.splitlines())
    new_proof_part = proofs[i][1]
    old_ids = list(range(1, new_lines_number + 1))  #dodalem
    new_ids = list(range(old_lines_number + 1, old_lines_number + new_lines_number + 1))  #dodalem
    new_proof_part = replace_full_numbers_not_after_letter(new_proof_part, old_ids, new_ids)  #dodalem
    if BigProof != "":
        BigProof += "\n"
    BigProof += new_proof_part
def int_from_start(s: str):
    m = re.match(r'\d+', s)
    return int(m.group()) if m else None
from tqdm import tqdm
def is_well_numbered(proof:str):
    #print("^^^^^^^^^^^^^^^^^^",proof,"___________________________")
    for i in tqdm(range(len(proof.splitlines()))):
        if int_from_start(proof.splitlines()[i]) != i+1:
            raise Exception("Nieprawidlowe numerywanie wierszy")
    return True
print(BigProof)


BigProof += "\n331. ¬¬p    assumption"
BigProof += "\n332. ¬¬¬p    assumption"
BigProof += "\n333. ⊥    ¬-elimination, 332, 331"
print(is_well_numbered(BigProof))
parseProof(BigProof)

In [ ]:
def extract_proof_of_tautology(proof, line_number):
    if  isinstance(proof, str):
        proof = proof.splitlines()
    idxs = set()
    idxs.add(line_number-1)
    new_idxs = set()#te które mamy w zbiorze, ale nie mamy jeszcze ich sąsiadów
    new_idxs.add(line_number-1)
    while new_idxs.__len__() != 0:
        i = random.choice(list(new_idxs))
        new_idxs.remove(i)
        newest_idxs = re.findall(r"\d+",proof[i])
        for j in newest_idxs:
            if not int(j)-1 in idxs:
                idxs.add(int(j)-1)
                new_idxs.add(int(j)-1)
    idxs = sorted(list(idxs))
    ans = ""
    for i in idxs:
        ans += proof[i] +"\n"
    ans = ans.splitlines()
    for i in range(ans.__len__()):
        ans[i] = re.split(r"([. ,–])",ans[i])
    new_idxs = dict()
    for i in range(idxs.__len__()):
        new_idxs[str(idxs[i]+1)] = str(i+1)
    for i in range(ans.__len__()):
        for j in range(ans[i].__len__()):
            if ans[i][j] in new_idxs.keys():
                ans[i][j] = new_idxs[ans[i][j]]
    for i in range(ans.__len__()):
        new_line = ""
        for j in ans[i]:
            new_line += j
        ans[i] = new_line
    final_ans = ""
    for i in ans:
        final_ans += i
        final_ans += "\n"
    return final_ans[:-1]


In [ ]:
import gc
import multiprocessing as mp
from copy import deepcopy
from concurrent.futures import ProcessPoolExecutor

_BIG_PROOF = None

def init_worker(big_proof):
    global _BIG_PROOF
    _BIG_PROOF = big_proof

def one_job(job_id):
    global _BIG_PROOF

    local_proofs_table = []

    print(job_id)

    long_proof = deepcopy(_BIG_PROOF)
    long_proof = generate_long_proofv1(long_proof, how_many_new_lines=15000)

    parsedProof = parseProof(long_proof)

    if isinstance(long_proof, str):
        long_proof_lines = long_proof.splitlines()
    else:
        long_proof_lines = long_proof

    table = []
    for i in parsedProof.keys():
        if parsedProof[i].rule not in [Assumption(), TruthIntroduction(), TND()]:
            table.append((i, len(parsedProof[i].LP.assumptions.additional_context)))

    for idx, add_ctx_len in table:
        x = random.randint(0,100)
        if add_ctx_len == 0:
            conclusion = parsedProof[idx].LP.conclusion
            rule = parsedProof[idx].rule

            if FreeVariables(conclusion) != set() or x == 0:
                if (not contains_TF(conclusion)) or x == 0:
                    if (not isinstance(rule, TND)) or x == 0:
                        if (not isinstance(rule, Assumption)) or x == 0:
                            extracted_proof = extract_proof_of_tautology(long_proof_lines, idx)
                            if len(extracted_proof.splitlines()) > 2 or x == 0:
                                if len(extracted_proof) <= 1024:
                                    parsed_next_proof = parseProof(extracted_proof)

                                    if parsed_next_proof[len(parsed_next_proof)].LP.assumptions.additional_context != []:
                                        raise TypeError("Bad proof")

                                    local_proofs_table.append(extracted_proof)

    del parsedProof, long_proof, long_proof_lines, table
    gc.collect()

    return job_id, local_proofs_table


def run_parallel(big_proof, total_jobs=200, workers=20):
    proofs_table = []
    ctx = mp.get_context("fork")

    with ProcessPoolExecutor(
        max_workers=workers,
        mp_context=ctx,
        initializer=init_worker,
        initargs=(big_proof,),
    ) as executor:
        for job_id, local_proofs in executor.map(one_job, range(total_jobs), chunksize=1):
            proofs_table.extend(local_proofs)

            del job_id, local_proofs
            gc.collect()

    gc.collect()
    return proofs_table



for i in range(0):
    print("I=",i)
    proofs_table = run_parallel(BigProof, total_jobs=100, workers=30)
    with open("KORPUSY/korpus_bez_założeń.txt", "a", encoding="utf-8") as f:
        print("--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n--------------------------------------------------------------------------------------------------------------------------------\n")
        for j in proofs_table:
            f.write("\n\n" + j  )
    proofs_table.clear()
    del proofs_table
    gc.collect()



In [ ]:
with open("KORPUSY/korpus_bez_założeń.txt", "r", encoding="utf-8") as f:
    proofs = f.read()

In [ ]:
proofs = proofs.split("\n\n")

In [ ]:
testy = '''¬(((t ∨ v) ∧ ¬((t ∨ v))))
¬(((p ∧ r) ∧ ¬((p ∧ r))))
¬(((u ∨ q) ∧ ¬((u ∨ q))))
((p ∧ ¬t) → ¬¬((p ∧ ¬t)))
¬(((¬u ∨ q) ∧ ¬((¬u ∨ q))))
((((t ∨ v) ∨ p) ∧ ¬p) → ¬p)
(((t ∧ p) ∧ (t ∧ p)) ↔ (t ∧ p))
(((u ↔ r) ∧ (u ↔ r)) ↔ (u ↔ r))
(((s ∧ p) ∨ (s ∧ p)) ↔ (s ∧ p))
((((s ↔ t) → ¬v) ∧ (s ↔ t)) → ¬v)
((t ∧ ¬q) → ((¬s ∨ q) → (t ∧ ¬q)))
((v → (t ∨ p)) → ¬¬((v → (t ∨ p))))
((p → (s ∨ r)) → ¬¬((p → (s ∨ r))))
(((t ∧ q) ∧ u) → ¬¬(((t ∧ q) ∧ u)))
(((u ∨ t) ∨ p) → ¬¬(((u ∨ t) ∨ p)))
(((u ∧ t) → p) → ¬¬(((u ∧ t) → p)))
(((u ∨ t) ∨ s) → ¬¬(((u ∨ t) ∨ s)))
¬(((s ∨ (v ∧ q)) ∧ ¬((s ∨ (v ∧ q)))))
¬(((q → (t ∨ s)) ∧ ¬((q → (t ∨ s)))))
¬((((p ∧ v) → u) ∧ ¬(((p ∧ v) → u))))
¬(((p → (q ∨ u)) ∧ ¬((p → (q ∨ u)))))
¬(((v ∨ (s ∧ u)) ∧ ¬((v ∨ (s ∧ u)))))
¬((((p ∧ t) ∧ u) ∧ ¬(((p ∧ t) ∧ u))))
¬(((p → (v ∨ q)) ∧ ¬((p → (v ∨ q)))))
¬((((u ∨ v) ∨ q) ∧ ¬(((u ∨ v) ∨ q))))
¬(((q ∧ (t ∨ p)) ∧ ¬((q ∧ (t ∨ p)))))
¬((((v ∧ s) ∧ q) ∧ ¬(((v ∧ s) ∧ q))))
¬(((q → (p ∨ r)) ∧ ¬((q → (p ∨ r)))))
¬((((t ∧ s) → r) ∧ ¬(((t ∧ s) → r))))
¬(((t ∧ (s ∨ v)) ∧ ¬((t ∧ (s ∨ v)))))
((¬((v ∧ ¬t)) → (v ∧ ¬t)) → (v ∧ ¬t))
((¬((¬u ∨ q)) → (¬u ∨ q)) → (¬u ∨ q))
¬((((u ∨ p) ∨ s) ∧ ¬(((u ∨ p) ∨ s))))
((p ↔ t) → (((p ∨ u) ∨ r) → (p ↔ t)))
(((v ↔ s) ∧ ((q ∨ p) ∨ s)) → (v ↔ s))
(((q → (u ∨ v)) ∧ (r ∧ u)) → (r ∧ u))
((¬s ∨ v) → ((¬s ∨ v) ∨ ((r ∧ u) ∧ s)))
((((s ∨ q) ∨ v) ∧ (¬u ∨ t)) → (¬u ∨ t))
((p ∧ ¬t) → ((r → (v ∨ t)) ∨ (p ∧ ¬t)))
(((v ∧ ¬r) ∧ (u ∧ (p ∨ r))) → (v ∧ ¬r))
(((v ∨ s) ∧ ((s ∧ u) ∧ r)) → ((s ∧ u) ∧ r))
((q → (r ∨ t)) → (t → ((q → (r ∨ t)) ∧ t)))
(((q ∨ r) ∨ p) → ((u ↔ r) ∨ ((q ∨ r) ∨ p)))
(((q ∧ p) → s) → ((p → u) → ((q ∧ p) → s)))
(((s ∨ r) ↔ (s → u)) → ((s ∨ r) → (s → u)))
((q → (r ∨ v)) → ((v ↔ q) ∨ (q → (r ∨ v))))
(((t ∧ v) ∧ q) → ((r ∨ q) → ((t ∧ v) ∧ q)))
(((q ∧ ¬u) ∧ (r ∧ (v ∨ p))) → (r ∧ (v ∨ p)))
(((v ∧ t) → s) → ((¬t ∨ q) → ((v ∧ t) → s)))
(((¬q ∨ p) ∧ ((s ∧ t) → r)) → ((s ∧ t) → r))
(((¬t ∨ v) ∧ ((q ∨ v) ∨ t)) → ((q ∨ v) ∨ t))
(((¬p ∨ t) ∧ (t → (q ∨ p))) → (t → (q ∨ p)))
(((v ↔ s) ∨ ((v ↔ s) ∧ (v ∧ ¬t))) ↔ (v ↔ s))
((q ∨ (s ∧ p)) → ((q ∨ (s ∧ p)) ∨ (t ∧ ¬r)))
((r → (q ∨ p)) → ((¬r ∨ q) → (r → (q ∨ p))))
(((u ∨ v) ∧ ((u ∨ v) ∨ (¬v ∨ q))) ↔ (u ∨ v))
((((u ↔ t) → (¬r ∨ p)) ∧ (u ↔ t)) → (¬r ∨ p))
(((¬t ∨ s) ↔ (¬t ∨ q)) → ((¬t ∨ q) → (¬t ∨ s)))
((r ∧ ¬q) → ((¬v ∨ q) → ((r ∧ ¬q) ∧ (¬v ∨ q))))
((((u ∧ s) ∨ (¬p ∨ q)) ∧ ¬((u ∧ s))) → (¬p ∨ q))
(¬(((u ∧ s) → (t → s))) ↔ ((u ∧ s) ∧ ¬((t → s))))
((((s ∧ p) ∧ v) ∧ (q → (t ∨ r))) → ((s ∧ p) ∧ v))
(((p ∧ q) ∧ s) → (((r ∧ s) ∧ p) ∨ ((p ∧ q) ∧ s)))
((p ∧ (r ∨ u)) → (((t ∧ s) ∧ u) ∨ (p ∧ (r ∨ u))))
(((s ∧ u) → q) → (((s ∧ u) → q) ∨ (p → (u ∨ r))))
(((t ∨ (s ∧ u)) ∧ ((r ∨ s) ∨ p)) → ((r ∨ s) ∨ p))
(((s ∨ u) ∨ p) → (((s ∨ u) ∨ p) ∨ ((v ∧ p) ∧ t)))
(((t → v) ∨ ((t → v) ∧ ((q ∧ t) → u))) ↔ (t → v))
((((p ∨ u) ∨ s) ∧ (t ∧ (q ∨ u))) → ((p ∨ u) ∨ s))
((((s ∨ u) ∨ r) ∨ ((s ∨ u) ∨ r)) ↔ ((s ∨ u) ∨ r))
(((v ∧ (p ∨ u)) ∨ (v ∧ (p ∨ u))) ↔ (v ∧ (p ∨ u)))
(((r ∨ s) ∨ ((r ∨ s) ∧ ((p ∨ t) ∨ s))) ↔ (r ∨ s))
(((s → (v ∨ t)) ∧ (t ∨ (s ∧ u))) → (t ∨ (s ∧ u)))
((p ∧ (u ∨ s)) → ((p ∧ (u ∨ s)) ∨ ((p ∨ v) ∨ s)))
((((v ∨ p) ∨ t) ∧ ((p ∨ v) ∨ q)) → ((v ∨ p) ∨ t))
((((s ∨ v) ∨ r) ∧ (p → (q ∨ u))) → (p → (q ∨ u)))
(((q ∧ p) → r) → (((q ∧ p) → r) ∨ ((s ∨ u) ∨ q)))
((((r ∨ u) ∨ v) ∧ (v ∨ (t ∧ q))) → ((r ∨ u) ∨ v))
(((u ∨ r) ∨ t) → (((r ∧ q) ∧ s) → ((u ∨ r) ∨ t)))
(((p ∧ v) ∧ q) → ((v ∨ (t ∧ r)) → ((p ∧ v) ∧ q)))
(((s → (r ∨ q)) ∧ (s → (r ∨ q))) ↔ (s → (r ∨ q)))
((((p ∧ t) → u) ∨ ((p ∧ t) → u)) ↔ ((p ∧ t) → u))
(((p ∨ (q ∧ t)) ∧ (v → (s ∨ u))) → (p ∨ (q ∧ t)))
(((r ∨ (q ∧ u)) ∧ (q → (u ∨ t))) → (q → (u ∨ t)))
(((s ∨ (t ∧ q)) ∧ (r ∨ (v ∧ t))) → (s ∨ (t ∧ q)))
(((p ∧ q) ∧ r) → (((r ∨ v) ∨ s) → ((p ∧ q) ∧ r)))
((r → (p ∨ s)) → (((r ∨ p) ∨ s) → (r → (p ∨ s))))
(((r ∨ p) ∨ q) → (((r ∨ p) ∨ q) ∨ ((q ∨ u) ∨ s)))
(((s ∧ p) → t) → ((u ∧ (p ∨ q)) → ((s ∧ p) → t)))
((u ∧ (v ∨ p)) → ((v ∨ (p ∧ q)) → (u ∧ (v ∨ p))))
(((u ∧ v) ∧ t) → (((u ∧ v) ∧ t) ∨ ((u ∧ v) → p)))
((v → (s ∨ q)) → ((v → (s ∨ q)) ∨ ((v ∧ p) → r)))
(((t ∨ (p ∧ u)) ∧ ((r ∧ u) ∧ s)) → (t ∨ (p ∧ u)))
(((v ∧ (p ∨ q)) ∧ (u → (s ∨ q))) → (u → (s ∨ q)))
((((q ∧ p) ∧ v) ∧ (u → (p ∨ r))) → ((q ∧ p) ∧ v))
((u ∨ (v ∧ r)) → ((u ∨ (v ∧ r)) ∨ ((p ∧ s) ∧ t)))
((q ∧ (u ∨ p)) → ((q ∧ (u ∨ p)) ∨ (r ∨ (t ∧ p))))
(((r ∨ (u ∧ p)) ∧ (r ∨ (u ∧ p))) ↔ (r ∨ (u ∧ p)))
((((u ∧ v) ∧ s) ∧ ((s ∨ t) ∨ u)) → ((s ∨ t) ∨ u))
(((v ∧ q) ∧ r) → (((s ∧ v) ∧ p) → ((v ∧ q) ∧ r)))
(((t → (q ∨ v)) ∧ (t → (q ∨ v))) ↔ (t → (q ∨ v)))
((((r ∧ u) → q) ∧ ((p ∧ t) → q)) → ((p ∧ t) → q))
(((u ∧ (r ∨ v)) ∧ (r → (q ∨ p))) → (u ∧ (r ∨ v)))
((t ∨ (r ∧ u)) → (((q ∧ r) ∧ s) ∨ (t ∨ (r ∧ u))))
((((u ∧ p) ∧ v) ∧ (r ∨ (v ∧ s))) → ((u ∧ p) ∧ v))
((((p ∧ q) → u) ∧ (q → (v ∨ u))) → (q → (v ∨ u)))
((u ∧ (p ∨ q)) → (((p ∧ q) ∧ v) ∨ (u ∧ (p ∨ q))))
(((q ∧ p) ∧ u) → (((q ∨ v) ∨ s) → ((q ∧ p) ∧ u)))
((v → (q ∨ s)) → ((v → (q ∨ s)) ∨ (u ∨ (q ∧ r))))
(((s → (t ∨ v)) ∧ (u ∧ (t ∨ s))) → (s → (t ∨ v)))
((((v ∧ r) → t) ∧ (t → (p ∨ v))) → (t → (p ∨ v)))
((u ∧ (r ∨ p)) → ((u ∧ (r ∨ p)) ∨ (r ∧ (s ∨ u))))
((q ∧ (t ∨ u)) → ((q ∧ (t ∨ u)) ∨ ((p ∨ t) ∨ s)))
((q ∨ (r ∧ u)) → (((t ∧ s) ∧ q) ∨ (q ∨ (r ∧ u))))
((((p ∧ s) ∧ v) ∧ (u → (v ∨ p))) → (u → (v ∨ p)))
((((p ∨ s) ∨ t) ∨ ((p ∨ s) ∨ t)) ↔ ((p ∨ s) ∨ t))
((s ∧ (u ∨ v)) → ((s ∧ (u ∨ v)) ∨ ((s ∧ r) ∧ q)))
(((r ∧ q) → s) → ((r → (v ∨ u)) → ((r ∧ q) → s)))
(((t ∨ r) ∨ q) → ((u ∧ (s ∨ t)) → ((t ∨ r) ∨ q)))
(((u ∨ (s ∧ q)) ∧ (s ∨ (t ∧ p))) → (s ∨ (t ∧ p)))
((((p ∧ r) ∧ v) ∨ ((p ∧ r) ∧ v)) ↔ ((p ∧ r) ∧ v))
((((r ∨ v) ∨ u) ∨ ((r ∨ v) ∨ u)) ↔ ((r ∨ v) ∨ u))
((u ∨ (p ∧ s)) → ((s ∨ (r ∧ p)) → (u ∨ (p ∧ s))))
(((r ∧ q) → t) → ((q → (p ∨ t)) ∨ ((r ∧ q) → t)))
((((t ∧ s) → u) ∧ ((t ∧ s) → u)) ↔ ((t ∧ s) → u))
((s → (t ∨ v)) → ((s → (t ∨ v)) ∨ (u ∧ (q ∨ p))))
((((q ∧ t) → r) ∧ ((u ∧ s) ∧ t)) → ((q ∧ t) → r))
(((v → (p ∨ u)) ∧ ((v ∧ r) ∧ q)) → (v → (p ∨ u)))
(((s ∨ (v ∧ r)) ∧ (r ∧ (v ∨ u))) → (s ∨ (v ∧ r)))
((t ∨ (r ∧ v)) → (((v ∧ r) → u) → (t ∨ (r ∧ v))))
(((v ∧ (r ∨ u)) ∧ (t ∨ (v ∧ u))) → (t ∨ (v ∧ u)))
((u ∨ (s ∧ t)) → (((u ∧ p) → t) ∨ (u ∨ (s ∧ t))))
((s → (v ∨ u)) → ((s → (v ∨ u)) ∨ ((s ∨ r) ∨ q)))
(((u → (r ∨ v)) ∧ (u ∧ (q ∨ p))) → (u → (r ∨ v)))
((((q ∨ r) ∨ v) ∧ (t ∨ (v ∧ u))) → ((q ∨ r) ∨ v))
(((q ∧ v) ∧ r) → ((r ∨ (p ∧ t)) ∨ ((q ∧ v) ∧ r)))
((((q ∧ v) ∧ p) ∨ ((q ∧ v) ∧ p)) ↔ ((q ∧ v) ∧ p))
((s → (u ∨ r)) → ((p ∧ (s ∨ u)) → (s → (u ∨ r))))
((t → (u ∨ v)) → (((t ∧ u) ∧ r) ∨ (t → (u ∨ v))))
((((p ∧ t) ∧ s) ∧ ((p ∧ t) ∧ s)) ↔ ((p ∧ t) ∧ s))
(((p ∨ u) ∨ t) → (((q ∧ v) → p) ∨ ((p ∨ u) ∨ t)))
((((t ∧ v) → u) ∧ ((t ∨ u) ∨ s)) → ((t ∨ u) ∨ s))
(((q ∧ s) ∧ u) → (((s ∧ q) → r) ∨ ((q ∧ s) ∧ u)))
(((p ∨ (v ∧ r)) ∨ (p ∨ (v ∧ r))) ↔ (p ∨ (v ∧ r)))
((((t ∧ u) → p) ∧ ((p ∨ v) ∨ u)) → ((t ∧ u) → p))
((p ∧ (v ∨ u)) → (((u ∨ p) ∨ s) → (p ∧ (v ∨ u))))
(((q ∨ t) ∨ s) → (((s ∧ q) → t) ∨ ((q ∨ t) ∨ s)))
(((s ∧ v) ∧ u) → ((p → (q ∨ s)) → ((s ∧ v) ∧ u)))
(((r ∨ (u ∧ s)) ∧ (r ∨ (u ∧ s))) ↔ (r ∨ (u ∧ s)))
((t → (r ∨ p)) → ((v ∨ (s ∧ p)) → (t → (r ∨ p))))
((t → (r ∨ q)) → ((q ∨ (r ∧ t)) ∨ (t → (r ∨ q))))
((s ∧ (r ∨ t)) → ((s ∧ (r ∨ t)) ∨ ((q ∧ r) → s)))
((v ∧ (q ∨ s)) → ((v ∧ (q ∨ s)) ∨ (v → (s ∨ r))))
((r → (v ∨ p)) → ((r ∨ (q ∧ t)) → (r → (v ∨ p))))
(((p ∨ (q ∧ s)) ∧ (p ∨ (q ∧ s))) ↔ (p ∨ (q ∧ s)))
((((t ∨ u) ∨ v) ∧ ((r ∨ s) ∨ t)) → ((t ∨ u) ∨ v))
((((r ∧ t) → u) ∧ (r → (q ∨ u))) → ((r ∧ t) → u))
(((r ∨ (v ∧ p)) ∧ ((u ∨ v) ∨ r)) → ((u ∨ v) ∨ r))
((r → (s ∨ q)) → ((r → (s ∨ t)) ∨ (r → (s ∨ q))))
((u ∨ (s ∧ q)) → (((r ∧ s) ∧ q) ∨ (u ∨ (s ∧ q))))
(((s ∧ (p ∨ q)) ∧ ((v ∧ q) ∧ t)) → ((v ∧ q) ∧ t))
((((q ∧ r) ∧ t) ∨ ((q ∧ r) ∧ t)) ↔ ((q ∧ r) ∧ t))
((v ∧ (q ∨ t)) → (((r ∧ p) → v) → (v ∧ (q ∨ t))))
((s ∨ (q ∧ p)) → ((s ∨ (q ∧ p)) ∨ ((v ∧ u) ∧ q)))
(((u ∧ p) → r) → (((u ∧ p) → r) ∨ (p ∧ (v ∨ q))))
((t → (p ∨ q)) → ((t → (p ∨ q)) ∨ (v ∨ (r ∧ u))))
(((q ∨ (v ∧ t)) ∧ ((p ∨ q) ∨ v)) → (q ∨ (v ∧ t)))
((((u ∧ v) → p) ∧ (v ∧ (t ∨ q))) → ((u ∧ v) → p))
((u ∨ (r ∧ t)) → ((u ∨ (r ∧ t)) ∨ ((t ∧ q) ∧ s)))
(((r → v) ∧ ((r → v) ∨ (r ∨ (v ∧ u)))) ↔ (r → v))
((v ∧ (p ∨ t)) → ((v ∧ (p ∨ t)) ∨ ((v ∧ t) → r)))
((p ∨ (q ∧ v)) → ((p ∨ (q ∧ v)) ∨ ((q ∧ s) → t)))
((((u ∧ t) → v) ∧ (p → (t ∨ q))) → ((u ∧ t) → v))
(((p ∨ (v ∧ t)) ∧ (p ∨ (v ∧ t))) ↔ (p ∨ (v ∧ t)))
((((v ∧ p) ∧ r) ∧ ((t ∧ p) ∧ s)) → ((t ∧ p) ∧ s))
((((t ∧ q) → s) ∧ (p → (r ∨ s))) → (p → (r ∨ s)))
(((v ∧ s) ∧ u) → (((v ∧ s) ∧ u) ∨ (u ∨ (q ∧ s))))
(¬((((v ∨ r) ∨ p) ∨ t)) ↔ (¬(((v ∨ r) ∨ p)) ∧ ¬t))
((((s ∧ p) → (¬r ∨ u)) ∧ ¬((¬r ∨ u))) → ¬((s ∧ p)))
((¬(((u ∨ v) ∨ p)) → ((u ∨ v) ∨ p)) → ((u ∨ v) ∨ p))
((¬(((s ∨ q) ∨ t)) → ((s ∨ q) ∨ t)) → ((s ∨ q) ∨ t))
((¬(((q ∨ v) ∨ t)) → ((q ∨ v) ∨ t)) → ((q ∨ v) ∨ t))
(((v ∧ ¬t) ∧ ((v ∧ ¬t) ∨ (q ∧ (s ∨ p)))) ↔ (v ∧ ¬t))
((¬((p ∨ (q ∧ u))) → (p ∨ (q ∧ u))) → (p ∨ (q ∧ u)))
((¬((t → (v ∨ u))) → (t → (v ∨ u))) → (t → (v ∨ u)))
(((((t ∧ q) ∧ v) → (q ↔ s)) ∧ ((t ∧ q) ∧ v)) → (q ↔ s))
(((v → (u ∨ t)) ∧ (t → p)) → ((t → p) ∧ (v → (u ∨ t))))
(((((r ∧ u) → q) → (s ∨ t)) ∧ ((r ∧ u) → q)) → (s ∨ t))
((((q → (v ∨ s)) → (u → t)) ∧ (q → (v ∨ s))) → (u → t))
(((s ∧ q) ↔ (q ∧ (s ∨ t))) → ((s ∧ q) → (q ∧ (s ∨ t))))
(((q → u) ∧ (q ∧ (r ∨ v))) → ((q ∧ (r ∨ v)) ∧ (q → u)))
(((t → u) ↔ (p → (q ∨ s))) → ((t → u) → (p → (q ∨ s))))
((((s ∨ q) ∨ p) ∨ (t ∨ v)) → ((t ∨ v) ∨ ((s ∨ q) ∨ p)))
(((t ∨ (q ∧ r)) ↔ (s ↔ q)) → ((s ↔ q) → (t ∨ (q ∧ r))))
((((u ↔ r) → (p → (u ∨ s))) ∧ (u ↔ r)) → (p → (u ∨ s)))
(((t ∨ u) ∨ (v ∧ (r ∨ s))) → ((v ∧ (r ∨ s)) ∨ (t ∨ u)))
(((q → v) ↔ (s ∨ (q ∧ t))) → ((s ∨ (q ∧ t)) → (q → v)))
(((((r ∨ s) ∨ v) → (s ∨ q)) ∧ ((r ∨ s) ∨ v)) → (s ∨ q))
((t ∧ (s ∨ u)) → ((q → t) → ((t ∧ (s ∨ u)) ∧ (q → t))))
((t ∨ r) → ((p ∨ (t ∧ q)) → ((t ∨ r) ∧ (p ∨ (t ∧ q)))))
(((u ∧ (v ∨ t)) ∧ (u → r)) → ((u → r) ∧ (u ∧ (v ∨ t))))
(((p ∧ u) → r) → ((q ↔ t) → (((p ∧ u) → r) ∧ (q ↔ t))))
(((r ∧ v) ↔ ((s ∧ p) ∧ t)) → ((r ∧ v) → ((s ∧ p) ∧ t)))
(((u ∨ p) ↔ ((p ∨ q) ∨ r)) → (((p ∨ q) ∨ r) → (u ∨ p)))
((((r ∧ q) ∧ s) ↔ (s → v)) → (((r ∧ q) ∧ s) → (s → v)))
(((v ∧ (r ∨ t)) ↔ (t ↔ v)) → ((t ↔ v) → (v ∧ (r ∨ t))))
((((s ∧ v) ∧ r) ∨ (t ∧ r)) → ((t ∧ r) ∨ ((s ∧ v) ∧ r)))
((((r ∧ t) ∧ s) ↔ (p ↔ t)) → (((r ∧ t) ∧ s) → (p ↔ t)))
(((u → v) ↔ (r ∧ (p ∨ s))) → ((u → v) → (r ∧ (p ∨ s))))
((((q ∧ u) ∧ s) ∧ (s ∨ v)) → ((s ∨ v) ∧ ((q ∧ u) ∧ s)))
(((r → s) ∧ (u → (p ∨ t))) → ((u → (p ∨ t)) ∧ (r → s)))
(((t → (u ∨ q)) ∨ (r ↔ s)) → ((r ↔ s) ∨ (t → (u ∨ q))))
(((((s ∧ v) ∧ u) → (s ↔ r)) ∧ ((s ∧ v) ∧ u)) → (s ↔ r))
((((v ∧ s) → t) ∧ (((v ∧ s) → t) ∨ ¬s)) ↔ ((v ∧ s) → t))
((((s ∧ ¬u) → ((v ∧ t) → r)) ∧ (s ∧ ¬u)) → ((v ∧ t) → r))
((((s ∧ ¬u) → (t ∨ (u ∧ r))) ∧ (s ∧ ¬u)) → (t ∨ (u ∧ r)))
(((t ∧ q) ∧ r) → ((v ∧ ¬t) → (((t ∧ q) ∧ r) ∧ (v ∧ ¬t))))
((((¬s ∨ p) → (s ∨ (q ∧ t))) ∧ (¬s ∨ p)) → (s ∨ (q ∧ t)))
((s ∧ ¬u) → (((s ∧ p) → u) → ((s ∧ ¬u) ∧ ((s ∧ p) → u))))
(((r ∨ v) ∨ t) → ((r ∧ ¬q) → (((r ∨ v) ∨ t) ∧ (r ∧ ¬q))))
(((¬r ∨ s) ↔ (s ∨ (q ∧ r))) → ((s ∨ (q ∧ r)) → (¬r ∨ s)))
(((q ∧ ¬v) ∨ (r ∨ (s ∧ v))) → ((r ∨ (s ∧ v)) ∨ (q ∧ ¬v)))
(((u ∧ ¬r) ∧ ((t ∧ r) → s)) → (((t ∧ r) → s) ∧ (u ∧ ¬r)))
(((t ∨ s) ∨ u) → ((¬t ∨ r) → (((t ∨ s) ∨ u) ∧ (¬t ∨ r))))
(((p ∧ ¬u) ↔ (p ∧ (t ∨ q))) → ((p ∧ (t ∨ q)) → (p ∧ ¬u)))
((r ∨ (p ∧ v)) → ((¬t ∨ r) → ((r ∨ (p ∧ v)) ∧ (¬t ∨ r))))
((((p ∧ ¬r) → ((s ∨ r) ∨ q)) ∧ (p ∧ ¬r)) → ((s ∨ r) ∨ q))
((¬v ∨ s) → ((s → (p ∨ u)) → ((¬v ∨ s) ∧ (s → (p ∨ u)))))
((((t ↔ s) ∨ (u → (q ∨ v))) ∧ ¬((t ↔ s))) → (u → (q ∨ v)))
(((((s ∧ t) ∧ u) ∨ (u ↔ v)) ∧ ¬((u ↔ v))) → ((s ∧ t) ∧ u))
((((p ∧ q) → v) → (t ∨ u)) ↔ (¬(((p ∧ q) → v)) ∨ (t ∨ u)))
((((t ∨ r) ∨ (p ∨ (t ∧ q))) ∧ ¬((p ∨ (t ∧ q)))) → (t ∨ r))
((((r ∨ v) ∨ (p → (s ∨ q))) ∧ ¬((r ∨ v))) → (p → (s ∨ q)))
(((v → (s ∨ t)) → (t ↔ u)) ↔ (¬((v → (s ∨ t))) ∨ (t ↔ u)))
((((s ∨ q) ∨ ((v ∧ s) → p)) ∧ ¬(((v ∧ s) → p))) → (s ∨ q))
((((r ∧ u) → v) → (t ∨ q)) ↔ (¬(((r ∧ u) → v)) ∨ (t ∨ q)))
(((((s ∧ u) → q) ∨ (r ↔ p)) ∧ ¬((r ↔ p))) → ((s ∧ u) → q))
(((t ∧ s) → ((t ∨ s) ∨ p)) ↔ (¬((t ∧ s)) ∨ ((t ∨ s) ∨ p)))
((((t ∨ (v ∧ u)) ∨ (u ∨ v)) ∧ ¬((u ∨ v))) → (t ∨ (v ∧ u)))
(((p ∨ (q ∧ v)) → (v ∧ ¬t)) ↔ (¬((p ∨ (q ∧ v))) ∨ (v ∧ ¬t)))
((((¬p ∨ u) ∨ (q ∨ (v ∧ p))) ∧ ¬((¬p ∨ u))) → (q ∨ (v ∧ p)))
(((q ∧ ¬v) → (r ∨ (t ∧ u))) ↔ (¬((q ∧ ¬v)) ∨ (r ∨ (t ∧ u))))
((((s ∧ (q ∨ u)) ∨ (¬v ∨ u)) ∧ ¬((¬v ∨ u))) → (s ∧ (q ∨ u)))
((((¬u ∨ v) ∨ ((q ∧ p) ∧ t)) ∧ ¬(((q ∧ p) ∧ t))) → (¬u ∨ v))
((((¬u ∨ p) ∨ ((u ∧ s) ∧ p)) ∧ ¬(((u ∧ s) ∧ p))) → (¬u ∨ p))
((((u → (p ∨ r)) ∨ (¬u ∨ r)) ∧ ¬((u → (p ∨ r)))) → (¬u ∨ r))
(((p ↔ q) → (t → (s ∨ u))) → (¬((t → (s ∨ u))) → ¬((p ↔ q))))
(((r → p) → ((u ∨ r) ∨ s)) → (¬(((u ∨ r) ∨ s)) → ¬((r → p))))
((¬((u → s)) → ¬((v ∧ (u ∨ q)))) → ((v ∧ (u ∨ q)) → (u → s)))
(((s ∧ r) → (p ∨ (r ∧ u))) → (¬((p ∨ (r ∧ u))) → ¬((s ∧ r))))
((¬((r ∨ p)) → ¬((t ∨ (s ∧ u)))) → ((t ∨ (s ∧ u)) → (r ∨ p)))
(¬((((v ∨ s) ∨ t) → (u ↔ q))) ↔ (((v ∨ s) ∨ t) ∧ ¬((u ↔ q))))
((¬(((q ∧ p) → t)) → ¬((s → r))) → ((s → r) → ((q ∧ p) → t)))
((((u ∨ t) ∨ s) → (p ↔ s)) → (¬((p ↔ s)) → ¬(((u ∨ t) ∨ s))))
(¬(((s ∧ u) → ((r ∧ s) ∧ u))) ↔ ((s ∧ u) ∧ ¬(((r ∧ s) ∧ u))))
(((s ∧ (p ∨ t)) ↔ (p → v)) → (¬((s ∧ (p ∨ t))) ↔ ¬((p → v))))
((((v ∧ t) ∧ p) → (t ∨ p)) → (¬((t ∨ p)) → ¬(((v ∧ t) ∧ p))))
(¬(((q → u) → ((p ∨ t) ∨ u))) ↔ ((q → u) ∧ ¬(((p ∨ t) ∨ u))))
((¬(((v ∧ q) ∧ t)) → ¬((v ∧ u))) → ((v ∧ u) → ((v ∧ q) ∧ t)))
((((p ∧ t) → s) ↔ (t ∨ v)) → (¬(((p ∧ t) → s)) ↔ ¬((t ∨ v))))
(((v ∧ (q ∨ p)) → (t → s)) → (¬((t → s)) → ¬((v ∧ (q ∨ p)))))
(((u → v) → (v ∧ (p ∨ t))) → (¬((v ∧ (p ∨ t))) → ¬((u → v))))
((((t ↔ p) → ((t ∧ p) ∧ s)) ∧ ¬(((t ∧ p) ∧ s))) → ¬((t ↔ p)))
((((s ∨ v) ∨ t) → (s ∧ v)) → (¬((s ∧ v)) → ¬(((s ∨ v) ∨ t))))
((¬((s → v)) → ¬((v ∨ (p ∧ q)))) → ((v ∨ (p ∧ q)) → (s → v)))
((((q ∧ p) ∧ u) ∨ (((q ∧ p) ∧ u) ∧ (s → p))) ↔ ((q ∧ p) ∧ u))
(((u ∨ (v ∧ r)) ∨ ((u ∨ (v ∧ r)) ∧ (v → t))) ↔ (u ∨ (v ∧ r)))
((((t ∨ q) ∨ s) ∧ (((t ∨ q) ∨ s) ∨ (s ∨ u))) ↔ ((t ∨ q) ∨ s))
((((p ∧ q) ∧ s) ∧ (((p ∧ q) ∧ s) ∨ (s ∨ v))) ↔ ((p ∧ q) ∧ s))
(((v ∨ (p ∧ r)) ∨ ((v ∨ (p ∧ r)) ∧ (s → u))) ↔ (v ∨ (p ∧ r)))
(((u → (r ∨ v)) ∨ ((u → (r ∨ v)) ∧ (q → u))) ↔ (u → (r ∨ v)))
((((s ∧ u) ∧ p) ∨ (((s ∧ u) ∧ p) ∧ (u ∨ q))) ↔ ((s ∧ u) ∧ p))
((((r ∧ u) → v) ∧ (((r ∧ u) → v) ∨ (s ∧ ¬t))) ↔ ((r ∧ u) → v))
((((t ∧ s) ∧ q) ∧ (((t ∧ s) ∧ q) ∨ (s ∧ ¬r))) ↔ ((t ∧ s) ∧ q))
(((¬s ∨ p) → ((v ∧ u) ∧ s)) → (¬(((v ∧ u) ∧ s)) → ¬((¬s ∨ p))))
(¬(((¬p ∨ q) → (q ∨ (v ∧ p)))) ↔ ((¬p ∨ q) ∧ ¬((q ∨ (v ∧ p)))))
(¬(((q ∧ ¬t) → ((p ∨ r) ∨ u))) ↔ ((q ∧ ¬t) ∧ ¬(((p ∨ r) ∨ u))))
((((t ∧ p) → r) → (¬q ∨ u)) → (¬((¬q ∨ u)) → ¬(((t ∧ p) → r))))
(¬((((q ∧ v) ∧ t) ∨ (t ∧ v))) ↔ (¬(((q ∧ v) ∧ t)) ∧ ¬((t ∧ v))))
(¬(((s ↔ q) ∨ (t ∧ (u ∨ p)))) ↔ (¬((s ↔ q)) ∧ ¬((t ∧ (u ∨ p)))))
(¬(((s ∧ (r ∨ u)) ∨ (r ∨ u))) ↔ (¬((s ∧ (r ∨ u))) ∧ ¬((r ∨ u))))
(¬(((v ∨ (q ∧ s)) ∧ (s ∧ ¬t))) ↔ (¬((v ∨ (q ∧ s))) ∨ ¬((s ∧ ¬t))))
(¬(((¬u ∨ s) ∨ (p ∧ (s ∨ v)))) ↔ (¬((¬u ∨ s)) ∧ ¬((p ∧ (s ∨ v)))))
(¬(((s ∧ ¬p) ∧ ((r ∨ v) ∨ p))) ↔ (¬((s ∧ ¬p)) ∨ ¬(((r ∨ v) ∨ p))))
(¬((((u ∧ s) ∧ q) ∨ (s ∧ ¬q))) ↔ (¬(((u ∧ s) ∧ q)) ∧ ¬((s ∧ ¬q))))
(¬(((¬p ∨ r) ∨ ((v ∨ q) ∨ s))) ↔ (¬((¬p ∨ r)) ∧ ¬(((v ∨ q) ∨ s))))
(¬((((q ∧ u) ∧ v) ∨ (¬r ∨ s))) ↔ (¬(((q ∧ u) ∧ v)) ∧ ¬((¬r ∨ s))))
((((t ∧ r) ∧ u) ∨ (((t ∧ r) ∧ u) ∧ (t → (p ∨ q)))) ↔ ((t ∧ r) ∧ u))
((u → (r ∨ t)) → (((u ∧ t) ∧ s) → ((u → (r ∨ t)) ∧ ((u ∧ t) ∧ s))))
((((r ∧ s) ∧ p) ∧ (((r ∧ s) ∧ p) ∨ ((t ∨ u) ∨ v))) ↔ ((r ∧ s) ∧ p))
(((s ∧ (p ∨ v)) ∧ (t ∨ (s ∧ r))) → ((t ∨ (s ∧ r)) ∧ (s ∧ (p ∨ v))))
((((t ∧ u) → q) ∧ ((t ∧ p) ∧ q)) → (((t ∧ p) ∧ q) ∧ ((t ∧ u) → q)))
((((u ∧ t) → r) ↔ (s → (p ∨ u))) → (((u ∧ t) → r) → (s → (p ∨ u))))
(((s ∧ (q ∨ t)) ∧ ((s ∧ (q ∨ t)) ∨ (r → (s ∨ v)))) ↔ (s ∧ (q ∨ t)))
((((v ∨ p) ∨ t) ↔ (s → (t ∨ v))) → ((s → (t ∨ v)) → ((v ∨ p) ∨ t)))
(((v ∨ (s ∧ u)) ∨ ((u ∨ p) ∨ s)) → (((u ∨ p) ∨ s) ∨ (v ∨ (s ∧ u))))
(((s ∨ t) ∨ u) → (((r ∧ s) ∧ t) → (((s ∨ t) ∨ u) ∧ ((r ∧ s) ∧ t))))
(((((t ∧ q) → r) → (q ∨ (t ∧ r))) ∧ ((t ∧ q) → r)) → (q ∨ (t ∧ r)))
((((u ∧ v) → q) ∨ ((s ∧ t) ∧ r)) → (((s ∧ t) ∧ r) ∨ ((u ∧ v) → q)))
((((q ∧ s) → p) ∨ (((q ∧ s) → p) ∧ ((v ∧ r) → q))) ↔ ((q ∧ s) → p))
((((p ∧ (t ∨ s)) → (s → (q ∨ u))) ∧ (p ∧ (t ∨ s))) → (s → (q ∨ u)))
((((s ∨ r) ∨ t) ∧ (r ∨ (s ∧ t))) → ((r ∨ (s ∧ t)) ∧ ((s ∨ r) ∨ t)))
(((v ∧ (p ∨ u)) ∧ ((v ∧ (p ∨ u)) ∨ ((s ∧ p) → t))) ↔ (v ∧ (p ∨ u)))
((((q ∨ u) ∨ t) ∨ ((q ∧ r) ∧ p)) → (((q ∧ r) ∧ p) ∨ ((q ∨ u) ∨ t)))
((((r ∨ t) ∨ u) ∨ (p ∧ (q ∨ r))) → ((p ∧ (q ∨ r)) ∨ ((r ∨ t) ∨ u)))
((((s ∧ r) ∧ u) ↔ ((v ∧ s) → t)) → (((v ∧ s) → t) → ((s ∧ r) ∧ u)))
(((u → (s ∨ v)) ∨ ((t ∧ u) ∧ s)) → (((t ∧ u) ∧ s) ∨ (u → (s ∨ v))))
((((p ∧ (q ∨ t)) → (t → (q ∨ u))) ∧ (p ∧ (q ∨ t))) → (t → (q ∨ u)))
((((t ∧ r) ∧ q) ↔ ((v ∧ s) → t)) → (((t ∧ r) ∧ q) → ((v ∧ s) → t)))
(((s ∧ (t ∨ r)) ∧ ((s ∧ (t ∨ r)) ∨ ((s ∨ u) ∨ t))) ↔ (s ∧ (t ∨ r)))
(((q ∧ (r ∨ t)) ∧ ((q ∧ (r ∨ t)) ∨ ((v ∧ r) ∧ u))) ↔ (q ∧ (r ∨ t)))
((((v ∨ p) ∨ s) ∨ (((v ∨ p) ∨ s) ∧ ((q ∧ r) → p))) ↔ ((v ∨ p) ∨ s))
((((q ∧ p) ∧ u) ∨ (((q ∧ p) ∧ u) ∧ ((r ∨ q) ∨ t))) ↔ ((q ∧ p) ∧ u))
(((s ∨ (q ∧ t)) ↔ (v ∨ (u ∧ r))) → ((s ∨ (q ∧ t)) → (v ∨ (u ∧ r))))
((((u ∧ r) → s) ∧ (((u ∧ r) → s) ∨ ((p ∨ t) ∨ u))) ↔ ((u ∧ r) → s))
(((((t ∧ q) → r) → ((r ∧ q) → p)) ∧ ((t ∧ q) → r)) → ((r ∧ q) → p))
(((q → (u ∨ t)) ↔ ((p ∧ r) → t)) → ((q → (u ∨ t)) → ((p ∧ r) → t)))
((((q ∨ r) ∨ t) ∧ (((q ∨ r) ∨ t) ∨ ((v ∧ r) ∧ u))) ↔ ((q ∨ r) ∨ t))
(((((t ∨ v) ∨ s) → (t ∧ (v ∨ s))) ∧ ((t ∨ v) ∨ s)) → (t ∧ (v ∨ s)))
((((s ∧ q) ∧ t) ↔ ((t ∧ q) → v)) → (((s ∧ q) ∧ t) → ((t ∧ q) → v)))
(((s ∧ (q ∨ u)) ↔ (v ∧ (u ∨ r))) → ((s ∧ (q ∨ u)) → (v ∧ (u ∨ r))))
((((v ∧ r) → u) ∨ (((v ∧ r) → u) ∧ ((u ∧ q) → r))) ↔ ((v ∧ r) → u))
(((p ∧ (u ∨ t)) ↔ ((p ∨ r) ∨ v)) → ((p ∧ (u ∨ t)) → ((p ∨ r) ∨ v)))
(((r ∧ (t ∨ u)) ∧ ((v ∧ s) → p)) → (((v ∧ s) → p) ∧ (r ∧ (t ∨ u))))
((((q ∨ t) ∨ r) ∨ (u → (p ∨ r))) → ((u → (p ∨ r)) ∨ ((q ∨ t) ∨ r)))
(((r ∧ (s ∨ v)) ↔ ((r ∧ u) → t)) → (((r ∧ u) → t) → (r ∧ (s ∨ v))))
((((p ∧ (r ∨ t)) → (p → (t ∨ r))) ∧ (p ∧ (r ∨ t))) → (p → (t ∨ r)))
((((p ∨ t) ∨ r) ∧ (u ∧ (v ∨ r))) → ((u ∧ (v ∨ r)) ∧ ((p ∨ t) ∨ r)))
(((v ∧ (u ∨ r)) ∨ ((s ∨ r) ∨ q)) → (((s ∨ r) ∨ q) ∨ (v ∧ (u ∨ r))))
((((r → (u ∨ v)) → (p → (s ∨ q))) ∧ (r → (u ∨ v))) → (p → (s ∨ q)))
((((s ∧ v) ∧ r) ∧ (q → (p ∨ t))) → ((q → (p ∨ t)) ∧ ((s ∧ v) ∧ r)))
((((v ∧ u) → s) ∧ (s ∨ (t ∧ p))) → ((s ∨ (t ∧ p)) ∧ ((v ∧ u) → s)))
((((u ∨ (s ∧ p)) → ((t ∧ v) → s)) ∧ (u ∨ (s ∧ p))) → ((t ∧ v) → s))
((p ∧ (v ∨ s)) → (((r ∧ q) ∧ t) → ((p ∧ (v ∨ s)) ∧ ((r ∧ q) ∧ t))))
(((u → (r ∨ s)) ↔ (q ∧ (p ∨ u))) → ((q ∧ (p ∨ u)) → (u → (r ∨ s))))
((((r ∨ s) ∨ p) ∨ (((r ∨ s) ∨ p) ∧ (p ∨ (v ∧ q)))) ↔ ((r ∨ s) ∨ p))
(((u → (v ∨ q)) ∧ ((u → (v ∨ q)) ∨ (p → (u ∨ q)))) ↔ (u → (v ∨ q)))
(((s ∧ v) → t) → ((r ∧ (q ∨ u)) → (((s ∧ v) → t) ∧ (r ∧ (q ∨ u)))))
(((q ∧ r) → v) → (((p ∧ s) ∧ r) → (((q ∧ r) → v) ∧ ((p ∧ s) ∧ r))))
((((r ∨ u) ∨ s) ∨ (r → (t ∨ v))) → ((r → (t ∨ v)) ∨ ((r ∨ u) ∨ s)))
(((r ∨ (u ∧ q)) ∨ ((r ∧ q) ∧ v)) → (((r ∧ q) ∧ v) ∨ (r ∨ (u ∧ q))))
(((v ∧ (t ∨ u)) ∨ ((v ∧ (t ∨ u)) ∧ ((p ∧ r) ∧ t))) ↔ (v ∧ (t ∨ u)))
((((r ∨ (p ∧ s)) → ((r ∧ q) ∧ p)) ∧ (r ∨ (p ∧ s))) → ((r ∧ q) ∧ p))
(((((s ∨ r) ∨ p) → (u ∧ (t ∨ q))) ∧ ((s ∨ r) ∨ p)) → (u ∧ (t ∨ q)))
((s ∨ (t ∧ p)) → (((s ∧ r) → v) → ((s ∨ (t ∧ p)) ∧ ((s ∧ r) → v))))
((((p ∨ (q ∧ v)) → (s ∧ (r ∨ p))) → (p ∨ (q ∧ v))) → (p ∨ (q ∧ v)))
((((s ∧ u) → v) ∨ (s ∨ (v ∧ u))) → ((s ∨ (v ∧ u)) ∨ ((s ∧ u) → v)))
(((s → (q ∨ v)) ↔ ((t ∧ p) ∧ r)) → ((s → (q ∨ v)) → ((t ∧ p) ∧ r)))
((((u → (r ∨ p)) → (u ∨ (t ∧ q))) ∧ (u → (r ∨ p))) → (u ∨ (t ∧ q)))
((((v ∧ u) → t) ∨ (((v ∧ u) → t) ∧ ((t ∧ q) ∧ s))) ↔ ((v ∧ u) → t))
((((t ∧ u) → s) ∧ (((t ∧ u) → s) ∨ ((r ∨ t) ∨ v))) ↔ ((t ∧ u) → s))
(((v ∨ (p ∧ t)) ∧ ((v ∨ (p ∧ t)) ∨ (v ∧ (u ∨ q)))) ↔ (v ∨ (p ∧ t)))
((((s ∧ t) ∧ p) ↔ ((v ∧ r) → u)) → (((v ∧ r) → u) → ((s ∧ t) ∧ p)))
(((((t ∨ r) ∨ q) → (p ∧ (q ∨ t))) ∧ ((t ∨ r) ∨ q)) → (p ∧ (q ∨ t)))
(((p → (t ∨ u)) ∧ ((p → (t ∨ u)) ∨ (q ∨ (u ∧ r)))) ↔ (p → (t ∨ u)))
((((t ∧ (p ∨ v)) → (r ∧ (p ∨ s))) ∧ (t ∧ (p ∨ v))) → (r ∧ (p ∨ s)))
(((((p ∧ q) ∧ r) → (p ∨ (q ∧ u))) ∧ ((p ∧ q) ∧ r)) → (p ∨ (q ∧ u)))
(((q ∨ p) ∨ s) → (((v ∨ s) ∨ u) → (((q ∨ p) ∨ s) ∧ ((v ∨ s) ∨ u))))
(((u ∧ (r ∨ p)) ↔ (t ∨ (s ∧ r))) → ((u ∧ (r ∨ p)) → (t ∨ (s ∧ r))))
(((v ∧ (r ∨ p)) ∧ ((v ∧ (r ∨ p)) ∨ (r ∧ (v ∨ q)))) ↔ (v ∧ (r ∨ p)))
((((u ∨ r) ∨ v) ↔ (q ∧ (v ∨ r))) → ((q ∧ (v ∨ r)) → ((u ∨ r) ∨ v)))
(((q → (r ∨ t)) ∧ ((q → (r ∨ t)) ∨ ((r ∧ q) ∧ t))) ↔ (q → (r ∨ t)))
((((t ∧ (q ∨ s)) → ((q ∧ t) → r)) ∧ (t ∧ (q ∨ s))) → ((q ∧ t) → r))
((((p ∧ u) ∧ v) ∨ ((t ∧ s) → r)) → (((t ∧ s) → r) ∨ ((p ∧ u) ∧ v)))
((((s ∧ (t ∨ q)) → ((p ∨ s) ∨ v)) ∧ (s ∧ (t ∨ q))) → ((p ∨ s) ∨ v))
(((v ∧ (u ∨ t)) ∨ ((v ∧ (u ∨ t)) ∧ ((p ∨ s) ∨ u))) ↔ (v ∧ (u ∨ t)))
(((u ↔ (q → (t ∨ s))) ∧ ((q → (t ∨ s)) ↔ (v ↔ q))) → (u ↔ (v ↔ q)))
((v ∧ (p ∨ u)) → (((u ∧ r) → q) → ((v ∧ (p ∨ u)) ∧ ((u ∧ r) → q))))
((((s ∧ p) ∧ q) ∧ (((s ∧ p) ∧ q) ∨ ((s ∧ u) ∧ t))) ↔ ((s ∧ p) ∧ q))
((((r ∧ q) ∧ t) ↔ ((r ∧ t) → q)) → (((r ∧ t) → q) → ((r ∧ q) ∧ t)))
(((r ∧ (q ∨ s)) ∨ ((v ∨ s) ∨ p)) → (((v ∨ s) ∨ p) ∨ (r ∧ (q ∨ s))))
(((u ∧ (t ∨ q)) ∨ ((q ∨ t) ∨ v)) → (((q ∨ t) ∨ v) ∨ (u ∧ (t ∨ q))))
(((((p ∧ v) ∧ u) → (t ∨ (q ∧ p))) ∧ ((p ∧ v) ∧ u)) → (t ∨ (q ∧ p)))
(((((q ∧ p) → u) → (u ∨ (t ∧ v))) → ((q ∧ p) → u)) → ((q ∧ p) → u))
(((s → (v ∨ r)) ↔ (u ∨ (r ∧ t))) → ((u ∨ (r ∧ t)) → (s → (v ∨ r))))
(((q ∧ (t ∨ u)) ∨ ((u ∧ t) → r)) → (((u ∧ t) → r) ∨ (q ∧ (t ∨ u))))
(((v ∨ (t ∧ p)) ∧ ((v ∨ (t ∧ p)) ∨ (t → (s ∨ v)))) ↔ (v ∨ (t ∧ p)))
((((v → (t ∨ q)) → ((u ∨ v) ∨ p)) ∧ (v → (t ∨ q))) → ((u ∨ v) ∨ p))
((((s ∨ r) ∨ t) ∧ (((s ∨ r) ∨ t) ∨ (p ∨ (u ∧ r)))) ↔ ((s ∨ r) ∨ t))
(((q ∧ p) ∧ t) → ((p ∧ (q ∨ r)) → (((q ∧ p) ∧ t) ∧ (p ∧ (q ∨ r)))))
(((r ∧ (u ∨ s)) ↔ (u ∧ (p ∨ t))) → ((r ∧ (u ∨ s)) → (u ∧ (p ∨ t))))
(((v ∧ (s ∨ q)) ∨ ((t ∧ r) → p)) → (((t ∧ r) → p) ∨ (v ∧ (s ∨ q))))
(((u → (p ∨ t)) ∨ ((p ∨ r) ∨ u)) → (((p ∨ r) ∨ u) ∨ (u → (p ∨ t))))
(((q ∧ (v ∨ r)) ↔ (r ∧ (s ∨ v))) → ((r ∧ (s ∨ v)) → (q ∧ (v ∨ r))))
(((q ∨ u) ∨ t) → (((r ∧ p) ∧ v) → (((q ∨ u) ∨ t) ∧ ((r ∧ p) ∧ v))))
(((((q ∨ u) ∨ s) → (s ∧ (t ∨ u))) → ((q ∨ u) ∨ s)) → ((q ∨ u) ∨ s))
(((t ∨ (q ∧ u)) ∧ ((p ∨ s) ∨ u)) → (((p ∨ s) ∨ u) ∧ (t ∨ (q ∧ u))))
((r ∧ (q ∨ p)) → (((v ∨ p) ∨ u) → ((r ∧ (q ∨ p)) ∧ ((v ∨ p) ∨ u))))
(((s ∨ (q ∧ r)) ∧ ((t ∧ q) ∧ v)) → (((t ∧ q) ∧ v) ∧ (s ∨ (q ∧ r))))
(((¬u ∨ r) → (u → (r ∧ (p ∨ t)))) → (((¬u ∨ r) ∧ u) → (r ∧ (p ∨ t))))
((((v ∨ (p ∧ t)) ∨ (u ∨ (q ∧ v))) ∧ ¬((v ∨ (p ∧ t)))) → (u ∨ (q ∧ v)))
(((((r ∧ t) ∧ p) ∨ ((v ∧ q) → u)) ∧ ¬(((v ∧ q) → u))) → ((r ∧ t) ∧ p))
(((((u ∨ s) ∨ v) → (r ∨ v)) ∧ (¬(((u ∨ s) ∨ v)) → (r ∨ v))) → (r ∨ v))
((((r → (v ∨ t)) ∨ ((r ∧ u) → q)) ∧ ¬((r → (v ∨ t)))) → ((r ∧ u) → q))
(((((u ∧ q) ∧ r) ∨ (r → (v ∨ s))) ∧ ¬((r → (v ∨ s)))) → ((u ∧ q) ∧ r))
(((((t ∨ p) ∨ q) ∨ ((t ∧ p) ∧ u)) ∧ ¬(((t ∨ p) ∨ q))) → ((t ∧ p) ∧ u))
(((((p ∧ u) → r) ∨ ((p ∧ t) → s)) ∧ ¬(((p ∧ t) → s))) → ((p ∧ u) → r))
((((u ∨ (r ∧ s)) ∨ (r → (q ∨ p))) ∧ ¬((r → (q ∨ p)))) → (u ∨ (r ∧ s)))
(((t ∧ (q ∨ p)) → (v ∧ (u ∨ q))) ↔ (¬((t ∧ (q ∨ p))) ∨ (v ∧ (u ∨ q))))
((((u ∧ (r ∨ s)) ∨ ((u ∧ t) → v)) ∧ ¬((u ∧ (r ∨ s)))) → ((u ∧ t) → v))
(((((s ∧ v) ∧ u) ∨ ((q ∧ u) → s)) ∧ ¬(((s ∧ v) ∧ u))) → ((q ∧ u) → s))
(((((u ∧ t) → s) ∨ (v ∨ (p ∧ q))) ∧ ¬((v ∨ (p ∧ q)))) → ((u ∧ t) → s))
((((r ∧ (t ∨ u)) ∨ ((v ∧ q) ∧ p)) ∧ ¬(((v ∧ q) ∧ p))) → (r ∧ (t ∨ u)))
((((v ∨ (u ∧ p)) ∨ ((s ∨ v) ∨ q)) ∧ ¬(((s ∨ v) ∨ q))) → (v ∨ (u ∧ p)))
((((s → (u ∨ v)) ∨ (r ∧ (u ∨ t))) ∧ ¬((r ∧ (u ∨ t)))) → (s → (u ∨ v)))
(((((p ∧ q) ∧ v) ∨ (r → (u ∨ q))) ∧ ¬(((p ∧ q) ∧ v))) → (r → (u ∨ q)))
(((((p ∨ t) ∨ v) ∨ (p ∨ (t ∧ u))) ∧ ¬((p ∨ (t ∧ u)))) → ((p ∨ t) ∨ v))
((((q → (u ∨ t)) ∨ (r → (p ∨ u))) ∧ ¬((r → (p ∨ u)))) → (q → (u ∨ t)))
((((p ∨ (s ∧ r)) ∨ ((p ∨ u) ∨ s)) ∧ ¬((p ∨ (s ∧ r)))) → ((p ∨ u) ∨ s))
(((((q ∧ t) ∧ s) ∨ (s → (r ∨ v))) ∧ ¬((s → (r ∨ v)))) → ((q ∧ t) ∧ s))
((((p ∧ s) ∧ q) → ((r ∨ q) ∨ s)) ↔ (¬(((p ∧ s) ∧ q)) ∨ ((r ∨ q) ∨ s)))
((((q ∨ (t ∧ s)) ∨ (q → (s ∨ r))) ∧ ¬((q ∨ (t ∧ s)))) → (q → (s ∨ r)))
((((v → (u ∨ p)) ∨ (s ∨ (v ∧ p))) ∧ ¬((s ∨ (v ∧ p)))) → (v → (u ∨ p)))
((((v ∧ (q ∨ r)) ∨ ((t ∨ u) ∨ r)) ∧ ¬(((t ∨ u) ∨ r))) → (v ∧ (q ∨ r)))
((((r ∧ (u ∨ t)) ∨ ((p ∧ q) → r)) ∧ ¬((r ∧ (u ∨ t)))) → ((p ∧ q) → r))
(((((u ∨ s) ∨ v) ∨ (u ∧ (v ∨ r))) ∧ ¬((u ∧ (v ∨ r)))) → ((u ∨ s) ∨ v))
((((v ∨ q) ∨ r) → (t ∧ (r ∨ s))) ↔ (¬(((v ∨ q) ∨ r)) ∨ (t ∧ (r ∨ s))))
(((p ∨ (s ∧ q)) → ((q ∨ r) ∨ v)) ↔ (¬((p ∨ (s ∧ q))) ∨ ((q ∨ r) ∨ v)))
(((((r ∧ v) → p) ∨ (v ∨ (r ∧ s))) ∧ ¬(((r ∧ v) → p))) → (v ∨ (r ∧ s)))
((((s ∧ (u ∨ r)) ∨ ((v ∨ s) ∨ u)) ∧ ¬((s ∧ (u ∨ r)))) → ((v ∨ s) ∨ u))
((((u ∨ s) ∨ r) → ((q ∨ r) ∨ s)) ↔ (¬(((u ∨ s) ∨ r)) ∨ ((q ∨ r) ∨ s)))
((((v ∧ (p ∨ t)) ∨ ((s ∧ u) ∧ t)) ∧ ¬(((s ∧ u) ∧ t))) → (v ∧ (p ∨ t)))
(((((u ∨ v) ∨ q) ∨ ((s ∨ u) ∨ p)) ∧ ¬(((u ∨ v) ∨ q))) → ((s ∨ u) ∨ p))
((((t ∧ u) → p) → (p ∧ (v ∨ u))) ↔ (¬(((t ∧ u) → p)) ∨ (p ∧ (v ∨ u))))
((((t ∧ s) → p) → ((t ∧ r) → p)) ↔ (¬(((t ∧ s) → p)) ∨ ((t ∧ r) → p)))
(((((p ∧ t) → r) ∨ (t ∨ (q ∧ r))) ∧ ¬(((p ∧ t) → r))) → (t ∨ (q ∧ r)))
((((u ∨ (q ∧ p)) ∨ ((s ∧ v) ∧ q)) ∧ ¬(((s ∧ v) ∧ q))) → (u ∨ (q ∧ p)))
((((r ∧ v) → u) → (s → (p ∨ v))) ↔ (¬(((r ∧ v) → u)) ∨ (s → (p ∨ v))))
((((u → (v ∨ r)) ∨ ((u ∧ p) ∧ r)) ∧ ¬(((u ∧ p) ∧ r))) → (u → (v ∨ r)))
((((q ∨ s) ∨ p) → ((s ∨ v) ∨ q)) ↔ (¬(((q ∨ s) ∨ p)) ∨ ((s ∨ v) ∨ q)))
(((((v ∧ p) → s) ∨ ((p ∨ v) ∨ t)) ∧ ¬(((v ∧ p) → s))) → ((p ∨ v) ∨ t))
(((((r ∧ v) ∧ p) ∨ ((r ∧ s) → u)) ∧ ¬(((r ∧ s) → u))) → ((r ∧ v) ∧ p))
(((((r ∨ p) ∨ v) ∨ ((s ∧ r) ∧ q)) ∧ ¬(((s ∧ r) ∧ q))) → ((r ∨ p) ∨ v))
((((q ∧ (s ∨ t)) ∨ ((s ∧ q) → v)) ∧ ¬(((s ∧ q) → v))) → (q ∧ (s ∨ t)))
(((((r ∧ u) → t) ∨ ((v ∧ q) → r)) ∧ ¬(((r ∧ u) → t))) → ((v ∧ q) → r))
(((t ∧ (v ∨ s)) → ((r ∨ v) ∨ p)) ↔ (¬((t ∧ (v ∨ s))) ∨ ((r ∨ v) ∨ p)))
(((((r ∨ q) ∨ v) ∨ ((r ∨ u) ∨ s)) ∧ ¬(((r ∨ q) ∨ v))) → ((r ∨ u) ∨ s))
(((((s ∧ v) → t) ∨ ((v ∧ s) ∧ r)) ∧ ¬(((s ∧ v) → t))) → ((v ∧ s) ∧ r))
((((p ∧ (v ∨ q)) ∨ (v ∧ (u ∨ q))) ∧ ¬((p ∧ (v ∨ q)))) → (v ∧ (u ∨ q)))
((((s → (r ∨ p)) ∨ ((p ∨ r) ∨ s)) ∧ ¬((s → (r ∨ p)))) → ((p ∨ r) ∨ s))
(((((s ∧ t) ∧ u) ∨ ((r ∧ s) ∧ p)) ∧ ¬(((s ∧ t) ∧ u))) → ((r ∧ s) ∧ p))
(((((t ∨ s) ∨ v) ∨ (t ∨ (s ∧ v))) ∧ ¬((t ∨ (s ∧ v)))) → ((t ∨ s) ∨ v))
((((t ∧ u) → r) → ((r ∧ q) → s)) ↔ (¬(((t ∧ u) → r)) ∨ ((r ∧ q) → s)))
((((v ∧ p) → s) → (s → (p ∨ r))) ↔ (¬(((v ∧ p) → s)) ∨ (s → (p ∨ r))))
((((q → (p ∨ r)) ∨ (q → (u ∨ v))) ∧ ¬((q → (p ∨ r)))) → (q → (u ∨ v)))
((((r → (v ∨ q)) ∨ (v ∨ (t ∧ u))) ∧ ¬((v ∨ (t ∧ u)))) → (r → (v ∨ q)))
(((((r ∧ q) ∧ s) ∨ (u ∨ (t ∧ s))) ∧ ¬((u ∨ (t ∧ s)))) → ((r ∧ q) ∧ s))
((((r → (p ∨ q)) ∨ ((t ∨ r) ∨ v)) ∧ ¬((r → (p ∨ q)))) → ((t ∨ r) ∨ v))
(((((s ∧ q) ∧ p) ∨ ((u ∧ q) ∧ s)) ∧ ¬(((s ∧ q) ∧ p))) → ((u ∧ q) ∧ s))
((((p → (s ∨ t)) ∨ ((p ∧ q) → r)) ∧ ¬((p → (s ∨ t)))) → ((p ∧ q) → r))
((((q ∧ (v ∨ s)) ∨ (u ∧ (q ∨ r))) ∧ ¬((q ∧ (v ∨ s)))) → (u ∧ (q ∨ r)))
((((u ∧ p) → r) → ((s ∧ v) → t)) → (¬(((s ∧ v) → t)) → ¬(((u ∧ p) → r))))
(¬((((t ∧ s) → v) → (u → (t ∨ s)))) ↔ (((t ∧ s) → v) ∧ ¬((u → (t ∨ s)))))
((¬(((r ∨ u) ∨ v)) → ¬((s ∧ (u ∨ t)))) → ((s ∧ (u ∨ t)) → ((r ∨ u) ∨ v)))
(¬((((p ∨ s) ∨ r) → (p ∨ (u ∧ t)))) ↔ (((p ∨ s) ∨ r) ∧ ¬((p ∨ (u ∧ t)))))
((¬(((u ∨ r) ∨ v)) → ¬(((t ∧ u) ∧ q))) → (((t ∧ u) ∧ q) → ((u ∨ r) ∨ v)))
((((u ∨ v) ∨ s) → ((s ∧ r) → v)) → (¬(((s ∧ r) → v)) → ¬(((u ∨ v) ∨ s))))
((((u → (v ∨ t)) → (r → (q ∨ p))) ∧ ¬((r → (q ∨ p)))) → ¬((u → (v ∨ t))))
(((p ∨ (u ∧ v)) → ((v ∧ p) → r)) → (¬(((v ∧ p) → r)) → ¬((p ∨ (u ∧ v)))))
((¬((p ∧ (v ∨ r))) → ¬((q ∨ (r ∧ t)))) → ((q ∨ (r ∧ t)) → (p ∧ (v ∨ r))))
(¬((((p ∧ s) ∧ q) → ((p ∧ u) → r))) ↔ (((p ∧ s) ∧ q) ∧ ¬(((p ∧ u) → r))))
(((((q ∨ v) ∨ u) → (s ∧ ¬q)) ∧ (¬(((q ∨ v) ∨ u)) → (s ∧ ¬q))) → (s ∧ ¬q))
(¬(((t ∨ (u ∧ q)) → ((v ∧ s) → u))) ↔ ((t ∨ (u ∧ q)) ∧ ¬(((v ∧ s) → u))))
(((((s ∨ r) ∨ q) → (v ∨ (p ∧ q))) ∧ ¬((v ∨ (p ∧ q)))) → ¬(((s ∨ r) ∨ q)))
((((v → (s ∨ q)) → (q → (t ∨ p))) ∧ ¬((q → (t ∨ p)))) → ¬((v → (s ∨ q))))
(¬((((v ∧ q) ∧ u) → ((s ∨ p) ∨ r))) ↔ (((v ∧ q) ∧ u) ∧ ¬(((s ∨ p) ∨ r))))
(¬(((t ∧ (q ∨ u)) → (u → (t ∨ s)))) ↔ ((t ∧ (q ∨ u)) ∧ ¬((u → (t ∨ s)))))
((((s ∨ v) ∨ q) → ((s ∨ p) ∨ q)) → (¬(((s ∨ p) ∨ q)) → ¬(((s ∨ v) ∨ q))))
((((u ∨ v) ∨ p) → (u ∧ (v ∨ t))) → (¬((u ∧ (v ∨ t))) → ¬(((u ∨ v) ∨ p))))
((¬(((v ∧ q) ∧ r)) → ¬(((u ∧ r) ∧ t))) → (((u ∧ r) ∧ t) → ((v ∧ q) ∧ r)))
(((((p ∧ r) → u) → ((p ∧ s) → q)) ∧ ¬(((p ∧ s) → q))) → ¬(((p ∧ r) → u)))
(((((p ∨ r) ∨ u) → ((q ∧ r) ∧ s)) ∧ ¬(((q ∧ r) ∧ s))) → ¬(((p ∨ r) ∨ u)))
(¬((((r ∧ p) → q) → ((p ∨ s) ∨ u))) ↔ (((r ∧ p) → q) ∧ ¬(((p ∨ s) ∨ u))))
(((v ∨ (t ∧ s)) → ((u ∧ r) ∧ p)) → (¬(((u ∧ r) ∧ p)) → ¬((v ∨ (t ∧ s)))))
((((q → (p ∨ v)) → (t ∨ (u ∧ v))) ∧ ¬((t ∨ (u ∧ v)))) → ¬((q → (p ∨ v))))
(((((t ∧ r) ∧ p) → ((q ∨ v) ∨ t)) ∧ ¬(((q ∨ v) ∨ t))) → ¬(((t ∧ r) ∧ p)))
(((t → (u ∨ q)) → (r → (p ∨ s))) → (¬((r → (p ∨ s))) → ¬((t → (u ∨ q)))))
(¬(((r ∧ (q ∨ v)) → (p ∨ (v ∧ r)))) ↔ ((r ∧ (q ∨ v)) ∧ ¬((p ∨ (v ∧ r)))))
(¬((((u ∧ q) ∧ v) → (q ∧ (p ∨ v)))) ↔ (((u ∧ q) ∧ v) ∧ ¬((q ∧ (p ∨ v)))))
((¬((v ∧ (s ∨ u))) → ¬(((u ∧ r) → v))) → (((u ∧ r) → v) → (v ∧ (s ∨ u))))
(((((q ∧ u) → p) → ((s ∧ u) ∧ q)) ∧ ¬(((s ∧ u) ∧ q))) → ¬(((q ∧ u) → p)))
((¬(((u ∧ q) → r)) → ¬((u ∨ (s ∧ v)))) → ((u ∨ (s ∧ v)) → ((u ∧ q) → r)))
(¬((((t ∨ p) ∨ u) → (u → (v ∨ p)))) ↔ (((t ∨ p) ∨ u) ∧ ¬((u → (v ∨ p)))))
((((p ∧ q) → t) → ((u ∧ p) → v)) → (¬(((u ∧ p) → v)) → ¬(((p ∧ q) → t))))
((((u ∧ t) ∧ q) ↔ ((u ∨ q) ∨ s)) → (¬(((u ∧ t) ∧ q)) ↔ ¬(((u ∨ q) ∨ s))))
(¬((((r ∧ v) → u) → (s ∨ (p ∧ r)))) ↔ (((r ∧ v) → u) ∧ ¬((s ∨ (p ∧ r)))))
((((v ∧ s) → r) ↔ (q ∧ (u ∨ r))) → (¬(((v ∧ s) → r)) ↔ ¬((q ∧ (u ∨ r)))))
(((((v ∨ p) ∨ s) → ((v ∧ t) → u)) ∧ ¬(((v ∧ t) → u))) → ¬(((v ∨ p) ∨ s)))
(((t ∨ (v ∧ r)) ↔ ((r ∧ q) ∧ s)) → (¬((t ∨ (v ∧ r))) ↔ ¬(((r ∧ q) ∧ s))))
(¬((((s ∨ r) ∨ v) → (q → (v ∨ r)))) ↔ (((s ∨ r) ∨ v) ∧ ¬((q → (v ∨ r)))))
(((v ∧ (r ∨ u)) → (r ∧ (t ∨ s))) → (¬((r ∧ (t ∨ s))) → ¬((v ∧ (r ∨ u)))))
((¬((t → (r ∨ v))) → ¬((s → (q ∨ p)))) → ((s → (q ∨ p)) → (t → (r ∨ v))))
(((((s ∧ p) → v) → ((v ∧ u) ∧ r)) ∧ ¬(((v ∧ u) ∧ r))) → ¬(((s ∧ p) → v)))
((((u ∨ (t ∧ q)) → ((r ∨ v) ∨ u)) ∧ ¬(((r ∨ v) ∨ u))) → ¬((u ∨ (t ∧ q))))
((((u ∨ r) ∨ t) ↔ ((s ∧ q) ∧ t)) → (¬(((u ∨ r) ∨ t)) ↔ ¬(((s ∧ q) ∧ t))))
(((((v ∧ u) ∧ r) → ((q ∨ p) ∨ u)) ∧ ¬(((q ∨ p) ∨ u))) → ¬(((v ∧ u) ∧ r)))
(((v ∨ (t ∧ q)) → ((r ∧ t) ∧ v)) → (¬(((r ∧ t) ∧ v)) → ¬((v ∨ (t ∧ q)))))
((((r ∨ v) ∨ u) ↔ ((s ∧ v) ∧ u)) → (¬(((r ∨ v) ∨ u)) ↔ ¬(((s ∧ v) ∧ u))))
((¬((u → (t ∨ v))) → ¬((t ∨ (q ∧ s)))) → ((t ∨ (q ∧ s)) → (u → (t ∨ v))))
((((r ∧ u) ∧ t) → ((v ∧ r) ∧ q)) → (¬(((v ∧ r) ∧ q)) → ¬(((r ∧ u) ∧ t))))
((((q ∧ u) ∧ t) → (r → (u ∨ s))) → (¬((r → (u ∨ s))) → ¬(((q ∧ u) ∧ t))))
(((((u ∧ q) → t) → (r → (s ∨ q))) ∧ ¬((r → (s ∨ q)))) → ¬(((u ∧ q) → t)))
((((r ∧ p) ∧ t) ↔ (p ∧ (s ∨ t))) → (¬(((r ∧ p) ∧ t)) ↔ ¬((p ∧ (s ∨ t)))))
((((s ∧ u) → p) → ((r ∧ s) ∧ u)) → (¬(((r ∧ s) ∧ u)) → ¬(((s ∧ u) → p))))
(((((v ∧ s) → t) → (q → (r ∨ v))) ∧ ¬((q → (r ∨ v)))) → ¬(((v ∧ s) → t)))
(((((v ∨ q) ∨ s) → ((p ∧ q) → v)) ∧ ¬(((p ∧ q) → v))) → ¬(((v ∨ q) ∨ s)))
((((p ∧ s) ∧ q) ↔ (s ∨ (p ∧ t))) → (¬(((p ∧ s) ∧ q)) ↔ ¬((s ∨ (p ∧ t)))))
(((((s ∧ v) ∧ u) → ((p ∧ v) → t)) ∧ ¬(((p ∧ v) → t))) → ¬(((s ∧ v) ∧ u)))
(((((p ∧ t) ∧ s) → (q → (r ∨ p))) ∧ ¬((q → (r ∨ p)))) → ¬(((p ∧ t) ∧ s)))
(((p ∧ (v ∨ r)) ↔ ((s ∧ r) → t)) → (¬((p ∧ (v ∨ r))) ↔ ¬(((s ∧ r) → t))))
((((p ∨ (v ∧ s)) → (p ∨ (q ∧ s))) ∧ ¬((p ∨ (q ∧ s)))) → ¬((p ∨ (v ∧ s))))
(¬(((p → (s ∨ r)) ∧ ((s ∧ q) ∧ u))) ↔ (¬((p → (s ∨ r))) ∨ ¬(((s ∧ q) ∧ u))))
(¬(((r → (v ∨ u)) ∧ ((s ∧ p) → v))) ↔ (¬((r → (v ∨ u))) ∨ ¬(((s ∧ p) → v))))
(¬(((r → (q ∨ s)) ∨ (p ∧ (v ∨ t)))) ↔ (¬((r → (q ∨ s))) ∧ ¬((p ∧ (v ∨ t)))))
(¬((((t ∧ u) → v) ∨ (t → (u ∨ q)))) ↔ (¬(((t ∧ u) → v)) ∧ ¬((t → (u ∨ q)))))
(¬(((v ∨ (u ∧ s)) ∨ ((q ∧ u) ∧ s))) ↔ (¬((v ∨ (u ∧ s))) ∧ ¬(((q ∧ u) ∧ s))))
(¬(((s ∨ (t ∧ u)) ∧ ((s ∧ u) ∧ t))) ↔ (¬((s ∨ (t ∧ u))) ∨ ¬(((s ∧ u) ∧ t))))
(¬(((s ∧ (p ∨ q)) ∨ ((s ∧ u) ∧ v))) ↔ (¬((s ∧ (p ∨ q))) ∧ ¬(((s ∧ u) ∧ v))))
(¬((((p ∧ q) ∧ s) ∧ (p ∧ (s ∨ u)))) ↔ (¬(((p ∧ q) ∧ s)) ∨ ¬((p ∧ (s ∨ u)))))
(¬(((p ∧ (t ∨ r)) ∧ ((v ∨ r) ∨ u))) ↔ (¬((p ∧ (t ∨ r))) ∨ ¬(((v ∨ r) ∨ u))))
(¬((((q ∧ p) ∧ v) ∧ ((u ∧ s) → r))) ↔ (¬(((q ∧ p) ∧ v)) ∨ ¬(((u ∧ s) → r))))
(¬((((s ∧ u) ∧ v) ∨ (r → (s ∨ p)))) ↔ (¬(((s ∧ u) ∧ v)) ∧ ¬((r → (s ∨ p)))))
(¬(((q ∧ (r ∨ p)) ∧ (r ∨ (u ∧ p)))) ↔ (¬((q ∧ (r ∨ p))) ∨ ¬((r ∨ (u ∧ p)))))
(¬(((p → (u ∨ s)) ∧ (q ∧ (v ∨ u)))) ↔ (¬((p → (u ∨ s))) ∨ ¬((q ∧ (v ∨ u)))))
(¬(((s ∨ (q ∧ r)) ∨ (q → (r ∨ v)))) ↔ (¬((s ∨ (q ∧ r))) ∧ ¬((q → (r ∨ v)))))
(¬(((p ∧ (q ∨ s)) ∨ ((p ∨ q) ∨ v))) ↔ (¬((p ∧ (q ∨ s))) ∧ ¬(((p ∨ q) ∨ v))))
(¬(((s ∧ (r ∨ v)) ∧ ((v ∧ u) → s))) ↔ (¬((s ∧ (r ∨ v))) ∨ ¬(((v ∧ u) → s))))
(¬(((q → (p ∨ u)) ∨ (p ∧ (s ∨ v)))) ↔ (¬((q → (p ∨ u))) ∧ ¬((p ∧ (s ∨ v)))))
(¬(((v ∨ (r ∧ t)) ∨ ((q ∨ v) ∨ t))) ↔ (¬((v ∨ (r ∧ t))) ∧ ¬(((q ∨ v) ∨ t))))
(¬(((p → (u ∨ q)) ∨ (p ∧ (u ∨ t)))) ↔ (¬((p → (u ∨ q))) ∧ ¬((p ∧ (u ∨ t)))))
(¬((((v ∧ q) → p) ∧ ((u ∧ t) ∧ p))) ↔ (¬(((v ∧ q) → p)) ∨ ¬(((u ∧ t) ∧ p))))
(¬((((r ∨ t) ∨ u) ∧ (v ∧ (s ∨ q)))) ↔ (¬(((r ∨ t) ∨ u)) ∨ ¬((v ∧ (s ∨ q)))))
(¬(((p → (q ∨ s)) ∧ (v → (p ∨ q)))) ↔ (¬((p → (q ∨ s))) ∨ ¬((v → (p ∨ q)))))
(¬(((q → (r ∨ v)) ∨ ((t ∨ v) ∨ r))) ↔ (¬((q → (r ∨ v))) ∧ ¬(((t ∨ v) ∨ r))))
(¬((((p ∨ v) ∨ u) ∧ (u ∧ (p ∨ v)))) ↔ (¬(((p ∨ v) ∨ u)) ∨ ¬((u ∧ (p ∨ v)))))
(¬(((p → (q ∨ s)) ∧ (q ∨ (p ∧ s)))) ↔ (¬((p → (q ∨ s))) ∨ ¬((q ∨ (p ∧ s)))))
(((((u ∧ t) → r) ∧ v) → ((u ∨ r) ∨ p)) → (((u ∧ t) → r) → (v → ((u ∨ r) ∨ p))))
(((v → s) → ((v → (r ∨ q)) → (r ↔ t))) → (((v → s) ∧ (v → (r ∨ q))) → (r ↔ t)))
((((u ∨ r) ∧ ((q ∨ t) ∨ s)) → (v ∧ s)) → ((u ∨ r) → (((q ∨ t) ∨ s) → (v ∧ s))))
(((((s ∧ t) ∧ p) ∨ ((q ∨ r) ∨ p)) ∨ u) → (((s ∧ t) ∧ p) ∨ (((q ∨ r) ∨ p) ∨ u)))
((((s ∨ u) ∨ (t ↔ v)) ∨ (s → (v ∨ q))) → ((s ∨ u) ∨ ((t ↔ v) ∨ (s → (v ∨ q)))))
((((r ∨ u) → (r ↔ t)) ∧ ((r ↔ t) → (q → (p ∨ u)))) → ((r ∨ u) → (q → (p ∨ u))))
((((r ↔ q) → ((r ∧ p) ∧ q)) ∧ (((r ∧ p) ∧ q) → (t ↔ v))) → ((r ↔ q) → (t ↔ v)))
((((q ∧ s) ∧ v) → ((q ∨ u) → (r ∧ t))) → ((((q ∧ s) ∧ v) ∧ (q ∨ u)) → (r ∧ t)))
((((t ∧ q) → p) ∨ ((t → v) ∨ (u → t))) → ((((t ∧ q) → p) ∨ (t → v)) ∨ (u → t)))
((((u ∧ p) → (q ∧ r)) ∧ ((q ∧ r) → (u → (t ∨ q)))) → ((u ∧ p) → (u → (t ∨ q))))
(((((t ∧ s) ∧ p) ∧ (t ∧ v)) ∧ (¬u ∨ r)) → (((t ∧ s) ∧ p) ∧ ((t ∧ v) ∧ (¬u ∨ r))))
((((¬v ∨ q) ∧ (v ∧ t)) ∧ (p ∨ (t ∧ r))) → ((¬v ∨ q) ∧ ((v ∧ t) ∧ (p ∨ (t ∧ r)))))
(((((r ∨ t) ∨ s) ∧ ¬t) ∧ ((s ∨ v) ∨ r)) → (((r ∨ t) ∨ s) ∧ (¬t ∧ ((s ∨ v) ∨ r))))
(((((u ∨ ((s ∧ p) → v)) ∨ ((p ∨ s) ∨ v)) ∧ ¬u) ∧ ¬(((s ∧ p) → v))) → ((p ∨ s) ∨ v))
((((p ∧ ¬u) → (s ∧ (t ∨ v))) ∧ ((s ∧ (t ∨ v)) → (¬r ∨ v))) → ((p ∧ ¬u) → (¬r ∨ v)))
((((s → (p ∨ r)) → (q ∨ r)) ∧ ((q ∨ r) → (s → (p ∨ r)))) → ((s → (p ∨ r)) ↔ (q ∨ r)))
((((v → (t ∨ q)) → (u ∧ t)) ∧ ((u ∧ t) → (v → (t ∨ q)))) → ((v → (t ∨ q)) ↔ (u ∧ t)))
((((((v ∨ p) ∨ t) → u) ∧ ((s ∨ (v ∧ q)) → u)) ∧ (((v ∨ p) ∨ t) ∨ (s ∨ (v ∧ q)))) → u)
(((((t ∧ s) ∧ p) → (u → t)) ∧ ((u → t) → ((t ∧ s) ∧ p))) → (((t ∧ s) ∧ p) ↔ (u → t)))
((((v ∧ s) → (q ∨ (t ∧ v))) ∧ ((q ∨ (t ∧ v)) → (v ∧ s))) → ((v ∧ s) ↔ (q ∨ (t ∧ v))))
((((r ∧ u) → (v → (s ∨ p))) ∧ ((v → (s ∨ p)) → (r ∧ u))) → ((r ∧ u) ↔ (v → (s ∨ p))))
((((q ↔ u) → ((s ∧ v) ∧ t)) ∧ (((s ∧ v) ∧ t) → (q ↔ u))) → ((q ↔ u) ↔ ((s ∧ v) ∧ t)))
((((q ↔ t) → (v ∨ (u ∧ r))) ∧ ((v ∨ (u ∧ r)) → (q ↔ t))) → ((q ↔ t) ↔ (v ∨ (u ∧ r))))
((((s → (q ∨ v)) → (s ∧ v)) ∧ ((s ∧ v) → (s → (q ∨ v)))) → ((s → (q ∨ v)) ↔ (s ∧ v)))
((((s ∨ q) → ((u ∧ s) → t)) ∧ (((u ∧ s) → t) → (s ∨ q))) → ((s ∨ q) ↔ ((u ∧ s) → t)))
(((((p ∧ t) → q) → ((t ∧ v) → q)) ∧ (¬(((p ∧ t) → q)) → ((t ∧ v) → q))) → ((t ∧ v) → q))
(((((t ∧ s) → v) → ((t ∧ v) ∧ r)) ∧ (¬(((t ∧ s) → v)) → ((t ∧ v) ∧ r))) → ((t ∧ v) ∧ r))
(((((v ∨ s) ∨ t) → (r ∧ ¬t)) ∧ ((r ∧ ¬t) → ((v ∨ s) ∨ t))) → (((v ∨ s) ∨ t) ↔ (r ∧ ¬t)))
(((((v ∧ u) → p) → (r ∧ (q ∨ v))) ∧ (¬(((v ∧ u) → p)) → (r ∧ (q ∨ v)))) → (r ∧ (q ∨ v)))
(((((q ∧ v) ∧ p) → (t ∧ ¬p)) ∧ ((t ∧ ¬p) → ((q ∧ v) ∧ p))) → (((q ∧ v) ∧ p) ↔ (t ∧ ¬p)))
(((((s ∧ t) → v) → ((s ∧ t) → u)) ∧ (¬(((s ∧ t) → v)) → ((s ∧ t) → u))) → ((s ∧ t) → u))
((((q ∧ (r ∨ s)) → ((s ∧ q) → u)) ∧ (¬((q ∧ (r ∨ s))) → ((s ∧ q) → u))) → ((s ∧ q) → u))
((((u → (q ∨ v)) → ((s ∧ v) ∧ t)) ∧ (¬((u → (q ∨ v))) → ((s ∧ v) ∧ t))) → ((s ∧ v) ∧ t))
((((t ∧ ¬r) → (q ∨ (s ∧ v))) ∧ ((q ∨ (s ∧ v)) → (t ∧ ¬r))) → ((t ∧ ¬r) ↔ (q ∨ (s ∧ v))))
(((((s ∨ q) ∨ t) → ((r ∧ u) ∧ p)) ∧ (¬(((s ∨ q) ∨ t)) → ((r ∧ u) ∧ p))) → ((r ∧ u) ∧ p))
((((¬p ∨ t) → (r → (p ∨ u))) ∧ ((r → (p ∨ u)) → (¬p ∨ t))) → ((¬p ∨ t) ↔ (r → (p ∨ u))))
((((q ∨ (t ∧ r)) → (¬s ∨ r)) ∧ ((¬s ∨ r) → (q ∨ (t ∧ r)))) → ((q ∨ (t ∧ r)) ↔ (¬s ∨ r)))
((((((¬r ∨ u) ∨ ((v ∨ s) ∨ t)) ∨ (p ∧ ¬u)) ∧ ¬((¬r ∨ u))) ∧ ¬(((v ∨ s) ∨ t))) → (p ∧ ¬u))
((((v → (u ∨ p)) ∨ ((s ∧ t) ∧ u)) ∨ (u → r)) → ((v → (u ∨ p)) ∨ (((s ∧ t) ∧ u) ∨ (u → r))))
((((s ∨ v) ∨ p) ∧ ((q ∨ u) ∧ ((q ∧ v) ∧ u))) → ((((s ∨ v) ∨ p) ∧ (q ∨ u)) ∧ ((q ∧ v) ∧ u)))
(((s ∧ q) → (((t ∨ q) ∨ v) → ((r ∨ t) ∨ q))) → (((t ∨ q) ∨ v) → ((s ∧ q) → ((r ∨ t) ∨ q))))
(((s ∧ (r ∨ t)) ∨ ((q ∨ r) ∨ ((p ∧ r) ∧ v))) → (((s ∧ (r ∨ t)) ∨ (q ∨ r)) ∨ ((p ∧ r) ∧ v)))
(((s → (p ∨ r)) ∧ ((r → p) ∧ (t → (q ∨ r)))) → (((s → (p ∨ r)) ∧ (r → p)) ∧ (t → (q ∨ r))))
(((((r ∨ p) ∨ u) ∧ (t ∨ r)) → (r ∧ (s ∨ t))) → (((r ∨ p) ∨ u) → ((t ∨ r) → (r ∧ (s ∨ t)))))
(((v → s) ∨ ((q ∨ (t ∧ r)) ∨ (t ∨ (q ∧ s)))) → (((v → s) ∨ (q ∨ (t ∧ r))) ∨ (t ∨ (q ∧ s))))
((((q ↔ s) ∧ ((s ∨ u) ∨ t)) → ((t ∧ q) ∧ s)) → ((q ↔ s) → (((s ∨ u) ∨ t) → ((t ∧ q) ∧ s))))
(((v ∧ p) ∨ ((q ∧ (u ∨ s)) ∨ ((v ∧ p) ∧ u))) → (((v ∧ p) ∨ (q ∧ (u ∨ s))) ∨ ((v ∧ p) ∧ u)))
((((v ∧ t) → u) ∧ ((q ∨ s) ∧ (u ∧ (s ∨ v)))) → ((((v ∧ t) → u) ∧ (q ∨ s)) ∧ (u ∧ (s ∨ v))))
((((v ∧ (s ∨ p)) ∨ (q ∨ s)) ∨ ((p ∧ v) ∧ r)) → ((v ∧ (s ∨ p)) ∨ ((q ∨ s) ∨ ((p ∧ v) ∧ r))))
((((u ∧ q) → t) → ((r → s) → (r ∨ (p ∧ s)))) → ((((u ∧ q) → t) ∧ (r → s)) → (r ∨ (p ∧ s))))
((((v ∨ u) → (u ∨ (p ∧ v))) ∧ ((u ∨ (p ∧ v)) → (r → (u ∨ q)))) → ((v ∨ u) → (r → (u ∨ q))))
((((r ∧ q) ∧ v) ∧ ((v ∧ u) ∧ (s ∨ (t ∧ q)))) → ((((r ∧ q) ∧ v) ∧ (v ∧ u)) ∧ (s ∨ (t ∧ q))))
(((((v ∧ p) ∧ t) → (r ∧ u)) ∧ ((r ∧ u) → ((q ∧ r) ∧ s))) → (((v ∧ p) ∧ t) → ((q ∧ r) ∧ s)))
((((s → q) → ((v ∨ p) ∨ q)) ∧ (((v ∨ p) ∨ q) → (v ∧ (u ∨ t)))) → ((s → q) → (v ∧ (u ∨ t))))
((((u ∧ p) ∨ (q ∨ (r ∧ u))) ∨ ((q ∨ r) ∨ p)) → ((u ∧ p) ∨ ((q ∨ (r ∧ u)) ∨ ((q ∨ r) ∨ p))))
(((((t ∧ q) ∧ r) ∧ (t ∧ p)) ∧ (q ∧ (t ∨ p))) → (((t ∧ q) ∧ r) ∧ ((t ∧ p) ∧ (q ∧ (t ∨ p)))))
((((r ∧ u) ↔ ((t ∧ r) ∧ v)) ∧ (((t ∧ r) ∧ v) ↔ ((r ∨ v) ∨ u))) → ((r ∧ u) ↔ ((r ∨ v) ∨ u)))
((((t ∨ q) ∧ ((s ∧ p) → u)) ∧ ((r ∧ t) → s)) → ((t ∨ q) ∧ (((s ∧ p) → u) ∧ ((r ∧ t) → s))))
(((p ∧ u) → ((t → (s ∨ u)) → ((p ∧ t) ∧ s))) → (((p ∧ u) ∧ (t → (s ∨ u))) → ((p ∧ t) ∧ s)))
((((r ∨ (u ∧ v)) ∧ ((r ∧ s) ∧ u)) → (u ∧ s)) → ((r ∨ (u ∧ v)) → (((r ∧ s) ∧ u) → (u ∧ s))))
((((u ∨ (v ∧ s)) ∧ (u ∧ r)) → ((q ∧ p) → s)) → ((u ∨ (v ∧ s)) → ((u ∧ r) → ((q ∧ p) → s))))
(((((s ∧ q) ∧ p) → (q → (r ∨ v))) ∧ ((q → (r ∨ v)) → (p ∨ u))) → (((s ∧ q) ∧ p) → (p ∨ u)))
(((t ∧ (q ∨ r)) → ((r ↔ t) → (r ∧ (s ∨ q)))) → ((r ↔ t) → ((t ∧ (q ∨ r)) → (r ∧ (s ∨ q)))))
((((t ∧ (r ∨ u)) ∧ (v ∧ (t ∨ p))) ∧ (q → s)) → ((t ∧ (r ∨ u)) ∧ ((v ∧ (t ∨ p)) ∧ (q → s))))
((((u ∨ s) ∨ p) → (((u ∧ v) ∧ p) → (u ∨ r))) → ((((u ∨ s) ∨ p) ∧ ((u ∧ v) ∧ p)) → (u ∨ r)))
((((t ∧ v) ∨ ((p ∨ q) ∨ r)) ∨ (s ∨ (q ∧ v))) → ((t ∧ v) ∨ (((p ∨ q) ∨ r) ∨ (s ∨ (q ∧ v)))))
(((s ↔ v) ↔ ((p ∨ v) ∨ s)) → (((s ↔ v) ∨ (v ∧ (s ∨ r))) ↔ (((p ∨ v) ∨ s) ∨ (v ∧ (s ∨ r)))))
(((u ∧ (p ∨ t)) ∧ ((v ∨ (q ∧ p)) ∧ (r ∨ s))) → (((u ∧ (p ∨ t)) ∧ (v ∨ (q ∧ p))) ∧ (r ∨ s)))
(((((q ∨ t) ∨ r) → (t → (r ∨ s))) ∧ ((t → (r ∨ s)) → (s ∨ t))) → (((q ∨ t) ∨ r) → (s ∨ t)))
(((r ↔ s) ∨ ((p ∨ (u ∧ v)) ∨ ((v ∨ p) ∨ t))) → (((r ↔ s) ∨ (p ∨ (u ∧ v))) ∨ ((v ∨ p) ∨ t)))
(((((p ∧ s) → q) → (u ∧ p)) ∧ ((u ∧ p) → ((q ∨ p) ∨ v))) → (((p ∧ s) → q) → ((q ∨ p) ∨ v)))
((((t ↔ q) ∧ (q → (v ∨ s))) ∧ (r → (q ∨ p))) → ((t ↔ q) ∧ ((q → (v ∨ s)) ∧ (r → (q ∨ p)))))
(((s ↔ q) ∧ (((t ∨ u) ∨ s) ∧ (r → (t ∨ q)))) → (((s ↔ q) ∧ ((t ∨ u) ∨ s)) ∧ (r → (t ∨ q))))
((((v ∧ p) ∧ s) ∧ (((p ∨ t) ∨ q) ∧ (u ∨ r))) → ((((v ∧ p) ∧ s) ∧ ((p ∨ t) ∨ q)) ∧ (u ∨ r)))
(((q → (p ∨ r)) ∨ ((v ↔ t) ∨ (v → (s ∨ t)))) → (((q → (p ∨ r)) ∨ (v ↔ t)) ∨ (v → (s ∨ t))))
((((u → (t ∨ q)) → (t ∧ s)) ∧ ((t ∧ s) → ((s ∨ u) ∨ r))) → ((u → (t ∨ q)) → ((s ∨ u) ∨ r)))
((((u ∨ (q ∧ p)) → (u → (t ∨ v))) ∧ ((u → (t ∨ v)) → (u ∨ s))) → ((u ∨ (q ∧ p)) → (u ∨ s)))
((((v ∧ u) → t) ↔ (p → s)) → ((((v ∧ u) → t) ∧ (u ∨ (p ∧ t))) ↔ ((p → s) ∧ (u ∨ (p ∧ t)))))
((((p ↔ r) ∧ ((u ∨ v) ∨ s)) ∧ (p ∧ (q ∨ u))) → ((p ↔ r) ∧ (((u ∨ v) ∨ s) ∧ (p ∧ (q ∨ u)))))
((((t ∧ u) ∧ v) ↔ (q ∧ t)) → ((((t ∧ u) ∧ v) ∨ (v ∨ (t ∧ s))) ↔ ((q ∧ t) ∨ (v ∨ (t ∧ s)))))
((((q ∨ p) ∨ v) ∧ ((p ∧ v) ∧ (v ∧ (u ∨ r)))) → ((((q ∨ p) ∨ v) ∧ (p ∧ v)) ∧ (v ∧ (u ∨ r))))
(((s ↔ v) → ((v ∧ (s ∨ r)) → (t ∨ (r ∧ p)))) → ((v ∧ (s ∨ r)) → ((s ↔ v) → (t ∨ (r ∧ p)))))
(((t ∧ ¬s) → ((u ∧ (t ∨ p)) → ((r ∧ t) ∧ v))) → (((t ∧ ¬s) ∧ (u ∧ (t ∨ p))) → ((r ∧ t) ∧ v)))
((((s ∨ v) ∨ q) ∧ ((¬q ∨ s) ∧ ((t ∧ v) → r))) → ((((s ∨ v) ∨ q) ∧ (¬q ∨ s)) ∧ ((t ∧ v) → r)))
((((t ∧ v) → s) ∧ (((q ∧ p) → v) ∧ (t ∧ ¬r))) → ((((t ∧ v) → s) ∧ ((q ∧ p) → v)) ∧ (t ∧ ¬r)))
((((t ∧ v) ∧ u) ∧ ((¬r ∨ t) ∧ ((u ∧ t) → s))) → ((((t ∧ v) ∧ u) ∧ (¬r ∨ t)) ∧ ((u ∧ t) → s)))
((((r ∧ ¬q) ∨ ((u ∧ t) → s)) ∨ (s ∨ (r ∧ t))) → ((r ∧ ¬q) ∨ (((u ∧ t) → s) ∨ (s ∨ (r ∧ t)))))
((((v ↔ q) → ((s ∧ p) ∧ q)) ∧ ((v ↔ q) → (s ∧ ¬u))) → ((v ↔ q) → (((s ∧ p) ∧ q) ∧ (s ∧ ¬u))))
(((((t ∧ s) ∧ q) ∧ (v ∧ ¬u)) → (q ∨ (r ∧ p))) → (((t ∧ s) ∧ q) → ((v ∧ ¬u) → (q ∨ (r ∧ p)))))
((((s ∨ (r ∧ v)) → ((p ∧ s) → v)) ∧ (((p ∧ s) → v) → (¬t ∨ v))) → ((s ∨ (r ∧ v)) → (¬t ∨ v)))
((((p ∧ v) → q) → ((p → (u ∨ q)) → (p ∧ ¬q))) → ((p → (u ∨ q)) → (((p ∧ v) → q) → (p ∧ ¬q))))
((((t ∨ (q ∧ p)) → (t → (r ∨ v))) ∧ ((t → (r ∨ v)) → (s ∧ ¬q))) → ((t ∨ (q ∧ p)) → (s ∧ ¬q)))
((((¬s ∨ t) → (p ∨ (v ∧ s))) ∧ ((p ∨ (v ∧ s)) → (q ∧ (p ∨ r)))) → ((¬s ∨ t) → (q ∧ (p ∨ r))))
(((r ∧ ¬u) ∨ (((v ∧ t) → s) ∨ (q ∨ (r ∧ s)))) → (((r ∧ ¬u) ∨ ((v ∧ t) → s)) ∨ (q ∨ (r ∧ s))))
((((q → (u ∨ s)) ∧ (r → (v ∨ t))) ∧ (q ∧ ¬r)) → ((q → (u ∨ s)) ∧ ((r → (v ∨ t)) ∧ (q ∧ ¬r))))
(((((¬r ∨ p) → (v ∧ u)) ∧ ((q → (u ∨ s)) → (v ∧ u))) ∧ ((¬r ∨ p) ∨ (q → (u ∨ s)))) → (v ∧ u))
(((t ∨ (p ∧ v)) ↔ ((t ∧ p) → s)) → (((t ∨ (p ∧ v)) ∧ (¬r ∨ u)) ↔ (((t ∧ p) → s) ∧ (¬r ∨ u))))
((((s ∧ ¬q) ∧ (u ∧ (t ∨ r))) → (t → (s ∨ u))) → ((s ∧ ¬q) → ((u ∧ (t ∨ r)) → (t → (s ∨ u)))))
(((t ∧ ¬q) ↔ ((q ∧ v) ∧ r)) → (((t ∧ ¬q) ∧ ((u ∧ r) ∧ p)) ↔ (((q ∧ v) ∧ r) ∧ ((u ∧ r) ∧ p))))
((((¬t ∨ q) ↔ ((s ∨ u) ∨ v)) ∧ (((s ∨ u) ∨ v) ↔ ((t ∧ r) ∧ s))) → ((¬t ∨ q) ↔ ((t ∧ r) ∧ s)))
(((s ∧ (u ∨ v)) → ((r ∧ ¬t) → ((q ∧ p) → v))) → ((r ∧ ¬t) → ((s ∧ (u ∨ v)) → ((q ∧ p) → v))))
((((u ∨ (s ∧ t)) ∧ (q ∧ ¬r)) → ((v ∧ u) ∧ q)) → ((u ∨ (s ∧ t)) → ((q ∧ ¬r) → ((v ∧ u) ∧ q))))
(((r ∨ (s ∧ v)) ∨ ((s ∨ (p ∧ u)) ∨ (s ∧ ¬t))) → (((r ∨ (s ∧ v)) ∨ (s ∨ (p ∧ u))) ∨ (s ∧ ¬t)))
(((((s ∧ r) → v) ∧ (p → (u ∨ t))) → (t ∧ ¬q)) → (((s ∧ r) → v) → ((p → (u ∨ t)) → (t ∧ ¬q))))
(((v ∧ ¬p) ∨ (((q ∧ r) ∧ s) ∨ (u ∨ (r ∧ v)))) → (((v ∧ ¬p) ∨ ((q ∧ r) ∧ s)) ∨ (u ∨ (r ∧ v))))
(((u ∨ (s ∧ q)) ∧ ((¬v ∨ r) ∧ ((v ∧ p) ∧ r))) → (((u ∨ (s ∧ q)) ∧ (¬v ∨ r)) ∧ ((v ∧ p) ∧ r)))
(((¬p ∨ q) ↔ (q ∧ (p ∨ s))) → (((¬p ∨ q) ∧ ((v ∨ q) ∨ s)) ↔ ((q ∧ (p ∨ s)) ∧ ((v ∨ q) ∨ s))))
((((¬t ∨ v) → (v ∧ (s ∨ r))) ∧ ((v ∧ (s ∨ r)) → (p → (v ∨ q)))) → ((¬t ∨ v) → (p → (v ∨ q))))
(((¬v ∨ r) → (((t ∨ v) ∨ u) → ((v ∧ p) → u))) → (((¬v ∨ r) ∧ ((t ∨ v) ∨ u)) → ((v ∧ p) → u)))
(((((t → (r ∨ s)) → ¬v) ∧ ((¬r ∨ v) → (t → u))) ∧ ((t → (r ∨ s)) ∨ (¬r ∨ v))) → (¬v ∨ (t → u)))
((((((p ∨ t) ∨ u) → (¬q ∨ t)) ∧ ((¬s ∨ u) → (¬q ∨ t))) ∧ (((p ∨ t) ∨ u) ∨ (¬s ∨ u))) → (¬q ∨ t))
((((((v → r) ∨ (v ∧ (p ∨ t))) ∨ (t ∨ (s ∧ u))) ∧ ¬((v → r))) ∧ ¬((v ∧ (p ∨ t)))) → (t ∨ (s ∧ u)))
(((((((u ∧ v) → p) ∨ ((t ∨ r) ∨ v)) ∨ (u ∨ q)) ∧ ¬(((u ∧ v) → p))) ∧ ¬(((t ∨ r) ∨ v))) → (u ∨ q))
((((((q ∨ (s ∧ v)) ∨ (s ∧ q)) ∨ (s ∨ (t ∧ u))) ∧ ¬((q ∨ (s ∧ v)))) ∧ ¬((s ∧ q))) → (s ∨ (t ∧ u)))
((((((u ∨ s) ∨ (p → (r ∨ t))) ∨ ((t ∧ v) → u)) ∧ ¬((u ∨ s))) ∧ ¬((p → (r ∨ t)))) → ((t ∧ v) → u))
((((((r ∧ (v ∨ t)) ∨ (r ∧ q)) ∨ ((p ∨ q) ∨ t)) ∧ ¬((r ∧ (v ∨ t)))) ∧ ¬((r ∧ q))) → ((p ∨ q) ∨ t))
((((((q ∧ (s ∨ t)) ∨ ((u ∨ r) ∨ s)) ∨ (p → s)) ∧ ¬((q ∧ (s ∨ t)))) ∧ ¬(((u ∨ r) ∨ s))) → (p → s))
((((((t → (r ∨ q)) ∨ (v ∧ (s ∨ u))) ∨ (p ∨ q)) ∧ ¬((t → (r ∨ q)))) ∧ ¬((v ∧ (s ∨ u)))) → (p ∨ q))
((((v → (s ∨ (u ∧ r))) ∧ (¬v → (s ∨ (u ∧ r)))) ∧ ((s ∨ (u ∧ r)) → ((t ∨ s) ∨ q))) → ((t ∨ s) ∨ q))
(((((((s ∧ q) ∧ t) ∨ (v → (r ∨ t))) ∨ (t ∧ ¬s)) ∧ ¬(((s ∧ q) ∧ t))) ∧ ¬((v → (r ∨ t)))) → (t ∧ ¬s))
(((((((t ∧ r) ∧ s) ∨ (¬v ∨ s)) ∨ ((q ∧ t) ∧ u)) ∧ ¬(((t ∧ r) ∧ s))) ∧ ¬((¬v ∨ s))) → ((q ∧ t) ∧ u))
((((¬r → ((u ∧ v) → q)) ∧ ((q ∧ (v ∨ r)) → ((u ∧ v) → q))) ∧ (¬r ∨ (q ∧ (v ∨ r)))) → ((u ∧ v) → q))
(((((s ∧ ¬u) → ((r ∨ t) ∨ u)) ∧ (¬((s ∧ ¬u)) → ((r ∨ t) ∨ u))) ∧ (((r ∨ t) ∨ u) → (r ∧ v))) → (r ∧ v))
(((((r ∨ v) ∨ s) → (u ∨ (p ∧ r))) ∧ ((u ∨ (p ∧ r)) → ((r ∧ s) → p))) → (((r ∨ v) ∨ s) → ((r ∧ s) → p)))
((((((v ∧ t) ∧ r) → (s ∧ p)) ∧ (((u ∧ t) ∧ s) → (s ∧ p))) ∧ (((v ∧ t) ∧ r) ∨ ((u ∧ t) ∧ s))) → (s ∧ p))
((((t → (p ∨ v)) ∧ ((u ∧ t) ∧ p)) ∧ (v → (t ∨ q))) → ((t → (p ∨ v)) ∧ (((u ∧ t) ∧ p) ∧ (v → (t ∨ q)))))
((((s ∨ r) ∨ u) ∧ ((u ∧ (t ∨ r)) ∧ (s ∧ (q ∨ v)))) → ((((s ∨ r) ∨ u) ∧ (u ∧ (t ∨ r))) ∧ (s ∧ (q ∨ v))))
((((t ∧ r) ∧ s) ↔ ((r ∨ v) ∨ s)) → ((((t ∧ r) ∧ s) ∧ (t ∧ (s ∨ v))) ↔ (((r ∨ v) ∨ s) ∧ (t ∧ (s ∨ v)))))
((((p → (r ∨ q)) ∨ (u → (v ∨ t))) ∨ (q ∧ (v ∨ p))) → ((p → (r ∨ q)) ∨ ((u → (v ∨ t)) ∨ (q ∧ (v ∨ p)))))
(((((v ∧ u) ∧ r) → (v ∨ (u ∧ s))) ∧ ((v ∨ (u ∧ s)) → ((v ∧ t) → q))) → (((v ∧ u) ∧ r) → ((v ∧ t) → q)))
(((((t ∧ p) → v) ∧ ((r ∨ s) ∨ t)) → (v ∧ (r ∨ t))) → (((t ∧ p) → v) → (((r ∨ s) ∨ t) → (v ∧ (r ∨ t)))))
(((((u ∧ p) → r) → ((v ∧ s) ∧ q)) ∧ (((v ∧ s) ∧ q) → ((r ∧ q) → s))) → (((u ∧ p) → r) → ((r ∧ q) → s)))
(((s → (t ∨ p)) ∧ (((t ∧ v) ∧ r) ∧ ((u ∧ p) ∧ q))) → (((s → (t ∨ p)) ∧ ((t ∧ v) ∧ r)) ∧ ((u ∧ p) ∧ q)))
(((((t ∧ q) ∧ r) → ((u ∨ p) ∨ r)) ∧ (((u ∨ p) ∨ r) → (q → (t ∨ v)))) → (((t ∧ q) ∧ r) → (q → (t ∨ v))))
((((r → (v ∨ s)) → ((q ∧ p) → u)) ∧ (((q ∧ p) → u) → (r → (v ∨ s)))) → ((r → (v ∨ s)) ↔ ((q ∧ p) → u)))
((((p ∧ (s ∨ u)) ∧ ((s ∧ q) ∧ t)) ∧ ((r ∧ t) ∧ s)) → ((p ∧ (s ∨ u)) ∧ (((s ∧ q) ∧ t) ∧ ((r ∧ t) ∧ s))))
(((t ∨ (q ∧ r)) ∨ ((s ∨ (v ∧ p)) ∨ (q ∨ (p ∧ t)))) → (((t ∨ (q ∧ r)) ∨ (s ∨ (v ∧ p))) ∨ (q ∨ (p ∧ t))))
(((((q ∧ p) → t) → (s ∧ (q ∨ t))) ∧ ((s ∧ (q ∨ t)) → ((q ∧ p) → s))) → (((q ∧ p) → t) → ((q ∧ p) → s)))
((((v ∧ p) → s) → ((r → (q ∨ s)) → ((v ∨ u) ∨ q))) → ((((v ∧ p) → s) ∧ (r → (q ∨ s))) → ((v ∨ u) ∨ q)))
(((((t ∨ s) ∨ r) ∧ ((p ∧ u) → t)) → ((s ∨ p) ∨ r)) → (((t ∨ s) ∨ r) → (((p ∧ u) → t) → ((s ∨ p) ∨ r))))
((((q ∧ s) → v) → (((t ∨ r) ∨ s) → ((t ∧ u) ∧ v))) → (((t ∨ r) ∨ s) → (((q ∧ s) → v) → ((t ∧ u) ∧ v))))
((((t → (u ∨ r)) ∧ (v → (p ∨ r))) → (r ∧ (s ∨ q))) → ((t → (u ∨ r)) → ((v → (p ∨ r)) → (r ∧ (s ∨ q)))))
(((((r ∨ v) ∨ q) ∧ (r → (q ∨ v))) → (v → (p ∨ r))) → (((r ∨ v) ∨ q) → ((r → (q ∨ v)) → (v → (p ∨ r)))))
((((q ∧ v) ∧ t) → ((r ∨ (u ∧ p)) → ((t ∧ q) ∧ v))) → ((((q ∧ v) ∧ t) ∧ (r ∨ (u ∧ p))) → ((t ∧ q) ∧ v)))
(((((s ∨ q) ∨ p) ∧ ((p ∧ t) → u)) ∧ ((t ∧ s) ∧ r)) → (((s ∨ q) ∨ p) ∧ (((p ∧ t) → u) ∧ ((t ∧ s) ∧ r))))
(((u ∨ (v ∧ s)) → ((r ∨ (s ∧ q)) → ((p ∧ u) → t))) → (((u ∨ (v ∧ s)) ∧ (r ∨ (s ∧ q))) → ((p ∧ u) → t)))
(((((s ∧ q) → v) ↔ (v ∧ (p ∨ s))) ∧ ((v ∧ (p ∨ s)) ↔ (p ∧ (t ∨ r)))) → (((s ∧ q) → v) ↔ (p ∧ (t ∨ r))))
(((t ∨ (p ∧ s)) ↔ (t ∧ (s ∨ p))) → (((t ∨ (p ∧ s)) ∧ ((q ∧ p) ∧ v)) ↔ ((t ∧ (s ∨ p)) ∧ ((q ∧ p) ∧ v))))
((((u ∨ (t ∧ q)) ∧ ((s ∧ q) ∧ t)) → (u ∧ (v ∨ r))) → ((u ∨ (t ∧ q)) → (((s ∧ q) ∧ t) → (u ∧ (v ∨ r)))))
((((r ∨ s) ∨ p) ∧ (((v ∧ q) → r) ∧ ((s ∧ v) ∧ u))) → ((((r ∨ s) ∨ p) ∧ ((v ∧ q) → r)) ∧ ((s ∧ v) ∧ u)))
(((((p ∧ t) ∧ r) ↔ (t ∨ (v ∧ q))) ∧ ((t ∨ (v ∧ q)) ↔ ((q ∧ s) ∧ p))) → (((p ∧ t) ∧ r) ↔ ((q ∧ s) ∧ p)))
((((u → (q ∨ r)) → ((v ∨ r) ∨ s)) ∧ (((v ∨ r) ∨ s) → (u → (q ∨ r)))) → ((u → (q ∨ r)) ↔ ((v ∨ r) ∨ s)))
((((r ∧ t) → s) ∧ (((t ∧ u) → q) ∧ ((r ∧ p) ∧ s))) → ((((r ∧ t) → s) ∧ ((t ∧ u) → q)) ∧ ((r ∧ p) ∧ s)))
(((((v ∨ t) ∨ u) → ((s ∧ r) ∧ v)) ∧ (((s ∧ r) ∧ v) → ((s ∧ t) → u))) → (((v ∨ t) ∨ u) → ((s ∧ t) → u)))
((((t ∨ q) ∨ u) ∨ (((r ∧ v) ∧ u) ∨ (r ∧ (u ∨ v)))) → ((((t ∨ q) ∨ u) ∨ ((r ∧ v) ∧ u)) ∨ (r ∧ (u ∨ v))))
((((u ∧ (r ∨ s)) ∨ ((s ∧ v) ∧ q)) ∨ ((r ∧ t) ∧ u)) → ((u ∧ (r ∨ s)) ∨ (((s ∧ v) ∧ q) ∨ ((r ∧ t) ∧ u))))
(((((s ∧ v) → q) ∧ ((p ∧ r) ∧ u)) → (p → (v ∨ r))) → (((s ∧ v) → q) → (((p ∧ r) ∧ u) → (p → (v ∨ r)))))
((((t ∨ (v ∧ q)) → (q ∨ (p ∧ s))) ∧ ((q ∨ (p ∧ s)) → (r ∨ (p ∧ t)))) → ((t ∨ (v ∧ q)) → (r ∨ (p ∧ t))))
((((t ∨ (p ∧ q)) → (r → (q ∨ u))) ∧ ((r → (q ∨ u)) → (s ∧ (v ∨ t)))) → ((t ∨ (p ∧ q)) → (s ∧ (v ∨ t))))
((((u ∨ r) ∨ s) ∧ (((q ∨ r) ∨ p) ∧ ((q ∨ u) ∨ v))) → ((((u ∨ r) ∨ s) ∧ ((q ∨ r) ∨ p)) ∧ ((q ∨ u) ∨ v)))
((((q ∨ (t ∧ s)) → (u ∨ (s ∧ q))) ∧ ((u ∨ (s ∧ q)) → ((q ∨ p) ∨ v))) → ((q ∨ (t ∧ s)) → ((q ∨ p) ∨ v)))
((((p → (u ∨ s)) ∧ ((q ∨ v) ∨ r)) → ((p ∨ u) ∨ q)) → ((p → (u ∨ s)) → (((q ∨ v) ∨ r) → ((p ∨ u) ∨ q))))
(((p ∧ (u ∨ r)) ↔ (u ∨ (r ∧ v))) → (((p ∧ (u ∨ r)) ∨ ((s ∧ t) → u)) ↔ ((u ∨ (r ∧ v)) ∨ ((s ∧ t) → u))))
((((t ∧ (p ∨ s)) ∧ ((r ∧ p) ∧ s)) → (s → (r ∨ u))) → ((t ∧ (p ∨ s)) → (((r ∧ p) ∧ s) → (s → (r ∨ u)))))
((((v ∧ (q ∨ p)) ↔ (s ∨ (t ∧ q))) ∧ ((s ∨ (t ∧ q)) ↔ (t ∨ (q ∧ u)))) → ((v ∧ (q ∨ p)) ↔ (t ∨ (q ∧ u))))
(((((u ∧ s) ∧ v) → ((v ∨ p) ∨ t)) ∧ (((v ∨ p) ∨ t) → ((u ∧ s) ∧ v))) → (((u ∧ s) ∧ v) ↔ ((v ∨ p) ∨ t)))
((((s ∨ (u ∧ p)) → ((p ∧ q) ∧ v)) ∧ (((p ∧ q) ∧ v) → (s ∨ (u ∧ p)))) → ((s ∨ (u ∧ p)) ↔ ((p ∧ q) ∧ v)))
(((u ∨ (p ∧ r)) ∧ (((r ∧ u) ∧ p) ∧ (s → (p ∨ q)))) → (((u ∨ (p ∧ r)) ∧ ((r ∧ u) ∧ p)) ∧ (s → (p ∨ q))))
((((s → (p ∨ t)) ∧ ((t ∧ u) → r)) ∧ (s → (v ∨ r))) → ((s → (p ∨ t)) ∧ (((t ∧ u) → r) ∧ (s → (v ∨ r)))))
(((((p ∧ q) → t) ∧ (t → (s ∨ v))) ∧ (r ∧ (v ∨ u))) → (((p ∧ q) → t) ∧ ((t → (s ∨ v)) ∧ (r ∧ (v ∨ u)))))
(((r ∧ (v ∨ u)) → (((v ∧ t) → s) → ((r ∧ s) → q))) → (((v ∧ t) → s) → ((r ∧ (v ∨ u)) → ((r ∧ s) → q))))
(((((p ∨ u) ∨ v) ∨ (u → (q ∨ r))) ∨ (r → (q ∨ u))) → (((p ∨ u) ∨ v) ∨ ((u → (q ∨ r)) ∨ (r → (q ∨ u)))))
(((((s ∨ v) ∨ u) ∧ ((u ∨ p) ∨ v)) ∧ (v ∧ (s ∨ p))) → (((s ∨ v) ∨ u) ∧ (((u ∨ p) ∨ v) ∧ (v ∧ (s ∨ p)))))
((((v ∧ q) → r) ↔ (q → (s ∨ t))) → ((((v ∧ q) → r) ∨ ((u ∧ s) → v)) ↔ ((q → (s ∨ t)) ∨ ((u ∧ s) → v))))
(((((u ∨ p) ∨ v) → ((u ∧ p) ∧ v)) ∧ (((u ∧ p) ∧ v) → ((u ∨ p) ∨ v))) → (((u ∨ p) ∨ v) ↔ ((u ∧ p) ∧ v)))
((((s ∨ r) ∨ v) ↔ ((u ∨ t) ∨ s)) → ((((s ∨ r) ∨ v) ∧ (r ∨ (s ∧ t))) ↔ (((u ∨ t) ∨ s) ∧ (r ∨ (s ∧ t)))))
(((((u ∧ s) ∧ q) → (p ∧ (v ∨ s))) ∧ ((p ∧ (v ∨ s)) → ((t ∧ v) → p))) → (((u ∧ s) ∧ q) → ((t ∧ v) → p)))
((((v ∧ (p ∨ s)) → (p ∨ (v ∧ q))) ∧ ((p ∨ (v ∧ q)) → (v ∧ (p ∨ s)))) → ((v ∧ (p ∨ s)) ↔ (p ∨ (v ∧ q))))
((((v ∧ (p ∨ u)) → ((p ∧ q) ∧ t)) ∧ (((p ∧ q) ∧ t) → ((u ∧ s) ∧ r))) → ((v ∧ (p ∨ u)) → ((u ∧ s) ∧ r)))
(((((p ∧ s) → u) ∧ ((p ∧ u) → t)) ∧ (q ∨ (p ∧ t))) → (((p ∧ s) → u) ∧ (((p ∧ u) → t) ∧ (q ∨ (p ∧ t)))))
(((((r ∨ s) ∨ p) ∧ ((p ∧ q) → t)) ∧ ((r ∧ p) → u)) → (((r ∨ s) ∨ p) ∧ (((p ∧ q) → t) ∧ ((r ∧ p) → u))))
(((p → (q ∨ r)) ∨ (((r ∧ p) ∧ t) ∨ (p → (t ∨ u)))) → (((p → (q ∨ r)) ∨ ((r ∧ p) ∧ t)) ∨ (p → (t ∨ u))))
(((((s ∨ t) ∨ u) → ((p ∨ r) ∨ v)) ∧ (((p ∨ r) ∨ v) → ((s ∨ t) ∨ u))) → (((s ∨ t) ∨ u) ↔ ((p ∨ r) ∨ v)))
((((((u ∧ s) → q) → (p ∨ q)) ∧ ((v ∧ (t ∨ p)) → (p ∨ q))) ∧ (((u ∧ s) → q) ∨ (v ∧ (t ∨ p)))) → (p ∨ q))
((((q → (s ∨ u)) ∧ (p ∨ (v ∧ t))) → ((u ∧ t) → r)) → ((q → (s ∨ u)) → ((p ∨ (v ∧ t)) → ((u ∧ t) → r))))
(((((u ∨ r) ∨ t) ∨ (t ∨ (q ∧ p))) ∨ (t ∧ (s ∨ v))) → (((u ∨ r) ∨ t) ∨ ((t ∨ (q ∧ p)) ∨ (t ∧ (s ∨ v)))))
((((v ∨ (t ∧ q)) → (q ∨ (u ∧ s))) ∧ ((q ∨ (u ∧ s)) → (p ∧ (q ∨ v)))) → ((v ∨ (t ∧ q)) → (p ∧ (q ∨ v))))
(((t ∨ (q ∧ r)) ∨ ((v ∧ (p ∨ r)) ∨ (s ∧ (q ∨ p)))) → (((t ∨ (q ∧ r)) ∨ (v ∧ (p ∨ r))) ∨ (s ∧ (q ∨ p))))
(((((q ∧ p) → s) → (p ∨ (v ∧ r))) ∧ ((p ∨ (v ∧ r)) → ((r ∨ v) ∨ p))) → (((q ∧ p) → s) → ((r ∨ v) ∨ p)))
((((v → (p ∨ q)) ∧ ((v ∧ q) → r)) ∧ (p → (t ∨ v))) → ((v → (p ∨ q)) ∧ (((v ∧ q) → r) ∧ (p → (t ∨ v)))))
(((((t ∧ p) → v) → (r ∧ (v ∨ t))) ∧ ((r ∧ (v ∨ t)) → ((p ∧ r) ∧ t))) → (((t ∧ p) → v) → ((p ∧ r) ∧ t)))
(((((v ∨ p) ∨ r) ∨ (r → (u ∨ p))) ∨ ((u ∧ q) ∧ r)) → (((v ∨ p) ∨ r) ∨ ((r → (u ∨ p)) ∨ ((u ∧ q) ∧ r))))
(((((q ∧ t) → r) ↔ (u → (p ∨ q))) ∧ ((u → (p ∨ q)) ↔ (p ∧ (s ∨ r)))) → (((q ∧ t) → r) ↔ (p ∧ (s ∨ r))))
((((t → (s ∨ u)) ∨ (p ∨ (s ∧ u))) ∨ ((t ∧ r) → u)) → ((t → (s ∨ u)) ∨ ((p ∨ (s ∧ u)) ∨ ((t ∧ r) → u))))
(((r ∨ (s ∧ p)) ∧ ((t ∨ (u ∧ r)) ∧ ((s ∧ q) → t))) → (((r ∨ (s ∧ p)) ∧ (t ∨ (u ∧ r))) ∧ ((s ∧ q) → t)))
((((u ∧ s) ∧ t) ∧ ((p ∨ (t ∧ s)) ∧ (t ∨ (v ∧ q)))) → ((((u ∧ s) ∧ t) ∧ (p ∨ (t ∧ s))) ∧ (t ∨ (v ∧ q))))
(((p → (t ∨ r)) → ((r ∧ (q ∨ t)) → ((q ∧ s) → t))) → (((p → (t ∨ r)) ∧ (r ∧ (q ∨ t))) → ((q ∧ s) → t)))
((((v ∨ (s ∧ r)) ∧ (t ∧ (u ∨ p))) ∧ (t ∨ (v ∧ r))) → ((v ∨ (s ∧ r)) ∧ ((t ∧ (u ∨ p)) ∧ (t ∨ (v ∧ r)))))
(((u ∧ (s ∨ p)) ↔ ((r ∧ s) → v)) → (((u ∧ (s ∨ p)) ∧ ((v ∧ u) ∧ q)) ↔ (((r ∧ s) → v) ∧ ((v ∧ u) ∧ q))))
(((((q ∨ r) ∨ p) → ((s ∨ v) ∨ t)) ∧ (((s ∨ v) ∨ t) → ((q ∨ r) ∨ p))) → (((q ∨ r) ∨ p) ↔ ((s ∨ v) ∨ t)))
(((p ∨ (u ∧ q)) ↔ (p ∧ (v ∨ s))) → (((p ∨ (u ∧ q)) ∨ (t ∨ (r ∧ s))) ↔ ((p ∧ (v ∨ s)) ∨ (t ∨ (r ∧ s)))))
(((((t ∧ s) ∧ v) → ((s ∧ u) → r)) ∧ (((s ∧ u) → r) → ((t ∨ p) ∨ q))) → (((t ∧ s) ∧ v) → ((t ∨ p) ∨ q)))
(((((s ∧ r) → p) ∧ ((r ∧ t) → p)) → ((t ∨ q) ∨ r)) → (((s ∧ r) → p) → (((r ∧ t) → p) → ((t ∨ q) ∨ r))))
(((t ∨ (s ∧ v)) ∧ (((r ∧ u) ∧ s) ∧ ((r ∨ p) ∨ u))) → (((t ∨ (s ∧ v)) ∧ ((r ∧ u) ∧ s)) ∧ ((r ∨ p) ∨ u)))
((((t ∧ (p ∨ u)) → ((t ∧ v) ∧ s)) ∧ (((t ∧ v) ∧ s) → (t ∧ (p ∨ u)))) → ((t ∧ (p ∨ u)) ↔ ((t ∧ v) ∧ s)))
(((((r ∨ u) ∨ t) → (v → (u ∨ p))) ∧ ((v → (u ∨ p)) → ((r ∨ u) ∨ t))) → (((r ∨ u) ∨ t) ↔ (v → (u ∨ p))))
((((t ∨ (r ∧ v)) → (q ∧ (u ∨ s))) ∧ ((q ∧ (u ∨ s)) → ((r ∨ p) ∨ q))) → ((t ∨ (r ∧ v)) → ((r ∨ p) ∨ q)))
(((r → (u ∨ s)) ↔ (t → (q ∨ v))) → (((r → (u ∨ s)) ∧ ((t ∧ p) ∧ q)) ↔ ((t → (q ∨ v)) ∧ ((t ∧ p) ∧ q))))
((((r ∧ (p ∨ u)) ∧ ((q ∧ r) ∧ s)) ∧ ((p ∧ u) ∧ v)) → ((r ∧ (p ∨ u)) ∧ (((q ∧ r) ∧ s) ∧ ((p ∧ u) ∧ v))))
((((p → (t ∨ u)) ↔ (s → (t ∨ u))) ∧ ((s → (t ∨ u)) ↔ (r ∧ (q ∨ v)))) → ((p → (t ∨ u)) ↔ (r ∧ (q ∨ v))))
(((q ∨ (s ∧ v)) ∨ ((p → (q ∨ u)) ∨ ((r ∧ u) ∧ t))) → (((q ∨ (s ∧ v)) ∨ (p → (q ∨ u))) ∨ ((r ∧ u) ∧ t)))
((((r ∧ s) → q) → (((s ∧ q) → v) → ((u ∧ s) ∧ t))) → ((((r ∧ s) → q) ∧ ((s ∧ q) → v)) → ((u ∧ s) ∧ t)))
((((p → (u ∨ q)) ∨ ((s ∧ q) ∧ r)) ∨ (s ∧ (t ∨ v))) → ((p → (u ∨ q)) ∨ (((s ∧ q) ∧ r) ∨ (s ∧ (t ∨ v)))))
((((v ∧ (t ∨ q)) ↔ ((t ∧ s) ∧ q)) ∧ (((t ∧ s) ∧ q) ↔ (q ∧ (r ∨ u)))) → ((v ∧ (t ∨ q)) ↔ (q ∧ (r ∨ u))))
((((q ∧ p) → s) → (((u ∧ r) ∧ v) → (p ∧ (u ∨ t)))) → ((((q ∧ p) → s) ∧ ((u ∧ r) ∧ v)) → (p ∧ (u ∨ t))))
((((u ∨ (t ∧ s)) → ((r ∨ u) ∨ q)) ∧ (((r ∨ u) ∨ q) → ((q ∨ r) ∨ t))) → ((u ∨ (t ∧ s)) → ((q ∨ r) ∨ t)))
((((q → (s ∨ v)) → ((t ∨ u) ∨ s)) ∧ (((t ∨ u) ∨ s) → (r ∧ (u ∨ q)))) → ((q → (s ∨ v)) → (r ∧ (u ∨ q))))
((((u ∧ (t ∨ r)) → ((t ∧ r) ∧ p)) ∧ (((t ∧ r) ∧ p) → ((p ∨ s) ∨ r))) → ((u ∧ (t ∨ r)) → ((p ∨ s) ∨ r)))
(((v ∨ (p ∧ q)) → ((r → (u ∨ p)) → (v ∧ (s ∨ r)))) → (((v ∨ (p ∧ q)) ∧ (r → (u ∨ p))) → (v ∧ (s ∨ r))))
(((((s ∧ v) ∧ r) ∧ ((t ∧ s) ∧ r)) → ((r ∧ t) → v)) → (((s ∧ v) ∧ r) → (((t ∧ s) ∧ r) → ((r ∧ t) → v))))
(((((p ∨ u) ∨ s) ↔ (t ∧ (v ∨ s))) ∧ ((t ∧ (v ∨ s)) ↔ ((p ∨ r) ∨ v))) → (((p ∨ u) ∨ s) ↔ ((p ∨ r) ∨ v)))
(((u ∨ (s ∧ q)) ∧ (((p ∧ r) ∧ v) ∧ ((t ∨ s) ∨ u))) → (((u ∨ (s ∧ q)) ∧ ((p ∧ r) ∧ v)) ∧ ((t ∨ s) ∨ u)))
((((q ∧ t) ∧ u) → ((p ∧ (s ∨ r)) → ((u ∨ v) ∨ s))) → ((p ∧ (s ∨ r)) → (((q ∧ t) ∧ u) → ((u ∨ v) ∨ s))))
(((((u ∧ q) ∧ v) → (p ∧ (v ∨ u))) ∧ ((p ∧ (v ∨ u)) → ((u ∧ v) ∧ p))) → (((u ∧ q) ∧ v) → ((u ∧ v) ∧ p)))
((((u ∨ q) ∨ t) ∨ ((v → (u ∨ s)) ∨ ((s ∨ r) ∨ u))) → ((((u ∨ q) ∨ t) ∨ (v → (u ∨ s))) ∨ ((s ∨ r) ∨ u)))
(((((u ∨ r) ∨ p) ∧ (v ∨ (r ∧ t))) ∧ (q → (u ∨ s))) → (((u ∨ r) ∨ p) ∧ ((v ∨ (r ∧ t)) ∧ (q → (u ∨ s)))))
(((((q ∨ r) ∨ v) ↔ ((s ∧ v) → q)) ∧ (((s ∧ v) → q) ↔ ((r ∧ u) ∧ t))) → (((q ∨ r) ∨ v) ↔ ((r ∧ u) ∧ t)))
(((((t ∨ r) ∨ s) → ((t ∨ p) ∨ v)) ∧ (((t ∨ p) ∨ v) → ((q ∧ v) ∧ s))) → (((t ∨ r) ∨ s) → ((q ∧ v) ∧ s)))
((((t ∧ (r ∨ q)) ∧ ((q ∨ u) ∨ t)) → (t ∨ (p ∧ s))) → ((t ∧ (r ∨ q)) → (((q ∨ u) ∨ t) → (t ∨ (p ∧ s)))))
((((p ∨ v) ∨ q) → ((t ∧ (p ∨ v)) → ((p ∧ t) ∧ u))) → ((((p ∨ v) ∨ q) ∧ (t ∧ (p ∨ v))) → ((p ∧ t) ∧ u)))
((((q ∧ (s ∨ v)) → (r → (v ∨ s))) ∧ ((r → (v ∨ s)) → ((r ∨ p) ∨ t))) → ((q ∧ (s ∨ v)) → ((r ∨ p) ∨ t)))
((((q ∨ s) ∨ u) ∧ ((s ∧ (p ∨ r)) ∧ (t → (q ∨ v)))) → ((((q ∨ s) ∨ u) ∧ (s ∧ (p ∨ r))) ∧ (t → (q ∨ v))))
((((q ∧ p) → u) → (((t ∧ p) → q) → ((r ∧ s) ∧ q))) → ((((q ∧ p) → u) ∧ ((t ∧ p) → q)) → ((r ∧ s) ∧ q)))
(((t ∨ (q ∧ p)) ∧ (((q ∧ u) ∧ p) ∧ (q → (p ∨ t)))) → (((t ∨ (q ∧ p)) ∧ ((q ∧ u) ∧ p)) ∧ (q → (p ∨ t))))
((((q ∨ p) ∨ t) → ((t ∨ (s ∧ p)) → ((u ∨ q) ∨ t))) → ((((q ∨ p) ∨ t) ∧ (t ∨ (s ∧ p))) → ((u ∨ q) ∨ t)))
(((p ∨ (t ∧ s)) ↔ (q ∨ (p ∧ v))) → (((p ∨ (t ∧ s)) ∧ (q ∧ (p ∨ v))) ↔ ((q ∨ (p ∧ v)) ∧ (q ∧ (p ∨ v)))))
((((v ∧ (r ∨ t)) → ((s ∧ v) ∧ u)) ∧ (((s ∧ v) ∧ u) → (v ∧ (r ∨ t)))) → ((v ∧ (r ∨ t)) ↔ ((s ∧ v) ∧ u)))
((((v ∨ (t ∧ q)) → (u ∨ (t ∧ r))) ∧ ((u ∨ (t ∧ r)) → ((r ∧ q) → t))) → ((v ∨ (t ∧ q)) → ((r ∧ q) → t)))
(((p → (q ∨ r)) ∨ ((t ∧ (v ∨ r)) ∨ ((v ∧ r) ∧ s))) → (((p → (q ∨ r)) ∨ (t ∧ (v ∨ r))) ∨ ((v ∧ r) ∧ s)))
((((u ∨ r) ∨ t) ∧ ((p ∧ (t ∨ v)) ∧ ((t ∨ s) ∨ r))) → ((((u ∨ r) ∨ t) ∧ (p ∧ (t ∨ v))) ∧ ((t ∨ s) ∨ r)))
((((p ∧ q) ∧ t) → ((p ∨ (v ∧ r)) → (v ∨ (s ∧ r)))) → ((((p ∧ q) ∧ t) ∧ (p ∨ (v ∧ r))) → (v ∨ (s ∧ r))))
(((u ∧ (r ∨ p)) ↔ (t ∧ (r ∨ s))) → (((u ∧ (r ∨ p)) ∨ (u → (s ∨ p))) ↔ ((t ∧ (r ∨ s)) ∨ (u → (s ∨ p)))))
(((((p ∨ t) ∨ s) ∧ (u → (q ∨ s))) ∧ (q ∨ (s ∧ t))) → (((p ∨ t) ∨ s) ∧ ((u → (q ∨ s)) ∧ (q ∨ (s ∧ t)))))
((((v → (p ∨ r)) ∧ ((p ∨ s) ∨ q)) ∧ ((u ∧ r) → s)) → ((v → (p ∨ r)) ∧ (((p ∨ s) ∨ q) ∧ ((u ∧ r) → s))))
((((r ∧ q) ∧ t) → (((q ∨ r) ∨ p) → ((q ∨ r) ∨ v))) → ((((r ∧ q) ∧ t) ∧ ((q ∨ r) ∨ p)) → ((q ∨ r) ∨ v)))
(((p ∧ (s ∨ u)) ∧ (((q ∧ p) → t) ∧ ((s ∧ t) ∧ u))) → (((p ∧ (s ∨ u)) ∧ ((q ∧ p) → t)) ∧ ((s ∧ t) ∧ u)))
((((p → (q ∨ v)) → ((t ∧ u) → q)) ∧ (((t ∧ u) → q) → (p → (q ∨ v)))) → ((p → (q ∨ v)) ↔ ((t ∧ u) → q)))
((((s ∨ (v ∧ r)) → (p → (q ∨ r))) ∧ ((p → (q ∨ r)) → (s ∨ (v ∧ r)))) → ((s ∨ (v ∧ r)) ↔ (p → (q ∨ r))))
(((((v ∨ (p ∧ r)) → (t ∨ (u ∧ s))) ∧ ((¬t ∨ u) → r)) ∧ ((v ∨ (p ∧ r)) ∨ (¬t ∨ u))) → ((t ∨ (u ∧ s)) ∨ r))
((((q ∧ ¬r) → (r → (u ∨ s))) ∧ ((q ∧ ¬r) → ((q ∧ v) ∧ r))) → ((q ∧ ¬r) → ((r → (u ∨ s)) ∧ ((q ∧ v) ∧ r))))
((((¬s ∨ t) → ((p ∨ t) ∨ u)) ∧ ((¬s ∨ t) → ((s ∧ u) → v))) → ((¬s ∨ t) → (((p ∨ t) ∨ u) ∧ ((s ∧ u) → v))))
(((((s → (t ∨ q)) → ((q ∧ p) ∧ t)) ∧ (¬((s → (t ∨ q))) → p)) ∧ ((((q ∧ p) ∧ t) ∨ p) → (q ∨ v))) → (q ∨ v))
(((((t ∧ v) → (¬u ∨ r)) ∧ ((p ∧ ¬q) → ((u ∧ v) → p))) ∧ ((t ∧ v) ∨ (p ∧ ¬q))) → ((¬u ∨ r) ∨ ((u ∧ v) → p)))
((((((t → (p ∨ r)) ∨ ((r ∨ s) ∨ q)) ∨ ((s ∧ p) ∧ q)) ∧ ¬((t → (p ∨ r)))) ∧ ¬(((r ∨ s) ∨ q))) → ((s ∧ p) ∧ q))
((((((s ∨ (q ∧ v)) ∨ ((r ∧ p) → u)) ∨ (r ∧ (t ∨ u))) ∧ ¬((s ∨ (q ∧ v)))) ∧ ¬(((r ∧ p) → u))) → (r ∧ (t ∨ u)))
((((((r ∧ (t ∨ q)) ∨ (t ∧ (u ∨ v))) ∨ ((u ∧ s) → p)) ∧ ¬((r ∧ (t ∨ q)))) ∧ ¬((t ∧ (u ∨ v)))) → ((u ∧ s) → p))
((((((t ∧ (u ∨ q)) ∨ (q → (t ∨ p))) ∨ (s ∨ (q ∧ v))) ∧ ¬((t ∧ (u ∨ q)))) ∧ ¬((q → (t ∨ p)))) → (s ∨ (q ∧ v)))
(((((((q ∨ p) ∨ s) ∨ (q ∧ (v ∨ r))) ∨ (s ∨ (u ∧ t))) ∧ ¬(((q ∨ p) ∨ s))) ∧ ¬((q ∧ (v ∨ r)))) → (s ∨ (u ∧ t)))
(((((((r ∧ s) → v) ∨ ((t ∧ v) ∧ u)) ∨ (r → (p ∨ s))) ∧ ¬(((r ∧ s) → v))) ∧ ¬(((t ∧ v) ∧ u))) → (r → (p ∨ s)))
(((((((r ∧ u) ∧ s) ∨ (v → (p ∨ u))) ∨ ((u ∨ q) ∨ s)) ∧ ¬(((r ∧ u) ∧ s))) ∧ ¬((v → (p ∨ u)))) → ((u ∨ q) ∨ s))
(((((((q ∧ u) → p) ∨ (u ∨ (s ∧ p))) ∨ ((v ∨ r) ∨ s)) ∧ ¬(((q ∧ u) → p))) ∧ ¬((u ∨ (s ∧ p)))) → ((v ∨ r) ∨ s))
(((((v → r) → (u → (t ∨ p))) ∧ ((p → (s ∨ t)) → (u → (t ∨ p)))) ∧ ((v → r) ∨ (p → (s ∨ t)))) → (u → (t ∨ p)))
(((((q ↔ p) → (r ∧ (t ∨ v))) ∧ (((p ∨ r) ∨ t) → (r ∧ (t ∨ v)))) ∧ ((q ↔ p) ∨ ((p ∨ r) ∨ t))) → (r ∧ (t ∨ v)))
(((((r ∧ t) → p) → ((r ∨ u) ∨ s)) ∧ (((r ∧ t) → p) → (v ∧ t))) → (((r ∧ t) → p) → (((r ∨ u) ∨ s) ∧ (v ∧ t))))
(((((q ↔ v) → ((u ∧ s) ∧ r)) ∧ (((u ∨ s) ∨ r) → ((u ∧ s) ∧ r))) ∧ ((q ↔ v) ∨ ((u ∨ s) ∨ r))) → ((u ∧ s) ∧ r))
((((t ∧ (u ∨ s)) → (q ∧ v)) ∧ ((t ∧ (u ∨ s)) → ((q ∧ v) → t))) → ((t ∧ (u ∨ s)) → ((q ∧ v) ∧ ((q ∧ v) → t))))
(((((p ∨ (s ∧ q)) → ((p ∨ t) ∨ r)) ∧ ((r → q) → ((p ∨ t) ∨ r))) ∧ ((p ∨ (s ∧ q)) ∨ (r → q))) → ((p ∨ t) ∨ r))
((((v → (u ∨ t)) → ((q ∧ r) ∧ p)) ∧ ((v → (u ∨ t)) → (u ↔ v))) → ((v → (u ∨ t)) → (((q ∧ r) ∧ p) ∧ (u ↔ v))))
(((((q ∧ t) ∧ r) → (q ↔ t)) ∧ (((q ∧ t) ∧ r) → (s → (v ∨ t)))) → (((q ∧ t) ∧ r) → ((q ↔ t) ∧ (s → (v ∨ t)))))
((((u ∨ (t ∧ p)) → (u → v)) ∧ ((u ∨ (t ∧ p)) → ((u ∨ p) ∨ v))) → ((u ∨ (t ∧ p)) → ((u → v) ∧ ((u ∨ p) ∨ v))))
((((v ∨ (t ∧ r)) → (p → (r ∨ v))) ∧ ((v ∨ (t ∧ r)) → (p ↔ v))) → ((v ∨ (t ∧ r)) → ((p → (r ∨ v)) ∧ (p ↔ v))))
(((((v ∨ (p ∧ t)) → ((u ∧ p) ∧ r)) ∧ (¬((v ∨ (p ∧ t))) → ((u ∧ p) ∧ r))) ∧ (((u ∧ p) ∧ r) → (s ↔ q))) → (s ↔ q))
(((((r ∨ (u ∧ p)) → (s ∧ v)) ∧ (((p ∧ v) → u) → (t ∨ u))) ∧ ((r ∨ (u ∧ p)) ∨ ((p ∧ v) → u))) → ((s ∧ v) ∨ (t ∨ u)))
(((((v ↔ u) → (t ∨ (r ∧ s))) ∧ ((p ∧ r) → ((u ∧ r) → p))) ∧ ((v ↔ u) ∧ (p ∧ r))) → ((t ∨ (r ∧ s)) ∧ ((u ∧ r) → p)))
((((((u ∨ q) ∨ v) → (v ∧ (u ∨ q))) ∧ ((v ∧ (u ∨ q)) → (r ↔ q))) ∧ ((r ↔ q) → (v ∨ q))) → (((u ∨ q) ∨ v) → (v ∨ q)))
(((((p → v) → (t ∨ u)) ∧ (((s ∧ t) ∧ q) → (p ∨ (q ∧ r)))) ∧ ((p → v) ∨ ((s ∧ t) ∧ q))) → ((t ∨ u) ∨ (p ∨ (q ∧ r))))
(((((p ∧ ¬v) → (v ∨ s)) ∧ ((v ∨ s) → ((t ∨ v) ∨ u))) ∧ (((t ∨ v) ∨ u) → (s ∧ (p ∨ q)))) → ((p ∧ ¬v) → (s ∧ (p ∨ q))))
((((((t ∨ q) ∨ s) → (q ↔ v)) ∧ ((¬p ∨ s) → (u ∨ (r ∧ q)))) ∧ (((t ∨ q) ∨ s) ∨ (¬p ∨ s))) → ((q ↔ v) ∨ (u ∨ (r ∧ q))))
((((¬r → (s ∧ (v ∨ r))) ∧ ((s ∧ (v ∨ r)) → ((u ∧ s) ∧ r))) ∧ (((u ∧ s) ∧ r) → ((r ∧ q) → s))) → (¬r → ((r ∧ q) → s)))
(((((v → (q ∨ u)) → (¬u ∨ p)) ∧ ((q ∨ s) → (t ∨ (p ∧ q)))) ∧ ((v → (q ∨ u)) ∨ (q ∨ s))) → ((¬u ∨ p) ∨ (t ∨ (p ∧ q))))
(((((¬q ∨ r) → (¬q ∨ v)) ∧ (((q ∧ r) ∧ v) → (v ∧ (u ∨ p)))) ∧ ((¬q ∨ r) ∧ ((q ∧ r) ∧ v))) → ((¬q ∨ v) ∧ (v ∧ (u ∨ p))))
((((v ∨ (p ∧ q)) → (q ∧ (t ∨ p))) ∧ ((v ∨ (p ∧ q)) → ((p ∧ s) ∧ u))) → ((v ∨ (p ∧ q)) → ((q ∧ (t ∨ p)) ∧ ((p ∧ s) ∧ u))))
(((((r → (q ∨ s)) → (t ∧ (r ∨ s))) ∧ (((t ∧ r) → q) → (t ∧ (r ∨ s)))) ∧ ((r → (q ∨ s)) ∨ ((t ∧ r) → q))) → (t ∧ (r ∨ s)))
((((((q ∧ r) ∧ u) → (p ∧ (v ∨ q))) ∧ ((r → (q ∨ p)) → (p ∧ (v ∨ q)))) ∧ (((q ∧ r) ∧ u) ∨ (r → (q ∨ p)))) → (p ∧ (v ∨ q)))
((((((r ∧ t) → s) → (p ∨ (r ∧ u))) ∧ ((t ∧ (p ∨ v)) → (p ∨ (r ∧ u)))) ∧ (((r ∧ t) → s) ∨ (t ∧ (p ∨ v)))) → (p ∨ (r ∧ u)))
((((((v ∧ p) ∧ r) → (s ∧ (r ∨ v))) ∧ ((r ∧ (v ∨ u)) → (s ∧ (r ∨ v)))) ∧ (((v ∧ p) ∧ r) ∨ (r ∧ (v ∨ u)))) → (s ∧ (r ∨ v)))
(((((r ∧ (t ∨ q)) → (q ∨ (u ∧ v))) ∧ (((r ∧ t) → v) → (q ∨ (u ∧ v)))) ∧ ((r ∧ (t ∨ q)) ∨ ((r ∧ t) → v))) → (q ∨ (u ∧ v)))
((((((u ∧ q) → v) → (v ∨ (p ∧ s))) ∧ (((u ∧ r) ∧ p) → (v ∨ (p ∧ s)))) ∧ (((u ∧ q) → v) ∨ ((u ∧ r) ∧ p))) → (v ∨ (p ∧ s)))
((((((u ∨ v) ∨ s) → (p ∧ (t ∨ u))) ∧ ((t → (p ∨ r)) → (p ∧ (t ∨ u)))) ∧ (((u ∨ v) ∨ s) ∨ (t → (p ∨ r)))) → (p ∧ (t ∨ u)))
((((t → (r ∨ q)) → (s ∨ (q ∧ v))) ∧ ((t → (r ∨ q)) → (s ∧ (p ∨ t)))) → ((t → (r ∨ q)) → ((s ∨ (q ∧ v)) ∧ (s ∧ (p ∨ t)))))
(((((p ∨ t) ∨ s) → (v ∨ (p ∧ u))) ∧ (((p ∨ t) ∨ s) → ((u ∧ v) ∧ s))) → (((p ∨ t) ∨ s) → ((v ∨ (p ∧ u)) ∧ ((u ∧ v) ∧ s))))
(((((t ∨ (s ∧ v)) → ((t ∧ q) ∧ s)) ∧ ((s → (v ∨ u)) → ((t ∧ q) ∧ s))) ∧ ((t ∨ (s ∧ v)) ∨ (s → (v ∨ u)))) → ((t ∧ q) ∧ s))
((((r → (u ∨ q)) → (t → (q ∨ v))) ∧ ((r → (u ∨ q)) → (u ∧ (q ∨ p)))) → ((r → (u ∨ q)) → ((t → (q ∨ v)) ∧ (u ∧ (q ∨ p)))))
(((((r ∨ (v ∧ q)) → ((r ∨ u) ∨ v)) ∧ ((p → (u ∨ q)) → ((r ∨ u) ∨ v))) ∧ ((r ∨ (v ∧ q)) ∨ (p → (u ∨ q)))) → ((r ∨ u) ∨ v))
((((((v ∨ u) ∨ s) → (s ∧ (v ∨ u))) ∧ (((u ∨ p) ∨ t) → (s ∧ (v ∨ u)))) ∧ (((v ∨ u) ∨ s) ∨ ((u ∨ p) ∨ t))) → (s ∧ (v ∨ u)))
(((((u ∧ (v ∨ s)) → ((t ∨ r) ∨ q)) ∧ (((s ∨ p) ∨ q) → ((t ∨ r) ∨ q))) ∧ ((u ∧ (v ∨ s)) ∨ ((s ∨ p) ∨ q))) → ((t ∨ r) ∨ q))
((((((v ∧ r) → p) → (u ∨ (r ∧ v))) ∧ ((s → (u ∨ r)) → (u ∨ (r ∧ v)))) ∧ (((v ∧ r) → p) ∨ (s → (u ∨ r)))) → (u ∨ (r ∧ v)))
(((((r → (t ∨ s)) → ((p ∨ q) ∨ v)) ∧ ((v → (q ∨ p)) → ((p ∨ q) ∨ v))) ∧ ((r → (t ∨ s)) ∨ (v → (q ∨ p)))) → ((p ∨ q) ∨ v))
(((((p ∨ (u ∧ r)) → ((u ∧ r) ∧ t)) ∧ (((u ∧ s) → p) → ((u ∧ r) ∧ t))) ∧ ((p ∨ (u ∧ r)) ∨ ((u ∧ s) → p))) → ((u ∧ r) ∧ t))
(((((r → (t ∨ s)) → (q → (s ∨ t))) ∧ (((r ∨ u) ∨ p) → (q → (s ∨ t)))) ∧ ((r → (t ∨ s)) ∨ ((r ∨ u) ∨ p))) → (q → (s ∨ t)))
(((((p ∧ (q ∨ r)) → (p ∨ (s ∧ v))) ∧ (((v ∨ p) ∨ u) → (p ∨ (s ∧ v)))) ∧ ((p ∧ (q ∨ r)) ∨ ((v ∨ p) ∨ u))) → (p ∨ (s ∧ v)))
((((((q ∧ t) → v) → ((v ∧ p) → q)) ∧ (((r ∧ v) ∧ p) → ((v ∧ p) → q))) ∧ (((q ∧ t) → v) ∨ ((r ∧ v) ∧ p))) → ((v ∧ p) → q))
((((v ∧ (s ∨ p)) → ((t ∧ s) ∧ q)) ∧ ((v ∧ (s ∨ p)) → ((t ∧ u) → s))) → ((v ∧ (s ∨ p)) → (((t ∧ s) ∧ q) ∧ ((t ∧ u) → s))))
(((((u → (v ∨ r)) → ((v ∧ s) ∧ p)) ∧ ((p ∧ (q ∨ r)) → ((v ∧ s) ∧ p))) ∧ ((u → (v ∨ r)) ∨ (p ∧ (q ∨ r)))) → ((v ∧ s) ∧ p))
(((((v ∨ r) ∨ t) → ((v ∨ t) ∨ r)) ∧ (((v ∨ r) ∨ t) → (v ∨ (u ∧ p)))) → (((v ∨ r) ∨ t) → (((v ∨ t) ∨ r) ∧ (v ∨ (u ∧ p)))))
(((((u ∨ (v ∧ q)) → ((t ∧ r) ∧ p)) ∧ (¬((u ∨ (v ∧ q))) → ((t ∧ r) ∧ p))) ∧ (((t ∧ r) ∧ p) → ((s ∨ r) ∨ u))) → ((s ∨ r) ∨ u))
(((((u → (t ∨ p)) → ((r ∧ u) → p)) ∧ (¬((u → (t ∨ p))) → ((r ∧ u) → p))) ∧ (((r ∧ u) → p) → (v → (r ∨ s)))) → (v → (r ∨ s)))
(((((r ∧ (v ∨ p)) → ((t ∨ u) ∨ v)) ∧ (¬((r ∧ (v ∨ p))) → ((t ∨ u) ∨ v))) ∧ (((t ∨ u) ∨ v) → (p → (q ∨ u)))) → (p → (q ∨ u)))
(((((t → (v ∨ r)) → (r ∧ (q ∨ v))) ∧ (((r ∨ s) ∨ q) → (r → t))) ∧ ((t → (v ∨ r)) ∨ ((r ∨ s) ∨ q))) → ((r ∧ (q ∨ v)) ∨ (r → t)))
(((((p → (q ∨ t)) → ((r ∨ s) ∨ t)) ∧ (((t ∧ q) → r) → (u → s))) ∧ ((p → (q ∨ t)) ∧ ((t ∧ q) → r))) → (((r ∨ s) ∨ t) ∧ (u → s)))
(((((r ∨ p) → (q → (u ∨ t))) ∧ ((q → (u ∨ t)) → (p ∨ (q ∧ u)))) ∧ ((p ∨ (q ∧ u)) → (s ∧ (v ∨ t)))) → ((r ∨ p) → (s ∧ (v ∨ t))))
(((((q → (p ∨ s)) → (s ∧ q)) ∧ (((s ∨ t) ∨ u) → (u ∧ (p ∨ t)))) ∧ ((q → (p ∨ s)) ∧ ((s ∨ t) ∨ u))) → ((s ∧ q) ∧ (u ∧ (p ∨ t))))
(((((s ∨ q) → ((r ∧ u) → q)) ∧ ((s → (u ∨ v)) → (v → (r ∨ s)))) ∧ ((s ∨ q) ∧ (s → (u ∨ v)))) → (((r ∧ u) → q) ∧ (v → (r ∨ s))))
((((((v ∧ u) → t) → ((s ∨ v) ∨ u)) ∧ ((u ∧ (s ∨ r)) → (q → v))) ∧ (((v ∧ u) → t) ∨ (u ∧ (s ∨ r)))) → (((s ∨ v) ∨ u) ∨ (q → v)))
((((((u ∧ s) → p) → (v ∧ u)) ∧ ((v ∧ u) → (q ∧ (t ∨ p)))) ∧ ((q ∧ (t ∨ p)) → (q ∧ (s ∨ p)))) → (((u ∧ s) → p) → (q ∧ (s ∨ p))))
(((((v → (t ∨ p)) → (v → (u ∨ q))) ∧ ((p ∨ r) → (t ∨ (u ∧ s)))) ∧ ((v → (t ∨ p)) ∨ (p ∨ r))) → ((v → (u ∨ q)) ∨ (t ∨ (u ∧ s))))
(((((q → (s ∨ v)) → (s ∨ (r ∧ v))) ∧ (((v ∨ p) ∨ s) → (t ↔ q))) ∧ ((q → (s ∨ v)) ∧ ((v ∨ p) ∨ s))) → ((s ∨ (r ∧ v)) ∧ (t ↔ q)))
(((((q ∧ (v ∨ p)) → (t ∧ (p ∨ v))) ∧ ((s ∧ t) → (q ∨ (t ∧ v)))) ∧ ((q ∧ (v ∨ p)) ∨ (s ∧ t))) → ((t ∧ (p ∨ v)) ∨ (q ∨ (t ∧ v))))
((((((p ∧ r) → v) → (v ∨ r)) ∧ ((q → (r ∨ v)) → (r ∧ (s ∨ q)))) ∧ (((p ∧ r) → v) ∧ (q → (r ∨ v)))) → ((v ∨ r) ∧ (r ∧ (s ∨ q))))
((((((r ∧ s) → p) → (v → (t ∨ p))) ∧ ((r ↔ p) → (s ∨ (r ∧ t)))) ∧ (((r ∧ s) → p) ∧ (r ↔ p))) → ((v → (t ∨ p)) ∧ (s ∨ (r ∧ t))))
(((((q → (p ∨ t)) → (u ∧ p)) ∧ ((q → (u ∨ r)) → ((s ∧ t) → r))) ∧ ((q → (p ∨ t)) ∧ (q → (u ∨ r)))) → ((u ∧ p) ∧ ((s ∧ t) → r)))
((((((t ∧ v) → s) → (r ∨ (p ∧ u))) ∧ ((¬p ∨ r) → (s → (p ∨ r)))) ∧ (((t ∧ v) → s) ∨ (¬p ∨ r))) → ((r ∨ (p ∧ u)) ∨ (s → (p ∨ r))))
(((((q ∧ (u ∨ r)) → (p ∨ (u ∧ s))) ∧ (((t ∧ r) → u) → (s ∧ ¬r))) ∧ ((q ∧ (u ∨ r)) ∧ ((t ∧ r) → u))) → ((p ∨ (u ∧ s)) ∧ (s ∧ ¬r)))
(((((p ∧ (u ∨ q)) → ((r ∧ v) ∧ t)) ∧ ((¬s ∨ v) → (t ∨ (q ∧ p)))) ∧ ((p ∧ (u ∨ q)) ∧ (¬s ∨ v))) → (((r ∧ v) ∧ t) ∧ (t ∨ (q ∧ p))))
(((((t → (r ∨ v)) → (¬r ∨ s)) ∧ ((u → (r ∨ s)) → ((u ∨ v) ∨ t))) ∧ ((t → (r ∨ v)) ∧ (u → (r ∨ s)))) → ((¬r ∨ s) ∧ ((u ∨ v) ∨ t)))
(((((s ∧ ¬q) → (t ∧ (s ∨ v))) ∧ (((r ∧ p) ∧ s) → (u ∧ (v ∨ r)))) ∧ ((s ∧ ¬q) ∧ ((r ∧ p) ∧ s))) → ((t ∧ (s ∨ v)) ∧ (u ∧ (v ∨ r))))
(((((v ∨ (u ∧ p)) → ((u ∧ s) → p)) ∧ ((¬t ∨ s) → ((r ∧ t) → p))) ∧ ((v ∨ (u ∧ p)) ∧ (¬t ∨ s))) → (((u ∧ s) → p) ∧ ((r ∧ t) → p)))
(((((s ∨ (t ∧ r)) → (p ∧ ¬q)) ∧ ((p ∧ ¬q) → (s ∨ (v ∧ t)))) ∧ ((s ∨ (v ∧ t)) → ((u ∧ v) ∧ r))) → ((s ∨ (t ∧ r)) → ((u ∧ v) ∧ r)))
(((((r → (q ∨ v)) → (v ∨ (s ∧ p))) ∧ (((q ∧ r) → v) → (p → (s ∨ r)))) ∧ ((r → (q ∨ v)) ∧ ((q ∧ r) → v))) → ((v ∨ (s ∧ p)) ∧ (p → (s ∨ r))))
((((((v ∧ q) → s) → ((p ∧ q) ∧ v)) ∧ (((p ∧ q) ∧ v) → ((r ∧ u) ∧ s))) ∧ (((r ∧ u) ∧ s) → ((v ∨ p) ∨ t))) → (((v ∧ q) → s) → ((v ∨ p) ∨ t)))
(((((v → (t ∨ s)) → (r → (u ∨ q))) ∧ (((q ∧ p) → t) → (u ∨ (p ∧ q)))) ∧ ((v → (t ∨ s)) ∨ ((q ∧ p) → t))) → ((r → (u ∨ q)) ∨ (u ∨ (p ∧ q))))
(((((t ∨ (s ∧ v)) → (p ∨ (q ∧ s))) ∧ ((t ∧ (s ∨ u)) → ((p ∨ q) ∨ s))) ∧ ((t ∨ (s ∧ v)) ∧ (t ∧ (s ∨ u)))) → ((p ∨ (q ∧ s)) ∧ ((p ∨ q) ∨ s)))
((((((p ∨ q) ∨ u) → ((r ∨ q) ∨ s)) ∧ (((r ∨ u) ∨ q) → (q ∧ (p ∨ s)))) ∧ (((p ∨ q) ∨ u) ∨ ((r ∨ u) ∨ q))) → (((r ∨ q) ∨ s) ∨ (q ∧ (p ∨ s))))
(((((p ∨ (s ∧ t)) → (v ∧ (u ∨ r))) ∧ (((t ∨ s) ∨ q) → ((p ∨ u) ∨ r))) ∧ ((p ∨ (s ∧ t)) ∨ ((t ∨ s) ∨ q))) → ((v ∧ (u ∨ r)) ∨ ((p ∨ u) ∨ r)))
(((((r ∨ (u ∧ q)) → (r ∨ (u ∧ s))) ∧ (((u ∧ s) → v) → (p ∧ (q ∨ r)))) ∧ ((r ∨ (u ∧ q)) ∧ ((u ∧ s) → v))) → ((r ∨ (u ∧ s)) ∧ (p ∧ (q ∨ r))))
((((((q ∧ t) ∧ r) → ((s ∧ q) → v)) ∧ ((p ∧ (s ∨ u)) → ((r ∧ v) ∧ t))) ∧ (((q ∧ t) ∧ r) ∨ (p ∧ (s ∨ u)))) → (((s ∧ q) → v) ∨ ((r ∧ v) ∧ t)))
((((((v ∨ r) ∨ s) → (v → (q ∨ p))) ∧ ((t → (v ∨ r)) → (s ∨ (v ∧ t)))) ∧ (((v ∨ r) ∨ s) ∧ (t → (v ∨ r)))) → ((v → (q ∨ p)) ∧ (s ∨ (v ∧ t))))
(((((p ∨ (s ∧ u)) → ((q ∨ p) ∨ v)) ∧ (((t ∨ q) ∨ p) → ((v ∨ u) ∨ t))) ∧ ((p ∨ (s ∧ u)) ∨ ((t ∨ q) ∨ p))) → (((q ∨ p) ∨ v) ∨ ((v ∨ u) ∨ t)))
(((((q ∧ (v ∨ s)) → ((q ∧ t) → r)) ∧ ((q ∨ (r ∧ s)) → ((p ∧ r) → t))) ∧ ((q ∧ (v ∨ s)) ∧ (q ∨ (r ∧ s)))) → (((q ∧ t) → r) ∧ ((p ∧ r) → t)))
((((((v ∧ s) → p) → ((s ∧ r) → u)) ∧ ((v ∨ (r ∧ p)) → (u ∨ (v ∧ s)))) ∧ (((v ∧ s) → p) ∨ (v ∨ (r ∧ p)))) → (((s ∧ r) → u) ∨ (u ∨ (v ∧ s))))
((((((s ∧ q) ∧ u) → (q ∨ (s ∧ v))) ∧ ((v → (r ∨ s)) → (v ∧ (p ∨ r)))) ∧ (((s ∧ q) ∧ u) ∧ (v → (r ∨ s)))) → ((q ∨ (s ∧ v)) ∧ (v ∧ (p ∨ r))))
(((((u → (s ∨ v)) → (t ∧ (q ∨ p))) ∧ ((q ∧ (v ∨ p)) → ((u ∧ t) → s))) ∧ ((u → (s ∨ v)) ∨ (q ∧ (v ∨ p)))) → ((t ∧ (q ∨ p)) ∨ ((u ∧ t) → s)))
((((((p ∧ q) ∧ t) → ((v ∨ r) ∨ q)) ∧ (((p ∨ s) ∨ r) → ((v ∧ u) ∧ q))) ∧ (((p ∧ q) ∧ t) ∧ ((p ∨ s) ∨ r))) → (((v ∨ r) ∨ q) ∧ ((v ∧ u) ∧ q)))
(((((q → (p ∨ v)) → ((t ∧ v) → p)) ∧ (((p ∧ q) → u) → ((q ∧ s) ∧ p))) ∧ ((q → (p ∨ v)) ∨ ((p ∧ q) → u))) → (((t ∧ v) → p) ∨ ((q ∧ s) ∧ p)))
((((((u ∧ q) ∧ t) → (u ∨ (s ∧ q))) ∧ ((r ∨ (p ∧ s)) → ((t ∧ s) ∧ r))) ∧ (((u ∧ q) ∧ t) ∨ (r ∨ (p ∧ s)))) → ((u ∨ (s ∧ q)) ∨ ((t ∧ s) ∧ r)))
((((((t ∧ s) ∧ r) → (v ∨ (s ∧ q))) ∧ ((q → (u ∨ r)) → ((v ∨ p) ∨ t))) ∧ (((t ∧ s) ∧ r) ∨ (q → (u ∨ r)))) → ((v ∨ (s ∧ q)) ∨ ((v ∨ p) ∨ t)))
((((((v ∧ q) → s) → (p ∨ (s ∧ u))) ∧ ((p ∨ (s ∧ u)) → (t → (r ∨ p)))) ∧ ((t → (r ∨ p)) → (v ∨ (q ∧ u)))) → (((v ∧ q) → s) → (v ∨ (q ∧ u))))
(((((u → (t ∨ v)) → ((t ∨ s) ∨ q)) ∧ (((v ∧ u) → t) → ((p ∧ u) → v))) ∧ ((u → (t ∨ v)) ∨ ((v ∧ u) → t))) → (((t ∨ s) ∨ q) ∨ ((p ∧ u) → v)))
(((((r ∨ (s ∧ t)) → ((u ∨ p) ∨ r)) ∧ ((v ∨ (s ∧ p)) → ((s ∧ r) → q))) ∧ ((r ∨ (s ∧ t)) ∧ (v ∨ (s ∧ p)))) → (((u ∨ p) ∨ r) ∧ ((s ∧ r) → q)))
((((((p ∨ u) ∨ r) → ((p ∧ s) ∧ v)) ∧ ((r → (s ∨ t)) → (v → (p ∨ s)))) ∧ (((p ∨ u) ∨ r) ∧ (r → (s ∨ t)))) → (((p ∧ s) ∧ v) ∧ (v → (p ∨ s))))
((((((q ∧ t) → u) → ((v ∧ t) → r)) ∧ ((r ∧ (u ∨ t)) → (r ∨ (t ∧ s)))) ∧ (((q ∧ t) → u) ∧ (r ∧ (u ∨ t)))) → (((v ∧ t) → r) ∧ (r ∨ (t ∧ s))))
(((((q → (s ∨ t)) → ((t ∧ v) → p)) ∧ ((t ∧ (u ∨ q)) → ((u ∧ t) → p))) ∧ ((q → (s ∨ t)) ∨ (t ∧ (u ∨ q)))) → (((t ∧ v) → p) ∨ ((u ∧ t) → p)))
((((((p ∧ u) ∧ v) → ((r ∨ v) ∨ p)) ∧ (((s ∨ u) ∨ r) → (t → (p ∨ r)))) ∧ (((p ∧ u) ∧ v) ∧ ((s ∨ u) ∨ r))) → (((r ∨ v) ∨ p) ∧ (t → (p ∨ r))))
((((((q ∧ s) → p) → (p ∧ (s ∨ u))) ∧ (((s ∨ v) ∨ t) → ((r ∨ s) ∨ q))) ∧ (((q ∧ s) → p) ∨ ((s ∨ v) ∨ t))) → ((p ∧ (s ∨ u)) ∨ ((r ∨ s) ∨ q)))
((((((v ∨ s) ∨ u) → (t ∨ (r ∧ u))) ∧ (((u ∨ r) ∨ v) → ((u ∨ v) ∨ s))) ∧ (((v ∨ s) ∨ u) ∧ ((u ∨ r) ∨ v))) → ((t ∨ (r ∧ u)) ∧ ((u ∨ v) ∨ s)))
(((((t → (p ∨ r)) → ((t ∧ r) → u)) ∧ (¬((t → (p ∨ r))) → ((v ∨ p) ∨ t))) ∧ ((((t ∧ r) → u) ∨ ((v ∨ p) ∨ t)) → ((s ∨ q) ∨ u))) → ((s ∨ q) ∨ u))
((((((u ∨ r) ∨ p) → (q ∨ (u ∧ v))) ∧ (¬(((u ∨ r) ∨ p)) → (v → (s ∨ q)))) ∧ (((q ∨ (u ∧ v)) ∨ (v → (s ∨ q))) → ((u ∧ p) ∧ t))) → ((u ∧ p) ∧ t))
((((((s ∧ r) → v) → (v ∨ (r ∧ u))) ∧ (¬(((s ∧ r) → v)) → (q ∧ (v ∨ p)))) ∧ (((v ∨ (r ∧ u)) ∨ (q ∧ (v ∨ p))) → (p ∧ (t ∨ v)))) → (p ∧ (t ∨ v)))
((((((u ∧ p) ∧ t) → ((u ∧ r) ∧ q)) ∧ (¬(((u ∧ p) ∧ t)) → ((v ∧ s) ∧ r))) ∧ ((((u ∧ r) ∧ q) ∨ ((v ∧ s) ∧ r)) → ((p ∧ q) ∧ v))) → ((p ∧ q) ∧ v))
(((((t → (q ∨ p)) → (u → (t ∨ q))) ∧ (¬((t → (q ∨ p))) → (u ∧ (v ∨ p)))) ∧ (((u → (t ∨ q)) ∨ (u ∧ (v ∨ p))) → (u ∧ (p ∨ s)))) → (u ∧ (p ∨ s)))
(((((p → (v ∨ q)) → ((r ∧ u) → s)) ∧ (¬((p → (v ∨ q))) → ((v ∧ p) ∧ u))) ∧ ((((r ∧ u) → s) ∨ ((v ∧ p) ∧ u)) → (v ∧ (t ∨ r)))) → (v ∧ (t ∨ r)))
((((((q ∧ p) ∧ t) → ((u ∨ p) ∨ s)) ∧ (¬(((q ∧ p) ∧ t)) → ((s ∨ v) ∨ p))) ∧ ((((u ∨ p) ∨ s) ∨ ((s ∨ v) ∨ p)) → ((r ∧ s) → p))) → ((r ∧ s) → p))
((((((t ∧ s) ∧ p) → (r ∨ (q ∧ s))) ∧ (¬(((t ∧ s) ∧ p)) → ((t ∧ u) ∧ q))) ∧ (((r ∨ (q ∧ s)) ∨ ((t ∧ u) ∧ q)) → (r → (p ∨ v)))) → (r → (p ∨ v)))
((((((u ∨ p) ∨ t) → ((v ∧ u) ∧ r)) ∧ (¬(((u ∨ p) ∨ t)) → (t ∨ (p ∧ v)))) ∧ ((((v ∧ u) ∧ r) ∨ (t ∨ (p ∧ v))) → ((q ∨ p) ∨ r))) → ((q ∨ p) ∨ r))
((((((v ∧ q) → (q ∨ (r ∧ t))) ∧ ((t ∧ (r ∨ q)) → (r ∧ (p ∨ t)))) ∧ (((q ∨ (r ∧ t)) ∧ (r ∧ (p ∨ t))) → (p → s))) ∧ ((v ∧ q) ∧ (t ∧ (r ∨ q)))) → (p → s))
((((((q ∧ t) → ((u ∧ r) → q)) ∧ (((q ∧ u) → v) → (r ∧ (t ∨ p)))) ∧ ((((u ∧ r) → q) ∧ (r ∧ (t ∨ p))) → (r → (p ∨ t)))) ∧ ((q ∧ t) ∧ ((q ∧ u) → v))) → (r → (p ∨ t)))
(((((((p ∨ u) ∨ t) → ((s ∧ r) → q)) ∧ ((p ∧ (v ∨ t)) → ((t ∧ q) → p))) ∧ ((((s ∧ r) → q) ∧ ((t ∧ q) → p)) → (r ∧ q))) ∧ (((p ∨ u) ∨ t) ∧ (p ∧ (v ∨ t)))) → (r ∧ q))
(((((((t ∧ u) → s) → (u → (q ∨ s))) ∧ ((t ∧ (u ∨ s)) → ((v ∨ u) ∨ p))) ∧ (((u → (q ∨ s)) ∧ ((v ∨ u) ∨ p)) → (p ∨ u))) ∧ (((t ∧ u) → s) ∧ (t ∧ (u ∨ s)))) → (p ∨ u))
(((((((u ∧ s) ∧ v) → ((t ∨ r) ∨ v)) ∧ (((t ∧ q) → p) → (r ∧ (s ∨ p)))) ∧ ((((t ∨ r) ∨ v) ∧ (r ∧ (s ∨ p))) → (r ∨ (p ∧ u)))) ∧ (((u ∧ s) ∧ v) ∧ ((t ∧ q) → p))) → (r ∨ (p ∧ u)))
(((((((p ∨ u) ∨ r) → ((u ∧ s) → v)) ∧ (((q ∧ v) ∧ s) → (q ∧ (u ∨ r)))) ∧ ((((u ∧ s) → v) ∧ (q ∧ (u ∨ r))) → (s ∨ (v ∧ q)))) ∧ (((p ∨ u) ∨ r) ∧ ((q ∧ v) ∧ s))) → (s ∨ (v ∧ q)))
(((((((q ∨ s) ∨ t) → (q ∨ (r ∧ u))) ∧ (((p ∧ u) → q) → ((q ∧ r) ∧ v))) ∧ (((q ∨ (r ∧ u)) ∧ ((q ∧ r) ∧ v)) → (s → (t ∨ q)))) ∧ (((q ∨ s) ∨ t) ∧ ((p ∧ u) → q))) → (s → (t ∨ q)))
(((((((q ∧ s) → p) → ((s ∧ t) ∧ r)) ∧ ((s → (p ∨ q)) → ((q ∧ r) ∧ p))) ∧ ((((s ∧ t) ∧ r) ∧ ((q ∧ r) ∧ p)) → ((v ∧ r) → u))) ∧ (((q ∧ s) → p) ∧ (s → (p ∨ q)))) → ((v ∧ r) → u))
(((((((u ∧ p) ∧ q) → ((q ∧ p) → t)) ∧ ((r ∧ (q ∨ s)) → (v ∨ (r ∧ p)))) ∧ ((((q ∧ p) → t) ∧ (v ∨ (r ∧ p))) → ((t ∨ p) ∨ u))) ∧ (((u ∧ p) ∧ q) ∧ (r ∧ (q ∨ s)))) → ((t ∨ p) ∨ u))
(((((((t ∨ q) ∨ u) → (q → (u ∨ p))) ∧ (((t ∧ v) ∧ r) → ((t ∨ s) ∨ u))) ∧ (((q → (u ∨ p)) ∧ ((t ∨ s) ∨ u)) → ((v ∧ p) → u))) ∧ (((t ∨ q) ∨ u) ∧ ((t ∧ v) ∧ r))) → ((v ∧ p) → u))'''.splitlines()


In [ ]:
for i in range(len(testy)):
    testy[i]= parse_infix(testy[i])

In [ ]:
testy_normalised = set(normalise(j) for j in testy)


def process_one_proof(proof_text):
    try:
        proof_parsed = parseProof(proof_text)
        conclusion = proof_parsed[len(proof_parsed)].LP.conclusion
        conclusion_norm = normalise(conclusion)

        if not (conclusion_norm in testy_normalised):
            return proof_text
        else:
            return None

    except Exception:
        return None


with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    results = list(
        tqdm(
            executor.map(process_one_proof, proofs, chunksize=10),
            total=len(proofs)
        )
    )

proofs_good = [x for x in results if x is not None]

In [ ]:
proofs_good

In [ ]:
#with open("KORPUSY/Korpus_bez_założeń_bez_testów.txt", "w", encoding="utf-8") as f:
#    f.write("\n\n".join(proofs_good))

In [2]:
with open("KORPUSY/Korpus_bez_założeń_bez_testów.txt", "r", encoding="utf-8") as f:
    proofs_table = f.read().split("\n\n")

from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import os
def worker(args):
    i, proof_text = args
    global bad_proof
    parsed = parseProof(proof_text)
    l = parsed.__len__()
    dict_key = parsed[parsed.__len__()]
    dict_key = normalise(dict_key.LP.conclusion)
    return i, l, dict_key

proofs_simplified_dict = dict()

with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    results = executor.map(worker, enumerate(proofs_table), chunksize=20)
    for i, l, dict_key in tqdm(results, total=len(proofs_table)):
        try:
            if dict_key in proofs_simplified_dict:
                proofs_simplified_dict[dict_key].append([i, l])
            else:
                proofs_simplified_dict[dict_key] = [[i, l]]
        except Exception as e:
            print(i, l, dict_key)
            raise e


 21%|██        | 390480/1850093 [00:48<03:00, 8074.02it/s] 


KeyboardInterrupt: 

In [ ]:
important_idxs = []
for i in tqdm(proofs_simplified_dict,total = proofs_simplified_dict.__len__()):
    idxs = proofs_simplified_dict[i]
    important_idx = idxs[0]
    for j in idxs:
        if j[1] < important_idx[1]:
            important_idx = j
    important_idxs.append(important_idx)

In [ ]:
for i in tqdm(important_idxs,total = important_idxs.__len__()):
    proof = proofs_table[i[0]]
    proof_parsed = parseProof(proof)
    thesis = proof_parsed[len(proof_parsed)].LP.assumptions
    if not thesis == Context([],[]):
        print(1)


In [ ]:
important_idxs

In [ ]:
proofs_normalised = []
for i in tqdm(important_idxs,total = len(important_idxs)):
    proof = proofs_table[i[0]]
    proof_parsed = parseProof(proof)
    thesis = proof_parsed[len(proof_parsed)].LP.conclusion
    proof_normalised = "Prove that: " + to_infix(thesis)+"\n"
    proof_normalised += proofs_table[i[0]]
    proofs_normalised.append(proof_normalised)

In [ ]:

#with open("KORPUSY/korpus_bez_założeń_normalised.txt", "w", encoding="utf-8") as f:
#    f.write("\n\n".join(proofs_normalised))

In [ ]:
for i in proofs_normalised:

    j = "\n".join(i.splitlines()[1:])
    print(i.splitlines()[0],"\n")
    print(j,"\n\n\n")

# Smashe

In [9]:
with open("KORPUSY/korpus_bez_założeń_normalised.txt", "r", encoding="utf-8") as f:
    proofs_normalised = f.read().split("\n\n")
def process_one(proof_text: str) -> str:
    proof_text = "\n".join(proof_text.splitlines()[1:])
    proof = parseProof(proof_text)
    parts = []
    for j in proof.keys():
        parts.append(pretty_print_smashed_problem(smash_with_proof(proof[j], proof)))
    for i in parts:
        check_smash_from_text(i)
    return parts

with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    smashed_proofs = list(
        tqdm(
            executor.map(
                process_one,
                proofs_normalised,
                chunksize=3
            ),
            total=len(proofs_normalised)
        )
    )

smashed_proofs = [x for sublist in smashed_proofs for x in sublist]


100%|██████████| 426101/426101 [00:44<00:00, 9667.29it/s] 


In [10]:
#with open("KORPUSY/korpus_smashe_raw.txt", "w", encoding="utf-8") as f:
#    f.write("\n\n".join(smashed_proofs))

In [18]:
#with open("KORPUSY/korpus_iffs.txt", "w", encoding="utf-8") as f:
#    f.write("\n\n".join(iffs))

In [11]:
#with open("KORPUSY/korpus_smashe_raw.txt", "r", encoding="utf-8") as f:
#    smashed_proofs = f.read().split("\n\n")


## Iffy

In [12]:
iffs = parallel_random_iffs(100000)

100%|██████████| 100000/100000 [04:20<00:00, 384.37it/s]


In [13]:
for i in tqdm(range(len(iffs))):
    iffs[i] = pretty_print_smashed_problem(iffs[i])

100%|██████████| 100000/100000 [00:01<00:00, 67534.35it/s]


In [14]:
for i in tqdm(range(len(iffs))):
    check_smash_from_text(iffs[i])

100%|██████████| 100000/100000 [00:43<00:00, 2316.90it/s]


In [15]:
smashed_proofs += iffs

In [16]:
for i in tqdm(smashed_proofs,total = len(smashed_proofs)):
    check_smash_from_text(i)

100%|██████████| 3710121/3710121 [09:34<00:00, 6460.40it/s] 


### Koniec generowania Iffów

In [21]:

from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm
import os
def process_one(smash_text: str) -> str:
    ans = normalise_smashed_problem(smash_text)
    check_smash_from_text(ans)
    return ans

with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    smashed_proofs_normalised = list(
        tqdm(
            executor.map(
                process_one,
                smashed_proofs,
                chunksize=3
            ),
            total=len(smashed_proofs)
        )
    )

100%|██████████| 3710121/3710121 [01:32<00:00, 40207.74it/s]


In [22]:
#with open("KORPUSY/korpus_smashe_z_ifff_z_znormalizowanymi_kontekstami.txt", "w", encoding="utf-8") as f:
#    f.write("\n\n".join(smashed_proofs_normalised))


In [11]:
#smashed_proofs_normalised

['Smash: (p → q) ∧ p ⊢ (p → q) ∧ p\nWith: Assumption\nTo: ',
 'Smash: (p → q) ∧ p ⊢ p → q\nWith: ConjunctionElimination1\nTo: (p → q) ∧ p ⊢ (p → q) ∧ p',
 'Smash: (p → q) ∧ p ⊢ p\nWith: ConjunctionElimination2\nTo: (p → q) ∧ p ⊢ (p → q) ∧ p',
 'Smash: (p → q) ∧ p ⊢ q\nWith: ImplicationElimination\nTo: (p → q) ∧ p ⊢ p → q,\n(p → q) ∧ p ⊢ p',
 'Smash:  ⊢ ((p → q) ∧ p) → q\nWith: ImplicationIntroduction\nTo: (p → q) ∧ p ⊢ q',
 'Smash: (p → q) ∧ p ⊢ (p → q) ∧ p\nWith: Assumption\nTo: ',
 'Smash: (p → q) ∧ p ⊢ p → q\nWith: ConjunctionElimination1\nTo: (p → q) ∧ p ⊢ (p → q) ∧ p',
 'Smash: (p → q) ∧ ¬q ⊢ (p → q) ∧ ¬q\nWith: Assumption\nTo: ',
 'Smash: (p → q) ∧ ¬q ⊢ p → q\nWith: ConjunctionElimination1\nTo: (p → q) ∧ ¬q ⊢ (p → q) ∧ ¬q',
 'Smash: (p → q) ∧ ¬q ⊢ ¬q\nWith: ConjunctionElimination2\nTo: (p → q) ∧ ¬q ⊢ (p → q) ∧ ¬q',
 'Smash: p ⊢ p\nWith: Assumption\nTo: ',
 'Smash: p, (p → q) ∧ ¬q ⊢ q\nWith: ImplicationElimination\nTo: p, (p → q) ∧ ¬q ⊢ p → q,\np, (p → q) ∧ ¬q ⊢ p',
 'Smash: p, (p

In [23]:

with open("KORPUSY/korpus_smashe_z_ifff_z_znormalizowanymi_kontekstami.txt", "r", encoding="utf-8") as f:
    smashes = f.read().split("\n\n")

In [24]:
smashes_grouped_by_rule = sets_to_sample("KORPUSY/korpus_smashe_z_ifff_z_znormalizowanymi_kontekstami.txt")

100%|██████████| 3710121/3710121 [00:01<00:00, 3181334.85it/s]


In [25]:
for i in smashes_grouped_by_rule:
    print(i, len(smashes_grouped_by_rule[i]))

With: Assumption 1070147
With: ConjunctionElimination1 446764
With: ConjunctionElimination2 125890
With: ImplicationElimination 123659
With: ImplicationIntroduction 303131
With: NegationElimination 129835
With: NegationIntroduction 75374
With: DisjunctionIntroduction1 237352
With: DisjunctionIntroduction2 226942
With: Weakening 47716
With: TND 190840
With: DisjunctionElimination 35576
With: ConjunctionIntroduction 62639
With: FromContext 145367
With: LieElimination 23283
With: IffIntroduction 242872
With: IffElimination1 45155
With: NegationOfNegation 44927
With: IffElimination2 42871
With: FromWeakenContext 29712
With: TruthIntroduction 11012
With: RAA 49057


In [26]:
interesting_iffs = []
for i in tqdm(smashes_grouped_by_rule['With: IffIntroduction'],total = len(smashes_grouped_by_rule['With: IffIntroduction'])):
    smashed_parsed = check_smash_from_text(i)
    if smashed_parsed.problem.conclusion.Left() != smashed_parsed.problem.conclusion.Right():
        interesting_iffs.append(i)
    elif random.randint(0, 100) == 0:
        interesting_iffs.append(i)

100%|██████████| 242872/242872 [01:55<00:00, 2105.76it/s]


In [27]:
smashes_grouped_by_rule['With: IffIntroduction'] = interesting_iffs

In [28]:
weights = {'With: Assumption' : 0.04,
'With: ConjunctionElimination1' :0.001,
'With: ConjunctionElimination2': 0.001,
'With: ImplicationElimination': 0.04,
'With: ImplicationIntroduction': 1,
'With: NegationElimination':0.01,
'With: NegationIntroduction':1,
'With: DisjunctionIntroduction1':0.4,
'With: DisjunctionIntroduction2':0.4,
'With: Weakening':0.01,
'With: FromContext':0.01,
'With: TND':0.1,
'With: DisjunctionElimination':0.7,
'With: ConjunctionIntroduction':1,
'With: LieElimination':1,
'With: IffIntroduction':1,

'With: IffElimination1':0.01,
'With: IffElimination2':0.01,

'With: NegationOfNegation':0.01,

'With: RAA':0.8,

'With: TruthIntroduction':0.0001,
'With: FromWeakenContext':0.01}

In [30]:
def corpus_with_probabilities(smashed_grouped_by_rule, weights_dict, how_many_each_one):
    rules = list(smashed_grouped_by_rule.keys())
    weights = []
    for i in rules:
        weights.append(weights_dict[i])
    sum = 0
    for i in weights:
        sum+= i
    for i in range(len(weights)):
        weights[i] = weights[i]/sum
    ans = []
    for i in tqdm(range(len(smashes_grouped_by_rule.keys())*how_many_each_one)):
        chosen = random.choices(rules,weights=weights)[0]
        ans.append(random.choice(smashes_grouped_by_rule[chosen]))
    return "\n\n".join(ans)

In [31]:
corpus = corpus_with_probabilities(smashes_grouped_by_rule, weights,200000)

100%|██████████| 4400000/4400000 [00:03<00:00, 1210503.94it/s]


In [32]:
with open("KORPUSY/korpus_smashe_z_ifff_z_znormalizowanymi_kontekstami_z_znormalizowanymi_prawdopodobienstwami.txt", "w", encoding="utf-8") as f:
    f.write(corpus)

In [35]:
corpus = corpus.split("\n\n")
for i in tqdm(corpus,total = len(corpus)):
    check_smash_from_text(i)

100%|██████████| 4400000/4400000 [20:04<00:00, 3654.16it/s] 


# Provy z kontekstem

In [26]:
with open("KORPUSY/korpus_bez_założeń_normalised.txt", "r", encoding="utf-8") as f:
    proofs_normalised = f.read().split("\n\n")
def _extract_and_check_one(proof_with_goal):
    proof = "\n".join(proof_with_goal.splitlines()[1:])
    proofs_extracted_text = give_all_extraction_text(proof)
    proofs_extracted = [
        p.strip()
        for p in re.split(r'\n\s*\n', proofs_extracted_text.strip())
        if p.strip()
    ]

    # walidacja
    for p in proofs_extracted:
        check_proof_with_assumptions(p)

    return proofs_extracted


poofs = []

with ProcessPoolExecutor() as ex:
    results = list(tqdm(ex.map(_extract_and_check_one, proofs_normalised), total=len(proofs_normalised)))

proofs = list(chain.from_iterable(results))

100%|██████████| 426101/426101 [04:44<00:00, 1495.40it/s]


In [27]:
with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    proofs_with_optimized_numbers = list(
        tqdm(
            executor.map(
                replace_all_number_by_smallest_number,
                proofs, #text
                chunksize=320
            ),
            total=len(proofs)
        )
    )

100%|██████████| 3610121/3610121 [13:34<00:00, 4430.04it/s]


In [28]:
proofs_with_optimized_numbers

['Prove that: (p → q) ∧ p\nassuming that: (p → q) ∧ p\n1. (p → q) ∧ p    context, 0',
 'Prove that: p → q\nassuming that: (p → q) ∧ p\n1. (p → q) ∧ p    context, 0\n2. p → q    ∧-elimination1, 1',
 'Prove that: p\nassuming that: (p → q) ∧ p\n1. (p → q) ∧ p    context, 0\n2. p    ∧-elimination2, 1',
 'Prove that: q\nassuming that: (p → q) ∧ p\n1. (p → q) ∧ p    context, 0\n2. p → q    ∧-elimination1, 1\n3. p    ∧-elimination2, 1\n4. q    →-elimination, 2, 3',
 'Prove that: ((p → q) ∧ p) → q\n1. (p → q) ∧ p    assumption\n2. p → q    ∧-elimination1, 1\n3. p    ∧-elimination2, 1\n4. q    →-elimination, 2, 3\n5. ((p → q) ∧ p) → q    →-introduction, 1, 4',
 'Prove that: (p → q) ∧ p\nassuming that: (p → q) ∧ p\n1. (p → q) ∧ p    context, 0',
 'Prove that: p → q\nassuming that: (p → q) ∧ p\n1. (p → q) ∧ p    context, 0\n2. p → q    ∧-elimination1, 1',
 'Prove that: (p → q) ∧ ¬q\nassuming that: (p → q) ∧ ¬q\n1. (p → q) ∧ ¬q    context, 0',
 'Prove that: p → q\nassuming that: (p → q) ∧ ¬q\n1. (

In [29]:


from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm


def is_taken_from_context(proof):
    proofParsed = check_proof_with_assumptions(proof)
    base_context = proofParsed[1].LP.assumptions.base_context
    goal = proofParsed[len(proofParsed)].LP.conclusion
    x = random.randint(0,100)
    if x == 0:
        return False
    return goal in base_context


with ProcessPoolExecutor(max_workers=15) as ex:
    mask = list(tqdm(
        ex.map(is_taken_from_context, proofs_with_optimized_numbers, chunksize=50),
        total=len(proofs_with_optimized_numbers)
    ))

proofs_without_trivial = [
    proof
    for proof, taken in zip(proofs_with_optimized_numbers, mask)
    if not taken
]

100%|██████████| 3610121/3610121 [00:46<00:00, 77739.06it/s]


In [30]:
print(len(proofs_without_trivial))
print(len(proofs_with_optimized_numbers))

2211106
3610121


In [31]:
def remove_extra_lines_at_the_end(proof):
    parsedProof = check_proof_with_assumptions(proof)
    smallest_last_idx = len(parsedProof)

    for i in parsedProof.keys():
        if parsedProof[i].LP == parsedProof[smallest_last_idx].LP:
            if i < smallest_last_idx:
                smallest_last_idx = i

    ans = ""
    splitted_proof = proof.splitlines()
    current_idx = 0

    for i in range(len(splitted_proof)):
        ans += splitted_proof[i]

        if current_idx >= 1:
            current_idx += 1

        if splitted_proof[i][0] == "1" and current_idx == 0:
            current_idx += 1

        if current_idx == smallest_last_idx:
            check_proof_with_assumptions(ans)
            return ans

        ans += "\n"


with ProcessPoolExecutor(max_workers=30) as ex:
    proofs_without_extra_lines_at_the_end = list(
        tqdm(
            ex.map(remove_extra_lines_at_the_end, proofs_without_trivial, chunksize=50),
            total=len(proofs_without_trivial)
        )
    )

100%|██████████| 2211106/2211106 [01:00<00:00, 36269.56it/s]


In [32]:
with ProcessPoolExecutor(max_workers=30) as ex:
    proofs_without_extra_lines_at_the_end = list(
        tqdm(
            ex.map(remove_extra_lines_at_the_end, proofs_without_trivial, chunksize=50),
            total=len(proofs_without_trivial)
        )
    )

100%|██████████| 2211106/2211106 [01:02<00:00, 35436.17it/s]


In [33]:
with ProcessPoolExecutor(max_workers=30) as ex:
    proofs_extracted = list(
        tqdm(
            ex.map(
                extract_proof_with_assumptions,
                proofs_without_extra_lines_at_the_end,
                chunksize=50
            ),
            total=len(proofs_without_extra_lines_at_the_end)
        )
    )

100%|██████████| 2211106/2211106 [00:27<00:00, 80684.05it/s]


In [34]:
for i in tqdm(proofs_extracted,total = len(proofs_extracted)):
    check_proof_with_assumptions(i)

100%|██████████| 2211106/2211106 [07:07<00:00, 5173.23it/s]


In [20]:
def normalise_smashed_problem(SP_text):
    SP = check_smash_from_text(SP_text)
    match SP.rule:
        case Assumption():
            return SP_text
        case Weakening():
            return SP_text
        case ImplicationIntroduction():
            return SP_text
        case NegationIntroduction():
            return SP_text
        case ImplicationElimination():
            SP.subproblems[0].assumptions = SP.problem.assumptions
            SP.subproblems[1].assumptions = SP.problem.assumptions
            is_smashed_correctly(SP)
            return pretty_print_smashed_problem(SP)
        case NegationElimination():
            SP.subproblems[0].assumptions = SP.problem.assumptions
            SP.subproblems[1].assumptions = SP.problem.assumptions
            is_smashed_correctly(SP)
            return pretty_print_smashed_problem(SP)
        case ConjunctionIntroduction():
            SP.subproblems[0].assumptions = SP.problem.assumptions
            SP.subproblems[1].assumptions = SP.problem.assumptions
            is_smashed_correctly(SP)
            return pretty_print_smashed_problem(SP)
        case DisjunctionIntroduction1():
            return SP_text
        case DisjunctionIntroduction2():
            return SP_text
        case TruthIntroduction():
            return SP_text
        case ConjunctionElimination1():
            return SP_text
        case ConjunctionElimination2():
            return SP_text
        case DisjunctionElimination():
            phi = SP.subproblems[0].conclusion.Left()
            psi = SP.subproblems[0].conclusion.Right()
            SP.subproblems[0].assumptions = SP.problem.assumptions
            SP.subproblems[1].assumptions.base_context = SP.problem.assumptions.base_context + [phi]
            SP.subproblems[2].assumptions.base_context = SP.problem.assumptions.base_context + [psi]
            is_smashed_correctly(SP)
            return pretty_print_smashed_problem(SP)
        case LieElimination():
            return SP_text
        case IffIntroduction():
            phi = SP.problem.conclusion.Left()
            psi = SP.problem.conclusion.Right()
            SP.subproblems[0].assumptions.base_context = SP.problem.assumptions.base_context + [phi]
            SP.subproblems[1].assumptions.base_context = SP.problem.assumptions.base_context + [psi]
            is_smashed_correctly(SP)
            return pretty_print_smashed_problem(SP)
        case IffElimination1():
            SP.subproblems[0].assumptions = SP.problem.assumptions
            SP.subproblems[1].assumptions = SP.problem.assumptions
            is_smashed_correctly(SP)
            return pretty_print_smashed_problem(SP)
        case IffElimination2():
            SP.subproblems[0].assumptions = SP.problem.assumptions
            SP.subproblems[1].assumptions = SP.problem.assumptions
            is_smashed_correctly(SP)
            return pretty_print_smashed_problem(SP)
        case RAA():
            return SP_text
        case NegationOfNegation():
            return SP_text
        case TND():
            return SP_text
        case FromContext():
            return SP_text
        case FromWeakenContext():
            return SP_text


In [35]:
#with open("KORPUSY/korpus_z_założeniami.txt", "w", encoding="utf-8") as f:
#    f.write("\n\n".join(proofs_extracted))


In [ ]:
#with open("KORPUSY/smashes_raw_with_iffs.txt", "r", encoding="utf-8") as f:
#    smashes = f.read().split("\n\n")


In [ ]:
for i in tqdm(smashes,total = len(smashes)):
    check_smash_from_text(i)

In [ ]:
for i in iffs:
    print(i)
    check_smash_from_text(i)